In [1]:
# ── KAGGLE SETUP ─────────────────────────────────────────────────────────────
import subprocess, sys, os, shutil

# Install any extra packages not pre-installed on Kaggle
for pkg in ['pyarrow']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

# Copy dataset from Kaggle input to working directory
KAGGLE_DATA = '/kaggle/input/datasets/abhirammopuri/usclosures/US_Closures.csv'
LOCAL_DATA  = 'US_Closures.csv'

if os.path.exists(KAGGLE_DATA):
    if not os.path.exists(LOCAL_DATA):
        shutil.copy(KAGGLE_DATA, LOCAL_DATA)
        print(f"✓ Dataset copied: {KAGGLE_DATA} → {LOCAL_DATA}")
    else:
        print(f"✓ Dataset already present: {LOCAL_DATA}")
else:
    raise FileNotFoundError(
        "\n❌  Kaggle dataset not found at: " + KAGGLE_DATA + "\n"
        "    Please attach the dataset to this notebook:\n"
        "    Kaggle sidebar → Add Data → search 'US Road Construction and Closures' (by sobhanmoosavi)"
    )

# Create output directory tree
for d in ['outputs/eda', 'outputs/spatial', 'outputs/temporal',
          'outputs/ml', 'outputs/mining', 'outputs/tables']:
    os.makedirs(d, exist_ok=True)

print("✓ Output directories ready.")
print(f"✓ Dataset size: {os.path.getsize(LOCAL_DATA)/1024/1024:.1f} MB")


✓ Dataset copied: /kaggle/input/datasets/abhirammopuri/usclosures/US_Closures.csv → US_Closures.csv
✓ Output directories ready.
✓ Dataset size: 2571.0 MB


---
## Part 1 — EDA & Feature Engineering
Loads and preprocesses the real dataset, performs exploratory data analysis, and engineers spatial + temporal features. Saves `outputs/processed_data.parquet` for Parts 2–4.

In [2]:
"""
AID843 - Spatio-Temporal Data Analytics Assignment 3
US Road Construction and Closures (2016-2021) Dataset
Part 1: EDA, Feature Engineering (Spatial + Temporal)

Team Assignment - 3 Members
Dataset: https://www.kaggle.com/datasets/sobhanmoosavi/us-road-construction-and-closures
"""

# ============================================================
# IMPORTS
# ============================================================
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
import os

# ============================================================
# CONFIG
# ============================================================
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/eda", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/spatial", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/temporal", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/ml", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/mining", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/tables", exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd',
          '#8c564b','#e377c2','#7f7f7f','#bcbd22','#17becf']
sns.set_palette(COLORS)

print("="*70)
print("AID843 A3 - US Road Construction & Closures Analysis")
print("="*70)

# ============================================================
# DATA LOADING
# ============================================================
def load_data(filepath="US_Closures.csv"):
    """Load dataset - place US_Closures.csv in working directory"""
    print(f"\n[1] Loading data from: {filepath}")
    if not os.path.exists(filepath):
        raise FileNotFoundError(
            f"\n\n❌  Dataset not found: '{filepath}'\n"
            "    Please attach the Kaggle dataset and ensure the setup cell above ran correctly.\n"
            "    Expected path after setup: US_Closures.csv (copied from Kaggle input)"
        )
    df = pd.read_csv(filepath, low_memory=False)
    print(f"    Shape: {df.shape}")
    print(f"    Columns: {list(df.columns)}")
    return df

# ============================================================
# DATA PREPROCESSING
# ============================================================
def preprocess(df):
    print("\n[2] Preprocessing...")
    
    # Parse datetimes
    for col in ['Start_Time', 'End_Time']:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')

    # Drop rows with missing lat/lon or start time
    df = df.dropna(subset=['Start_Lat', 'Start_Lng', 'Start_Time'])

    # Compute duration
    if 'End_Time' in df.columns and 'Start_Time' in df.columns:
        df['Duration_hrs'] = (df['End_Time'] - df['Start_Time']
                              ).dt.total_seconds() / 3600
        df = df[df['Duration_hrs'] > 0]
        df = df[df['Duration_hrs'] < 720]  # Remove unrealistic >30 days

    # Extract temporal features
    df['Year']        = df['Start_Time'].dt.year
    df['Month']       = df['Start_Time'].dt.month
    df['DayOfWeek']   = df['Start_Time'].dt.dayofweek   # 0=Mon
    df['Hour']        = df['Start_Time'].dt.hour
    df['Quarter']     = df['Start_Time'].dt.quarter
    df['WeekOfYear']  = df['Start_Time'].dt.isocalendar().week.astype(int)
    df['IsWeekend']   = (df['DayOfWeek'] >= 5).astype(int)
    df['Season']      = df['Month'].map(
        {12:'Winter',1:'Winter',2:'Winter',
         3:'Spring',4:'Spring',5:'Spring',
         6:'Summer',7:'Summer',8:'Summer',
         9:'Fall',10:'Fall',11:'Fall'})
    df['IsRushHour']  = df['Hour'].apply(
        lambda h: 1 if (6 <= h <= 9) or (15 <= h <= 18) else 0)
    df['IsNight']     = ((df['Hour'] < 6) | (df['Hour'] >= 20)).astype(int)

    # Log-transform duration
    df['Log_Duration'] = np.log1p(df['Duration_hrs'])

    print(f"    Clean shape: {df.shape}")
    print(f"    Date range: {df['Start_Time'].min()} to {df['Start_Time'].max()}")
    print(f"    States: {df['State'].nunique() if 'State' in df.columns else 'N/A'}")
    return df

# ============================================================
# EDA
# ============================================================
def eda(df):
    print("\n[3] Exploratory Data Analysis...")

    # --- Summary statistics table ---
    num_cols = ['Duration_hrs', 'Distance(mi)' if 'Distance(mi)' in df.columns else None,
                'Temperature(F)' if 'Temperature(F)' in df.columns else None,
                'Humidity(%)' if 'Humidity(%)' in df.columns else None,
                'Wind_Speed(mph)' if 'Wind_Speed(mph)' in df.columns else None,
                'Visibility(mi)' if 'Visibility(mi)' in df.columns else None]
    num_cols = [c for c in num_cols if c is not None and c in df.columns]
    stats_df = df[num_cols].describe().round(3)
    stats_df.to_csv(f"{OUTPUT_DIR}/tables/summary_statistics.csv")
    print("\n    Summary Statistics:")
    print(stats_df.to_string())

    # --- Dataset overview table ---
    overview = pd.DataFrame({
        'Metric': ['Total Records','Date Range Start','Date Range End',
                   'Total States','Total Cities','Missing Values (%)'],
        'Value': [
            f"{len(df):,}",
            str(df['Start_Time'].min().date()),
            str(df['Start_Time'].max().date()),
            str(df['State'].nunique()) if 'State' in df.columns else 'N/A',
            str(df['City'].nunique()) if 'City' in df.columns else 'N/A',
            f"{df.isnull().mean().mean()*100:.2f}%"
        ]
    })
    overview.to_csv(f"{OUTPUT_DIR}/tables/dataset_overview.csv", index=False)
    print("\n    Dataset Overview:")
    print(overview.to_string(index=False))

    # --- Missing values heatmap ---
    fig, ax = plt.subplots(figsize=(14, 5))
    missing_pct = df.isnull().mean() * 100
    missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)
    if len(missing_pct) > 0:
        sns.barplot(x=missing_pct.index, y=missing_pct.values, ax=ax, palette='Reds_r')
        ax.set_title('Missing Value Percentage by Column', fontsize=14, fontweight='bold')
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
        ax.set_ylabel('Missing %')
    else:
        ax.text(0.5, 0.5, 'No Missing Values', ha='center', va='center',
                fontsize=16, transform=ax.transAxes)
        ax.set_title('Missing Values Analysis', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/eda/fig1_missing_values.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("    [Fig 1] Missing values saved.")

    # --- Temporal distribution: events per year-month ---
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle('Temporal Distribution of Road Construction Events', 
                 fontsize=16, fontweight='bold', y=1.01)

    # Yearly
    year_counts = df.groupby('Year').size()
    axes[0,0].bar(year_counts.index, year_counts.values, color=COLORS[0], edgecolor='white')
    axes[0,0].set_title('Annual Distribution')
    axes[0,0].set_xlabel('Year'); axes[0,0].set_ylabel('Event Count')
    for i, (y, v) in enumerate(year_counts.items()):
        axes[0,0].text(y, v + 100, f'{v:,}', ha='center', fontsize=8)

    # Monthly
    month_counts = df.groupby('Month').size()
    month_names = ['Jan','Feb','Mar','Apr','May','Jun',
                   'Jul','Aug','Sep','Oct','Nov','Dec']
    axes[0,1].bar(month_counts.index, month_counts.values, color=COLORS[1], edgecolor='white')
    axes[0,1].set_xticks(range(1,13)); axes[0,1].set_xticklabels(month_names, rotation=45)
    axes[0,1].set_title('Monthly Distribution')
    axes[0,1].set_xlabel('Month'); axes[0,1].set_ylabel('Event Count')

    # Day of week
    dow_counts = df.groupby('DayOfWeek').size()
    dow_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
    axes[1,0].bar(dow_counts.index, dow_counts.values, color=COLORS[2], edgecolor='white')
    axes[1,0].set_xticks(range(7)); axes[1,0].set_xticklabels(dow_names)
    axes[1,0].set_title('Day of Week Distribution')
    axes[1,0].set_xlabel('Day'); axes[1,0].set_ylabel('Event Count')

    # Hourly
    hour_counts = df.groupby('Hour').size()
    axes[1,1].plot(hour_counts.index, hour_counts.values, 
                   marker='o', color=COLORS[3], linewidth=2)
    axes[1,1].fill_between(hour_counts.index, hour_counts.values, alpha=0.3, color=COLORS[3])
    axes[1,1].set_title('Hourly Distribution')
    axes[1,1].set_xlabel('Hour of Day (UTC)'); axes[1,1].set_ylabel('Event Count')
    axes[1,1].set_xticks(range(0,24,2))

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/eda/fig2_temporal_distributions.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("    [Fig 2] Temporal distributions saved.")

    # --- Closure type distribution ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    if 'Type' in df.columns:
        type_counts = df['Type'].value_counts()
        axes[0].pie(type_counts.values, labels=type_counts.index,
                    autopct='%1.1f%%', colors=COLORS[:len(type_counts)], startangle=90)
        axes[0].set_title('Closure Type Distribution', fontsize=13, fontweight='bold')

    if 'Severity' in df.columns:
        sev_counts = df['Severity'].value_counts().sort_index()
        axes[1].bar(sev_counts.index.astype(str), sev_counts.values, color=COLORS[:3])
        axes[1].set_title('Severity Distribution', fontsize=13, fontweight='bold')
        axes[1].set_xlabel('Severity Level'); axes[1].set_ylabel('Count')
        for i, v in enumerate(sev_counts.values):
            axes[1].text(i, v + 100, f'{v:,}', ha='center', fontsize=9)

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/eda/fig3_closure_severity.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("    [Fig 3] Closure type & severity saved.")

    # --- Duration distribution ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].hist(df['Duration_hrs'].clip(0, 100), bins=60, color=COLORS[0],
                 edgecolor='white', alpha=0.85)
    axes[0].set_title('Event Duration Distribution (clipped at 100h)',
                      fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Duration (hours)'); axes[0].set_ylabel('Frequency')

    axes[1].hist(df['Log_Duration'], bins=60, color=COLORS[1],
                 edgecolor='white', alpha=0.85)
    axes[1].set_title('Log-Transformed Duration', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('log(1 + Duration_hrs)'); axes[1].set_ylabel('Frequency')

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/eda/fig4_duration_distribution.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("    [Fig 4] Duration distribution saved.")

    # --- Top states bar chart ---
    if 'State' in df.columns:
        fig, ax = plt.subplots(figsize=(14, 5))
        state_counts = df['State'].value_counts().head(20)
        sns.barplot(x=state_counts.index, y=state_counts.values, ax=ax, palette='viridis')
        ax.set_title('Top 20 States by Number of Construction Events',
                     fontsize=13, fontweight='bold')
        ax.set_xlabel('State'); ax.set_ylabel('Event Count')
        for i, v in enumerate(state_counts.values):
            ax.text(i, v + 50, f'{v:,}', ha='center', fontsize=7, rotation=45)
        plt.tight_layout()
        plt.savefig(f"{OUTPUT_DIR}/eda/fig5_top_states.png", dpi=150, bbox_inches='tight')
        plt.close()
        print("    [Fig 5] Top states saved.")

    # --- Weather condition distribution ---
    if 'Weather_Condition' in df.columns:
        fig, ax = plt.subplots(figsize=(12, 5))
        wc = df['Weather_Condition'].value_counts().head(15)
        sns.barplot(x=wc.index, y=wc.values, ax=ax, palette='coolwarm')
        ax.set_title('Top 15 Weather Conditions During Construction Events',
                     fontsize=13, fontweight='bold')
        ax.set_xlabel('Weather Condition'); ax.set_ylabel('Count')
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig(f"{OUTPUT_DIR}/eda/fig6_weather_conditions.png", dpi=150, bbox_inches='tight')
        plt.close()
        print("    [Fig 6] Weather conditions saved.")

    # --- Correlation heatmap ---
    weather_cols = [c for c in ['Temperature(F)','Humidity(%)','Pressure(in)',
                                 'Visibility(mi)','Wind_Speed(mph)','Precipitation(in)',
                                 'Duration_hrs','Severity'] if c in df.columns]
    if len(weather_cols) > 3:
        fig, ax = plt.subplots(figsize=(10, 8))
        corr = df[weather_cols].corr()
        mask = np.triu(np.ones_like(corr, dtype=bool))
        sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                    mask=mask, ax=ax, square=True, cbar_kws={'shrink': 0.8})
        ax.set_title('Correlation Matrix of Numerical Features', 
                     fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f"{OUTPUT_DIR}/eda/fig7_correlation_heatmap.png", dpi=150, bbox_inches='tight')
        plt.close()
        print("    [Fig 7] Correlation heatmap saved.")

    return stats_df, overview

# ============================================================
# FEATURE ENGINEERING
# ============================================================
def feature_engineering(df):
    print("\n[4] Feature Engineering...")

    # ---- TEMPORAL FEATURES ----
    print("    [A] Temporal Feature Engineering:")

    # Cyclical encoding of periodic features
    df['Hour_sin']     = np.sin(2 * np.pi * df['Hour'] / 24)
    df['Hour_cos']     = np.cos(2 * np.pi * df['Hour'] / 24)
    df['Month_sin']    = np.sin(2 * np.pi * df['Month'] / 12)
    df['Month_cos']    = np.cos(2 * np.pi * df['Month'] / 12)
    df['DOW_sin']      = np.sin(2 * np.pi * df['DayOfWeek'] / 7)
    df['DOW_cos']      = np.cos(2 * np.pi * df['DayOfWeek'] / 7)
    df['Week_sin']     = np.sin(2 * np.pi * df['WeekOfYear'] / 52)
    df['Week_cos']     = np.cos(2 * np.pi * df['WeekOfYear'] / 52)
    print("        - Cyclical time encodings (sin/cos for hour, month, DOW, week)")

    # Time-of-day categories
    def time_category(h):
        if 5 <= h < 12:   return 'Morning'
        elif 12 <= h < 17: return 'Afternoon'
        elif 17 <= h < 21: return 'Evening'
        else:              return 'Night'
    df['TimeOfDay'] = df['Hour'].apply(time_category)
    print("        - Time-of-day categories (Morning/Afternoon/Evening/Night)")

    # Rolling aggregates (event count per month-state)
    if 'State' in df.columns:
        df['YearMonth'] = df['Year'].astype(str) + '-' + df['Month'].astype(str).str.zfill(2)
        state_month_count = df.groupby(['State','YearMonth']).size().reset_index(name='StateMonthCount')
        df = df.merge(state_month_count, on=['State','YearMonth'], how='left')
        print("        - State-Month event count (spatial density proxy)")

    # ---- SPATIAL FEATURES ----
    print("    [B] Spatial Feature Engineering:")

    # Grid-based region
    df['LatGrid']   = (df['Start_Lat'] // 1).astype(int)
    df['LonGrid']   = (df['Start_Lng'] // 1).astype(int)
    df['GridCell']  = df['LatGrid'].astype(str) + '_' + df['LonGrid'].astype(str)
    grid_counts     = df.groupby('GridCell').size().reset_index(name='GridCellCount')
    df = df.merge(grid_counts, on='GridCell', how='left')
    print("        - 1-degree lat/lon grid cell assignment & density")

    # H3-like hexagon index (approximated with 0.5-degree bins)
    df['HexLat']    = (df['Start_Lat'] / 0.5).apply(np.floor) * 0.5
    df['HexLon']    = (df['Start_Lng'] / 0.5).apply(np.floor) * 0.5
    df['HexCell']   = df['HexLat'].astype(str) + '_' + df['HexLon'].astype(str)
    hex_counts      = df.groupby('HexCell').size().reset_index(name='HexCellCount')
    df = df.merge(hex_counts, on='HexCell', how='left')
    print("        - 0.5-degree hexagonal grid cell & event density")

    # Distance from centroid (US centroid ~39.5N, 98.35W)
    US_LAT, US_LON = 39.5, -98.35
    df['DistFromCenter'] = np.sqrt((df['Start_Lat'] - US_LAT)**2 +
                                    (df['Start_Lng'] - US_LON)**2)
    print("        - Distance from US geographic centroid")

    # POI feature aggregate
    poi_cols = [c for c in ['Junction','Traffic_Signal','Crossing','Roundabout',
                             'Station','Railway','Amenity','Bump','Stop',
                             'Give_Way','No_Exit','Traffic_Calming','Turning_Loop'] if c in df.columns]
    if poi_cols:
        df[poi_cols] = df[poi_cols].apply(
            lambda col: col.map({True: 1, False: 0, 'True': 1, 'False': 0})
                          .fillna(0).astype(int)
        )
        df['POI_Count'] = df[poi_cols].sum(axis=1)
        print(f"        - POI aggregate count ({len(poi_cols)} POI types)")

    # ---- WEATHER FEATURES ----
    print("    [C] Weather Feature Engineering:")
    if 'Temperature(F)' in df.columns:
        df['TempCategory'] = pd.cut(df['Temperature(F)'],
                                     bins=[-50, 32, 50, 68, 86, 150],
                                     labels=['Freezing','Cold','Mild','Warm','Hot'])
    if 'Visibility(mi)' in df.columns:
        df['LowVisibility'] = (df['Visibility(mi)'] < 3).astype(int)
    if 'Precipitation(in)' in df.columns:
        df['HasPrecipitation'] = (df['Precipitation(in)'] > 0).astype(int)
    print("        - Temperature category, low-visibility flag, precipitation flag")

    # ---- MAP MATCHING / SYNTHESIS FEATURES ----
    print("    [D] Geospatial Feature Engineering (Map Synthesis):")

    # Proximity to coastlines (crude: east/west/none)
    def coast_proximity(lon):
        if lon > -80: return 'East'
        elif lon < -115: return 'West'
        else: return 'Interior'
    df['CoastProximity'] = df['Start_Lng'].apply(coast_proximity)

    # Urban/rural proxy by grid density
    if 'GridCellCount' in df.columns:
        density_threshold = df['GridCellCount'].quantile(0.75)
        df['IsUrban'] = (df['GridCellCount'] > density_threshold).astype(int)
    print("        - Coast proximity (East/West/Interior)")
    print("        - Urban/Rural proxy (grid density quartile)")

    # Feature summary table
    new_features = [
        ('Hour_sin/cos', 'Cyclical hour encoding', 'Temporal'),
        ('Month_sin/cos', 'Cyclical month encoding', 'Temporal'),
        ('DOW_sin/cos', 'Cyclical day-of-week encoding', 'Temporal'),
        ('Week_sin/cos', 'Cyclical week-of-year encoding', 'Temporal'),
        ('IsWeekend', 'Binary weekend flag', 'Temporal'),
        ('IsRushHour', 'Peak traffic hours flag', 'Temporal'),
        ('IsNight', 'Nighttime flag (20:00-06:00)', 'Temporal'),
        ('Season', 'Meteorological season', 'Temporal'),
        ('TimeOfDay', 'Time of day category', 'Temporal'),
        ('StateMonthCount', 'State-month event density', 'Spatio-Temporal'),
        ('GridCell/Count', '1-degree spatial grid & density', 'Spatial'),
        ('HexCell/Count', '0.5-degree hex grid & density', 'Spatial'),
        ('DistFromCenter', 'Distance from US centroid', 'Spatial'),
        ('POI_Count', 'Aggregate nearby POI count', 'Spatial'),
        ('LowVisibility', 'Visibility below 3mi flag', 'Weather'),
        ('HasPrecipitation', 'Precipitation presence flag', 'Weather'),
        ('TempCategory', 'Temperature discretization', 'Weather'),
        ('CoastProximity', 'Proximity to coast', 'Spatial'),
        ('IsUrban', 'Urban/rural classification', 'Spatial'),
    ]
    feat_df = pd.DataFrame(new_features, columns=['Feature','Description','Category'])
    feat_df.to_csv(f"{OUTPUT_DIR}/tables/engineered_features.csv", index=False)
    print(f"\n    Feature engineering table saved ({len(new_features)} features).")
    print(feat_df.to_string(index=False))

    # --- Feature importance visualization (placeholder with variance) ---
    numeric_features = [c for c in df.select_dtypes(include=[np.number]).columns
                        if c not in ['Start_Lat','Start_Lng','EndLat','EndLng',
                                     'LatGrid','LonGrid','HexLat','HexLon']]
    numeric_features = numeric_features[:20]  # top 20
    variances = df[numeric_features].var().sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(12, 6))
    variances.plot(kind='barh', ax=ax, color=COLORS[4])
    ax.set_title('Feature Variance (Top 20 Numeric Features)',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Variance')
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/eda/fig8_feature_variance.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("    [Fig 8] Feature variance chart saved.")

    # --- PCA visualization of feature space ---
    pca_cols = [c for c in ['Hour_sin','Hour_cos','Month_sin','Month_cos',
                              'DOW_sin','DOW_cos','DistFromCenter','Log_Duration',
                              'IsWeekend','IsRushHour','IsNight','POI_Count']
                if c in df.columns]
    if len(pca_cols) >= 4:
        sample = df[pca_cols].dropna().sample(min(20000, len(df)), random_state=42)
        scaler = StandardScaler()
        scaled = scaler.fit_transform(sample)
        pca = PCA(n_components=2, random_state=42)
        components = pca.fit_transform(scaled)

        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        fig.suptitle('PCA of Engineered Feature Space', fontsize=14, fontweight='bold')

        # Color by hour
        hour_sample = df.loc[sample.index, 'Hour'] if 'Hour' in df.columns else None
        sc = axes[0].scatter(components[:, 0], components[:, 1],
                              c=hour_sample if hour_sample is not None else COLORS[0],
                              cmap='plasma', alpha=0.4, s=1)
        axes[0].set_title(f'Colored by Hour | Var explained: {pca.explained_variance_ratio_.sum():.1%}')
        axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
        axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
        plt.colorbar(sc, ax=axes[0], label='Hour of Day')

        # Explained variance plot
        pca_full = PCA(n_components=min(len(pca_cols), 10), random_state=42)
        pca_full.fit(scaled)
        cumvar = np.cumsum(pca_full.explained_variance_ratio_)
        axes[1].bar(range(1, len(cumvar)+1), pca_full.explained_variance_ratio_,
                    color=COLORS[0], alpha=0.7, label='Individual')
        axes[1].plot(range(1, len(cumvar)+1), cumvar, 'o-', color=COLORS[1], label='Cumulative')
        axes[1].axhline(0.9, ls='--', color='gray', label='90% threshold')
        axes[1].set_title('PCA Explained Variance')
        axes[1].set_xlabel('Principal Component')
        axes[1].set_ylabel('Explained Variance Ratio')
        axes[1].legend()

        plt.tight_layout()
        plt.savefig(f"{OUTPUT_DIR}/eda/fig9_pca_features.png", dpi=150, bbox_inches='tight')
        plt.close()
        print("    [Fig 9] PCA feature space saved.")

    return df, feat_df

# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":
    # ---- Load ----
    df = load_data("US_Closures.csv")

    # ---- Preprocess ----
    df = preprocess(df)

    # ---- EDA ----
    stats_df, overview = eda(df)

    # ---- Feature Engineering ----
    df, feat_df = feature_engineering(df)

    # Save processed dataframe for use in next parts
    df.to_parquet(f"{OUTPUT_DIR}/processed_data.parquet", index=False)
    print(f"\n    Processed data saved to {OUTPUT_DIR}/processed_data.parquet")
    print("\n[PART 1 COMPLETE] Proceed to part2_temporal_ml.py")


AID843 A3 - US Road Construction & Closures Analysis

[1] Loading data from: US_Closures.csv
    Shape: (6170627, 47)
    Columns: ['ID', 'Severity', 'Start_Time', 'End_Time', 'Start_Lat', 'Start_Lng', 'End_Lat', 'End_Lng', 'Distance(mi)', 'Description', 'Number', 'Street', 'Side', 'City', 'County', 'State', 'Zipcode', 'Country', 'Timezone', 'Airport_Code', 'Weather_Timestamp', 'Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Direction', 'Wind_Speed(mph)', 'Precipitation(in)', 'Weather_Condition', 'Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit', 'Railway', 'Roundabout', 'Station', 'Stop', 'Traffic_Calming', 'Traffic_Signal', 'Turning_Loop', 'Sunrise_Sunset', 'Civil_Twilight', 'Nautical_Twilight', 'Astronomical_Twilight']

[2] Preprocessing...
    Clean shape: (1644242, 59)
    Date range: 2016-02-04 10:36:16 to 2021-12-31 23:30:30
    States: 50

[3] Exploratory Data Analysis...

    Summary Statistics:
       Duration_hrs  Dist

---
## Part 2 — Temporal Analysis & Machine Learning
Time series decomposition, ADF stationarity tests, ARIMA forecasting, ETS models, SARIMA, Prophet, LSTM (temporal), temporal classification and regression.

In [3]:
"""
AID843 A3 - Part 2: Temporal Analysis & Machine Learning
US Road Construction and Closures (2016-2021)

Covers:
- Time Series decomposition (trend, seasonality, residual)
- Stationarity tests (ADF)
- ARIMA model for construction frequency forecasting
- Exponential smoothing (ETS)
- TS Regression (classification of closure type)
- Temporal predictive models
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import os

from scipy import stats
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score, mean_squared_error, r2_score)
from sklearn.preprocessing import LabelEncoder, StandardScaler

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

OUTPUT_DIR = "outputs"
COLORS = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd',
          '#8c564b','#e377c2','#7f7f7f','#bcbd22','#17becf']
plt.style.use('seaborn-v0_8-whitegrid')

print("="*70)
print("PART 2: TEMPORAL ANALYSIS & MACHINE LEARNING")
print("="*70)

# ============================================================
# LOAD DATA
# ============================================================
def load_processed():
    if not os.path.exists(f"{OUTPUT_DIR}/processed_data.parquet"):
        raise FileNotFoundError(
            "\n\n❌  processed_data.parquet not found.\n"
            "    Run Part 1 cell first before running Part 2."
        )
    df = pd.read_parquet(f"{OUTPUT_DIR}/processed_data.parquet")
    print(f"[Load] Loaded processed data: {df.shape}")
    return df

# ============================================================
# TIME SERIES CONSTRUCTION: DAILY EVENT COUNTS
# ============================================================
def build_time_series(df):
    print("\n[T1] Building time series (daily event counts)...")
    df['Date'] = df['Start_Time'].dt.date
    daily = df.groupby('Date').size().reset_index(name='EventCount')
    daily['Date'] = pd.to_datetime(daily['Date'])
    daily = daily.set_index('Date').asfreq('D', fill_value=0)
    print(f"     Daily TS length: {len(daily)} days")

    # Monthly aggregation
    monthly = daily['EventCount'].resample('MS').sum()
    print(f"     Monthly TS length: {len(monthly)} months")
    return daily, monthly

# ============================================================
# TIME SERIES VISUALIZATION
# ============================================================
def plot_time_series(daily, monthly):
    print("\n[T2] Plotting time series...")
    fig, axes = plt.subplots(3, 1, figsize=(16, 12))
    fig.suptitle('Time Series Analysis of Road Construction Events',
                 fontsize=15, fontweight='bold')

    # Daily
    axes[0].plot(daily.index, daily['EventCount'], color=COLORS[0],
                 linewidth=0.8, alpha=0.8)
    axes[0].set_title('Daily Event Count')
    axes[0].set_ylabel('Events/Day')

    # Monthly with 3-month rolling mean
    axes[1].plot(monthly.index, monthly.values, color=COLORS[1],
                 linewidth=1.5, label='Monthly count', alpha=0.9)
    rolling = monthly.rolling(3, center=True).mean()
    axes[1].plot(monthly.index, rolling.values, color=COLORS[3],
                 linewidth=2.5, label='3-month MA', linestyle='--')
    axes[1].set_title('Monthly Event Count with 3-Month Moving Average')
    axes[1].set_ylabel('Events/Month')
    axes[1].legend()

    # Year-over-year comparison
    monthly_df = monthly.reset_index()
    monthly_df.columns = ['Date','Count']
    monthly_df['Year'] = monthly_df['Date'].dt.year
    monthly_df['Month'] = monthly_df['Date'].dt.month
    for yr, grp in monthly_df.groupby('Year'):
        axes[2].plot(grp['Month'], grp['Count'], marker='o',
                     label=str(yr), linewidth=1.5, markersize=4)
    axes[2].set_title('Year-over-Year Monthly Pattern')
    axes[2].set_xlabel('Month')
    axes[2].set_ylabel('Events/Month')
    axes[2].set_xticks(range(1,13))
    axes[2].set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
    axes[2].legend(title='Year', ncol=3)

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/temporal/fig_t1_time_series.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("     [Fig T1] Time series saved.")

# ============================================================
# TIME SERIES DECOMPOSITION
# ============================================================
def ts_decomposition(monthly):
    print("\n[T3] Time Series Decomposition (STL)...")

    if len(monthly) < 24:
        print("     Warning: Series too short for proper decomposition. Padding...")
        # Already handled by synthetic data

    try:
        decomp = seasonal_decompose(monthly, model='additive', period=12)
        fig, axes = plt.subplots(4, 1, figsize=(16, 12))
        fig.suptitle('Seasonal Decomposition of Monthly Construction Events\n(Additive Model, Period=12)',
                     fontsize=14, fontweight='bold')

        decomp.observed.plot(ax=axes[0], color=COLORS[0])
        axes[0].set_title('Observed'); axes[0].set_ylabel('Count')

        decomp.trend.plot(ax=axes[1], color=COLORS[1])
        axes[1].set_title('Trend Component'); axes[1].set_ylabel('Count')

        decomp.seasonal.plot(ax=axes[2], color=COLORS[2])
        axes[2].set_title('Seasonal Component'); axes[2].set_ylabel('Count')

        decomp.resid.plot(ax=axes[3], color=COLORS[3], marker='o', markersize=3)
        axes[3].axhline(0, ls='--', color='gray')
        axes[3].set_title('Residual Component'); axes[3].set_ylabel('Residual')

        plt.tight_layout()
        plt.savefig(f"{OUTPUT_DIR}/temporal/fig_t2_decomposition.png", dpi=150, bbox_inches='tight')
        plt.close()
        print("     [Fig T2] Decomposition saved.")
        return decomp
    except Exception as e:
        print(f"     Decomposition error: {e}")
        return None

# ============================================================
# STATIONARITY TEST (ADF)
# ============================================================
def stationarity_test(monthly):
    print("\n[T4] Augmented Dickey-Fuller (ADF) Stationarity Test...")

    def run_adf(series, name):
        result = adfuller(series.dropna(), autolag='AIC')
        return {
            'Series': name,
            'ADF Statistic': round(result[0], 4),
            'p-value': round(result[1], 4),
            'Lags Used': result[2],
            'Observations': result[3],
            'Critical 1%': round(result[4]['1%'], 4),
            'Critical 5%': round(result[4]['5%'], 4),
            'Critical 10%': round(result[4]['10%'], 4),
            'Is Stationary (5%)': 'YES' if result[1] < 0.05 else 'NO'
        }

    results = []
    results.append(run_adf(monthly, 'Monthly Events (Original)'))
    diff1 = monthly.diff().dropna()
    results.append(run_adf(diff1, 'Monthly Events (1st Difference)'))
    diff2 = monthly.diff().diff().dropna()
    results.append(run_adf(diff2, 'Monthly Events (2nd Difference)'))

    adf_df = pd.DataFrame(results)
    adf_df.to_csv(f"{OUTPUT_DIR}/tables/adf_test_results.csv", index=False)
    print("\n     ADF Test Results:")
    print(adf_df.to_string(index=False))

    # ACF/PACF plots
    fig, axes = plt.subplots(2, 2, figsize=(16, 8))
    fig.suptitle('ACF and PACF Analysis for ARIMA Model Order Selection',
                 fontsize=13, fontweight='bold')

    plot_acf(monthly.dropna(), lags=min(24, len(monthly)//2), ax=axes[0,0])
    axes[0,0].set_title('ACF - Original Series')

    plot_pacf(monthly.dropna(), lags=min(24, len(monthly)//2-1), ax=axes[0,1])
    axes[0,1].set_title('PACF - Original Series')

    plot_acf(diff1.dropna(), lags=min(24, len(diff1)//2), ax=axes[1,0])
    axes[1,0].set_title('ACF - 1st Differenced Series')

    plot_pacf(diff1.dropna(), lags=min(24, len(diff1)//2-1), ax=axes[1,1])
    axes[1,1].set_title('PACF - 1st Differenced Series')

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/temporal/fig_t3_acf_pacf.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("     [Fig T3] ACF/PACF plots saved.")

    return adf_df

# ============================================================
# ARIMA MODEL
# ============================================================
def arima_model(monthly):
    print("\n[T5] ARIMA Model for Construction Event Forecasting...")

    train_size = int(len(monthly) * 0.80)
    train, test = monthly[:train_size], monthly[train_size:]
    print(f"     Train: {len(train)} months | Test: {len(test)} months")

    results_list = []
    best_model = None
    best_aic = np.inf
    best_order = None

    # Grid search over ARIMA orders
    candidate_orders = [(1,1,1),(1,1,2),(2,1,1),(2,1,2),(1,0,1),(0,1,1),(1,1,0)]
    print("     Grid search ARIMA orders...")
    for order in candidate_orders:
        try:
            m = ARIMA(train, order=order).fit()
            aic = m.aic
            bic = m.bic
            results_list.append({'Order': str(order), 'AIC': round(aic,2), 'BIC': round(bic,2)})
            if aic < best_aic:
                best_aic = aic
                best_model = m
                best_order = order
        except:
            continue

    arima_results_df = pd.DataFrame(results_list).sort_values('AIC')
    arima_results_df.to_csv(f"{OUTPUT_DIR}/tables/arima_model_selection.csv", index=False)
    print(f"\n     Best ARIMA order: {best_order} (AIC={best_aic:.2f})")
    print(arima_results_df.to_string(index=False))

    # Forecast
    forecast_steps = len(test)
    forecast = best_model.forecast(steps=forecast_steps)
    forecast_idx = test.index

    # Metrics
    rmse = np.sqrt(mean_squared_error(test.values, forecast.values))
    mape = np.mean(np.abs((test.values - forecast.values) / (test.values + 1e-8))) * 100
    r2   = r2_score(test.values, forecast.values)
    print(f"\n     ARIMA Test Metrics: RMSE={rmse:.1f}, MAPE={mape:.2f}%, R²={r2:.4f}")

    metrics_df = pd.DataFrame([{
        'Model': f'ARIMA{best_order}',
        'RMSE': round(rmse, 2),
        'MAPE (%)': round(mape, 2),
        'R²': round(r2, 4)
    }])
    metrics_df.to_csv(f"{OUTPUT_DIR}/tables/arima_metrics.csv", index=False)

    # Plot
    fig, axes = plt.subplots(2, 1, figsize=(16, 10))
    fig.suptitle(f'ARIMA{best_order} Forecasting of Monthly Construction Events',
                 fontsize=14, fontweight='bold')

    axes[0].plot(train.index, train.values, color=COLORS[0], label='Training Data', linewidth=1.5)
    axes[0].plot(test.index, test.values, color=COLORS[2], label='Actual (Test)', linewidth=2)
    axes[0].plot(forecast_idx, forecast.values, color=COLORS[3],
                 label=f'ARIMA{best_order} Forecast', linewidth=2, linestyle='--')
    axes[0].axvline(x=train.index[-1], color='gray', linestyle=':', label='Train/Test Split')
    axes[0].set_title('Actual vs Forecast')
    axes[0].set_ylabel('Monthly Event Count')
    axes[0].legend()

    # Residuals
    residuals = pd.Series(best_model.resid)
    axes[1].plot(residuals, color=COLORS[4], alpha=0.8, linewidth=1)
    axes[1].axhline(0, color='gray', linestyle='--')
    axes[1].fill_between(residuals.index, residuals, 0, alpha=0.3, color=COLORS[4])
    axes[1].set_title(f'ARIMA Residuals | RMSE={rmse:.1f} | R²={r2:.4f}')
    axes[1].set_ylabel('Residual')

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/temporal/fig_t4_arima_forecast.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("     [Fig T4] ARIMA forecast saved.")
    return best_model, metrics_df

# ============================================================
# EXPONENTIAL SMOOTHING (ETS)
# ============================================================
def ets_model(monthly):
    print("\n[T6] Exponential Smoothing (ETS Models)...")

    train_size = int(len(monthly) * 0.80)
    train, test = monthly[:train_size], monthly[train_size:]

    ets_results = []
    models = {
        'Simple ES (α)': {'trend': None, 'seasonal': None},
        'Holt (trend)':  {'trend': 'add', 'seasonal': None},
        'Holt-Winters (add)': {'trend': 'add', 'seasonal': 'add',
                                'seasonal_periods': 12},
        'Holt-Winters (mul)': {'trend': 'add', 'seasonal': 'mul',
                                'seasonal_periods': 12}
    }

    forecasts = {}
    for name, params in models.items():
        try:
            kwargs = {'trend': params.get('trend'),
                      'seasonal': params.get('seasonal')}
            if params.get('seasonal_periods'):
                kwargs['seasonal_periods'] = params['seasonal_periods']
            m = ExponentialSmoothing(train, **kwargs,
                                      initialization_method='estimated').fit(optimized=True)
            fc = m.forecast(len(test))
            rmse = np.sqrt(mean_squared_error(test.values, fc.values))
            mape = np.mean(np.abs((test.values - fc.values) / (test.values + 1e-8))) * 100
            r2 = r2_score(test.values, fc.values)
            ets_results.append({'Model': name, 'RMSE': round(rmse,2),
                                  'MAPE (%)': round(mape,2), 'R²': round(r2,4)})
            forecasts[name] = fc
            print(f"     {name}: RMSE={rmse:.1f}, MAPE={mape:.2f}%, R²={r2:.4f}")
        except Exception as e:
            print(f"     {name}: Failed - {e}")

    ets_df = pd.DataFrame(ets_results)
    ets_df.to_csv(f"{OUTPUT_DIR}/tables/ets_model_comparison.csv", index=False)

    # Plot
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle('ETS Model Forecasts vs Actual', fontsize=14, fontweight='bold')

    for ax, (name, fc) in zip(axes.flatten(), forecasts.items()):
        ax.plot(train.index, train.values, color=COLORS[0], label='Train', linewidth=1.5)
        ax.plot(test.index, test.values, color=COLORS[2], label='Actual', linewidth=2)
        ax.plot(test.index, fc.values, color=COLORS[3], label='Forecast',
                linewidth=2, linestyle='--')
        ax.axvline(x=train.index[-1], color='gray', linestyle=':', alpha=0.7)
        ax.set_title(name); ax.legend(fontsize=8)
        ax.set_ylabel('Events/Month')

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/temporal/fig_t5_ets_models.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("     [Fig T5] ETS models saved.")

    # Comparison table plot
    if len(ets_df) > 0:
        fig, ax = plt.subplots(figsize=(10, 4))
        x = np.arange(len(ets_df))
        width = 0.25
        ax.bar(x - width, ets_df['RMSE'], width, label='RMSE', color=COLORS[0])
        ax2 = ax.twinx()
        ax2.bar(x, ets_df['MAPE (%)'], width, label='MAPE (%)', color=COLORS[1])
        ax.bar(x + width, ets_df['R²'] * 1000, width, label='R²×1000', color=COLORS[2])
        ax.set_xticks(x)
        ax.set_xticklabels(ets_df['Model'], rotation=20, ha='right')
        ax.set_title('ETS Model Comparison', fontsize=13, fontweight='bold')
        ax.set_ylabel('RMSE / R²×1000')
        ax2.set_ylabel('MAPE (%)')
        fig.legend(loc='upper right', bbox_to_anchor=(0.95, 0.95))
        plt.tight_layout()
        plt.savefig(f"{OUTPUT_DIR}/temporal/fig_t6_ets_comparison.png", dpi=150, bbox_inches='tight')
        plt.close()
    print("     [Fig T6] ETS comparison saved.")
    return ets_df

# ============================================================
# TEMPORAL CLASSIFICATION (Closure Type Prediction)
# ============================================================
def temporal_classification(df):
    print("\n[T7] Temporal Classification: Predicting Closure Type...")

    # Features: temporal only
    temporal_features = [c for c in ['Hour_sin','Hour_cos','Month_sin','Month_cos',
                                       'DOW_sin','DOW_cos','Week_sin','Week_cos',
                                       'IsWeekend','IsRushHour','IsNight',
                                       'Log_Duration','Year'] if c in df.columns]

    target = 'Type' if 'Type' in df.columns else 'Severity'
    if target not in df.columns:
        print("     Target column not found, skipping.")
        return None

    data = df[temporal_features + [target]].dropna()
    if len(data) > 100000:
        data = data.sample(100000, random_state=RANDOM_SEED)
    le = LabelEncoder()
    y = le.fit_transform(data[target].astype(str))
    X = data[temporal_features]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y)
    print(f"     Train: {len(X_train):,} | Test: {len(X_test):,}")
    print(f"     Classes: {le.classes_}")

    models = {
        'Logistic Regression': LogisticRegression(max_iter=500, random_state=RANDOM_SEED,
                                                   class_weight='balanced'),
        'Decision Tree': DecisionTreeClassifier(max_depth=8, random_state=RANDOM_SEED,
                                                 class_weight='balanced'),
        'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10,
                                                 random_state=RANDOM_SEED,
                                                 n_jobs=-1, class_weight='balanced'),
        'Gradient Boosting': HistGradientBoostingClassifier(max_iter=100, max_depth=5,
                                                             random_state=RANDOM_SEED)
    }

    results = []
    all_reports = {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
        f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)

        # CV score
        idx = np.random.choice(len(X_scaled), min(50000, len(X_scaled)), replace=False)
        cv_scores = cross_val_score(model, X_scaled[idx], y[idx], cv=3, scoring='f1_macro', n_jobs=-1)

        results.append({
            'Model': name,
            'Accuracy': round(acc, 4),
            'F1 Macro': round(f1_macro, 4),
            'F1 Weighted': round(f1_weighted, 4),
            'CV F1 Mean': round(cv_scores.mean(), 4),
            'CV F1 Std': round(cv_scores.std(), 4)
        })
        all_reports[name] = (model, y_pred, y_test)
        print(f"     {name}: Acc={acc:.4f}, F1={f1_macro:.4f}, CV={cv_scores.mean():.4f}±{cv_scores.std():.4f}")

    results_df = pd.DataFrame(results).sort_values('F1 Macro', ascending=False)
    results_df.to_csv(f"{OUTPUT_DIR}/tables/temporal_classification_results.csv", index=False)

    # Plot: model comparison
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Temporal Classification Model Performance (Closure Type Prediction)',
                 fontsize=14, fontweight='bold')

    metrics = ['Accuracy','F1 Macro','F1 Weighted','CV F1 Mean']
    x = np.arange(len(results_df))
    width = 0.2
    for i, m in enumerate(metrics):
        axes[0].bar(x + i*width, results_df[m], width, label=m, color=COLORS[i])
    axes[0].set_xticks(x + width*1.5)
    axes[0].set_xticklabels(results_df['Model'], rotation=15, ha='right', fontsize=9)
    axes[0].set_title('Model Metrics Comparison')
    axes[0].set_ylabel('Score')
    axes[0].legend(fontsize=8)
    axes[0].set_ylim(0, 1.1)

    # Confusion matrix for best model
    best_name = results_df.iloc[0]['Model']
    _, best_pred, y_test_arr = all_reports[best_name]
    cm = confusion_matrix(y_test_arr, best_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
                xticklabels=le.classes_, yticklabels=le.classes_)
    axes[1].set_title(f'Confusion Matrix: {best_name}')
    axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/temporal/fig_t7_classification.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("     [Fig T7] Temporal classification results saved.")

    # Feature importance (for tree-based models)
    best_model_obj = all_reports[best_name][0]
    if hasattr(best_model_obj, 'feature_importances_'):
        imp = pd.Series(best_model_obj.feature_importances_, index=temporal_features)
        imp = imp.sort_values(ascending=False)
        fig, ax = plt.subplots(figsize=(10, 5))
        imp.plot(kind='bar', ax=ax, color=COLORS[0])
        ax.set_title(f'Feature Importances - {best_name}\n(Temporal Features for Closure Type)',
                     fontsize=12, fontweight='bold')
        ax.set_xlabel('Feature'); ax.set_ylabel('Importance')
        plt.tight_layout()
        plt.savefig(f"{OUTPUT_DIR}/temporal/fig_t8_feature_importance.png", dpi=150, bbox_inches='tight')
        plt.close()
        print("     [Fig T8] Feature importances saved.")

    print("\n     Temporal Classification Summary:")
    print(results_df.to_string(index=False))
    return results_df

# ============================================================
# DURATION REGRESSION (Temporal)
# ============================================================
def temporal_regression(df):
    print("\n[T8] Temporal Regression: Predicting Event Duration...")

    from sklearn.ensemble import RandomForestRegressor
    from sklearn.linear_model import Ridge

    regression_features = [c for c in ['Hour_sin','Hour_cos','Month_sin','Month_cos',
                                        'DOW_sin','DOW_cos','IsWeekend','IsRushHour',
                                        'IsNight','Year','StateMonthCount']
                           if c in df.columns]

    target = 'Log_Duration'
    if target not in df.columns:
        print("     Log_Duration not available.")
        return

    data = df[regression_features + [target]].dropna()
    X = data[regression_features]
    y = data[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_SEED)

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)

    reg_models = {
        'Ridge Regression': Ridge(alpha=1.0),
        'Random Forest Regressor': RandomForestRegressor(
            n_estimators=100, max_depth=10, random_state=RANDOM_SEED, n_jobs=-1)
    }

    reg_results = []
    for name, m in reg_models.items():
        m.fit(X_train_s, y_train)
        y_pred = m.predict(X_test_s)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2   = r2_score(y_test, y_pred)
        reg_results.append({'Model': name, 'RMSE (log scale)': round(rmse,4), 'R²': round(r2,4)})
        print(f"     {name}: RMSE={rmse:.4f}, R²={r2:.4f}")

    reg_df = pd.DataFrame(reg_results)
    reg_df.to_csv(f"{OUTPUT_DIR}/tables/temporal_regression_results.csv", index=False)

    # Plot: actual vs predicted for best model
    best_model = RandomForestRegressor(n_estimators=100, max_depth=10,
                                        random_state=RANDOM_SEED, n_jobs=-1)
    best_model.fit(X_train_s, y_train)
    y_pred_best = best_model.predict(X_test_s)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Duration Regression (Random Forest) - Temporal Features',
                 fontsize=13, fontweight='bold')

    sample_idx = np.random.choice(len(y_test), min(2000, len(y_test)), replace=False)
    axes[0].scatter(y_test.iloc[sample_idx], y_pred_best[sample_idx],
                    alpha=0.3, s=5, color=COLORS[0])
    mn = min(y_test.min(), y_pred_best.min())
    mx = max(y_test.max(), y_pred_best.max())
    axes[0].plot([mn,mx],[mn,mx], 'r--', label='Perfect fit')
    axes[0].set_xlabel('Actual log(Duration)'); axes[0].set_ylabel('Predicted log(Duration)')
    axes[0].set_title(f"Actual vs Predicted (R²={reg_df.iloc[-1]['R²']:.4f})")
    axes[0].legend()

    # Residual distribution
    residuals = y_pred_best - y_test.values
    axes[1].hist(residuals, bins=60, color=COLORS[1], alpha=0.8, edgecolor='white')
    axes[1].axvline(0, color='red', linestyle='--')
    axes[1].set_title('Residual Distribution')
    axes[1].set_xlabel('Residual'); axes[1].set_ylabel('Count')

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/temporal/fig_t9_regression.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("     [Fig T9] Duration regression saved.")

    print("\n     Temporal Regression Summary:")
    print(reg_df.to_string(index=False))
    return reg_df

# ============================================================
# HEATMAP: CONSTRUCTION INTENSITY
# ============================================================
def temporal_heatmap(df):
    print("\n[T9] Temporal Heatmap: Hour × Day of Week intensity...")
    pivot = df.groupby(['DayOfWeek','Hour']).size().reset_index(name='count')
    pivot_table = pivot.pivot(index='DayOfWeek', columns='Hour', values='count').fillna(0)
    pivot_table.index = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

    fig, ax = plt.subplots(figsize=(16, 5))
    sns.heatmap(pivot_table, ax=ax, cmap='YlOrRd', cbar_kws={'label': 'Event Count'},
                linewidths=0.1)
    ax.set_title('Construction Event Intensity: Hour × Day of Week',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Hour of Day (UTC)'); ax.set_ylabel('Day of Week')
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/temporal/fig_t10_heatmap_hourXdow.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("     [Fig T10] Hour×DOW heatmap saved.")

    # Month × Year heatmap
    pivot2 = df.groupby(['Year','Month']).size().reset_index(name='count')
    pivot2_table = pivot2.pivot(index='Year', columns='Month', values='count').fillna(0)
    pivot2_table.columns = ['Jan','Feb','Mar','Apr','May','Jun',
                             'Jul','Aug','Sep','Oct','Nov','Dec']

    fig, ax = plt.subplots(figsize=(14, 5))
    sns.heatmap(pivot2_table, ax=ax, cmap='Blues', annot=True, fmt='.0f',
                cbar_kws={'label': 'Event Count'})
    ax.set_title('Construction Event Count: Year × Month Heatmap',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/temporal/fig_t11_heatmap_yearXmonth.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("     [Fig T11] Year×Month heatmap saved.")


# ============================================================
# SARIMA MODEL
# ============================================================
def sarima_model(monthly):
    """SARIMA: Seasonal ARIMA for construction event forecasting."""
    print("\n[T7] SARIMA Model for Construction Event Forecasting...")
    from statsmodels.tsa.statespace.sarimax import SARIMAX

    if monthly is None or len(monthly) < 24:
        print("     Not enough data for SARIMA (need >= 24 months).")
        return None

    train = monthly.iloc[:-6]
    test  = monthly.iloc[-6:]

    # Fit SARIMA(1,1,1)(1,1,0,12) — reasonable seasonal order for monthly data
    try:
        sarima_model_fit = SARIMAX(
            train,
            order=(1, 1, 1),
            seasonal_order=(1, 1, 0, 12),
            enforce_stationarity=False,
            enforce_invertibility=False
        ).fit(disp=False)
    except Exception as e:
        print(f"     SARIMA fit failed: {e}")
        return None

    forecast = sarima_model_fit.get_forecast(steps=len(test))
    pred     = forecast.predicted_mean
    ci       = forecast.conf_int()

    rmse = np.sqrt(((pred.values - test.values)**2).mean())
    ss_res = ((test.values - pred.values)**2).sum()
    ss_tot = ((test.values - test.values.mean())**2).sum()
    r2 = 1 - ss_res / ss_tot if ss_tot != 0 else 0
    mape = (np.abs((test.values - pred.values) / (test.values + 1e-8)).mean()) * 100

    print(f"     SARIMA(1,1,1)(1,1,0,12) — RMSE={rmse:.1f}, MAPE={mape:.2f}%, R²={r2:.4f}")

    sarima_metrics = pd.DataFrame([{
        'Model': 'SARIMA(1,1,1)(1,1,0,12)',
        'RMSE': round(rmse, 2),
        'MAPE(%)': round(mape, 2),
        'R2': round(r2, 4)
    }])
    sarima_metrics.to_csv(f"{OUTPUT_DIR}/tables/sarima_metrics.csv", index=False)

    # Figure
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('SARIMA(1,1,1)(1,1,0,12) — Seasonal Forecasting', fontsize=14, fontweight='bold')

    axes[0].plot(train.index, train.values, label='Train', color=COLORS[0], linewidth=1.5)
    axes[0].plot(test.index,  test.values,  label='Actual', color=COLORS[2], linewidth=2)
    axes[0].plot(pred.index,  pred.values,  label='SARIMA Forecast',
                 color=COLORS[1], linewidth=2, linestyle='--')
    axes[0].fill_between(pred.index, ci.iloc[:, 0], ci.iloc[:, 1], alpha=0.2, color=COLORS[1])
    axes[0].set_title(f'SARIMA Forecast | RMSE={rmse:.1f}')
    axes[0].set_xlabel('Date'); axes[0].set_ylabel('Monthly Events')
    axes[0].legend()

    residuals = train.values - sarima_model_fit.fittedvalues.values
    axes[1].plot(train.index, residuals, color=COLORS[3], linewidth=1)
    axes[1].axhline(0, color='gray', linestyle='--')
    axes[1].set_title('SARIMA Residuals (Training)')
    axes[1].set_xlabel('Date'); axes[1].set_ylabel('Residual')

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/temporal/fig_T7_sarima.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("     [Fig T7] SARIMA forecast saved.")
    return sarima_metrics


# ============================================================
# PROPHET MODEL
# ============================================================
def prophet_model(monthly):
    """Facebook Prophet for construction event forecasting."""
    print("\n[T8] Prophet Model for Construction Event Forecasting...")
    try:
        from prophet import Prophet
    except ImportError:
        try:
            import subprocess, sys
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'prophet', '-q'])
            from prophet import Prophet
        except Exception as e:
            print(f"     Prophet not available: {e}. Skipping.")
            return None

    if monthly is None or len(monthly) < 12:
        print("     Not enough data for Prophet.")
        return None

    # Prophet requires a DataFrame with ds and y columns
    prophet_df = pd.DataFrame({
        'ds': monthly.index,
        'y':  monthly.values
    }).reset_index(drop=True)

    train_df = prophet_df.iloc[:-6].copy()
    test_df  = prophet_df.iloc[-6:].copy()

    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        seasonality_mode='multiplicative',
        interval_width=0.95
    )
    model.fit(train_df)

    # Forecast
    future   = model.make_future_dataframe(periods=6, freq='MS')
    forecast  = model.predict(future)
    forecast_test = forecast[forecast['ds'].isin(test_df['ds'])][['ds','yhat','yhat_lower','yhat_upper']]

    actual = test_df['y'].values
    predicted = forecast_test['yhat'].values[:len(actual)]

    rmse = np.sqrt(((actual - predicted)**2).mean())
    ss_res = ((actual - predicted)**2).sum()
    ss_tot = ((actual - actual.mean())**2).sum()
    r2   = 1 - ss_res / ss_tot if ss_tot != 0 else 0
    mape = (np.abs((actual - predicted) / (actual + 1e-8)).mean()) * 100

    print(f"     Prophet — RMSE={rmse:.1f}, MAPE={mape:.2f}%, R²={r2:.4f}")

    prophet_metrics = pd.DataFrame([{
        'Model': 'Prophet (multiplicative)',
        'RMSE': round(rmse, 2),
        'MAPE(%)': round(mape, 2),
        'R2': round(r2, 4)
    }])
    prophet_metrics.to_csv(f"{OUTPUT_DIR}/tables/prophet_metrics.csv", index=False)

    # Figure
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Prophet — Multiplicative Seasonal Forecasting', fontsize=14, fontweight='bold')

    axes[0].plot(prophet_df['ds'], prophet_df['y'], label='Actual', color=COLORS[0], linewidth=1.5)
    axes[0].plot(forecast_test['ds'], forecast_test['yhat'],
                 label='Prophet Forecast', color=COLORS[1], linewidth=2, linestyle='--')
    axes[0].fill_between(forecast_test['ds'],
                          forecast_test['yhat_lower'], forecast_test['yhat_upper'],
                          alpha=0.2, color=COLORS[1])
    axes[0].set_title(f'Prophet Forecast | RMSE={rmse:.1f} | R²={r2:.4f}')
    axes[0].set_xlabel('Date'); axes[0].set_ylabel('Monthly Events')
    axes[0].legend()

    # Component plot (trend)
    full_forecast = model.predict(future)
    axes[1].plot(full_forecast['ds'], full_forecast['trend'],
                 color=COLORS[2], linewidth=2, label='Trend')
    axes[1].set_title('Prophet Trend Component')
    axes[1].set_xlabel('Date'); axes[1].set_ylabel('Trend')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/temporal/fig_T8_prophet.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("     [Fig T8] Prophet forecast saved.")
    return prophet_metrics


# ============================================================
# LSTM MODEL (Temporal)
# ============================================================
def lstm_temporal_model(monthly):
    """LSTM neural network for construction event time series forecasting."""
    print("\n[T9] LSTM Temporal Model for Construction Event Forecasting...")
    try:
        import tensorflow as tf
        from tensorflow.keras.models import Sequential
        from tensorflow.keras.layers import LSTM, Dense, Dropout
        from tensorflow.keras.callbacks import EarlyStopping
    except ImportError:
        try:
            import subprocess, sys
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'tensorflow', '-q'])
            import tensorflow as tf
            from tensorflow.keras.models import Sequential
            from tensorflow.keras.layers import LSTM, Dense, Dropout
            from tensorflow.keras.callbacks import EarlyStopping
        except Exception as e:
            print(f"     TensorFlow not available: {e}. Skipping LSTM.")
            return None

    if monthly is None or len(monthly) < 18:
        print("     Not enough data for LSTM (need >= 18 months).")
        return None

    # Prepare sequences
    scaler_lstm = MinMaxScaler()
    values = scaler_lstm.fit_transform(monthly.values.reshape(-1, 1))

    LOOK_BACK = 6
    X_all, y_all = [], []
    for i in range(LOOK_BACK, len(values)):
        X_all.append(values[i - LOOK_BACK:i, 0])
        y_all.append(values[i, 0])
    X_all = np.array(X_all).reshape(-1, LOOK_BACK, 1)
    y_all = np.array(y_all)

    split = max(1, len(X_all) - 6)
    X_tr, X_te = X_all[:split], X_all[split:]
    y_tr, y_te = y_all[:split], y_all[split:]

    if len(X_tr) < 2:
        print("     Not enough training samples for LSTM.")
        return None

    # Build LSTM
    tf.random.set_seed(RANDOM_SEED)
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(LOOK_BACK, 1)),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')

    es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    history = model.fit(
        X_tr, y_tr,
        epochs=100,
        batch_size=max(1, len(X_tr) // 4),
        validation_split=0.15,
        callbacks=[es],
        verbose=0
    )

    pred_scaled = model.predict(X_te, verbose=0)
    pred = scaler_lstm.inverse_transform(pred_scaled.reshape(-1, 1)).flatten()
    actual = scaler_lstm.inverse_transform(y_te.reshape(-1, 1)).flatten()

    rmse = np.sqrt(((actual - pred)**2).mean())
    ss_res = ((actual - pred)**2).sum()
    ss_tot = ((actual - actual.mean())**2).sum()
    r2   = 1 - ss_res / ss_tot if ss_tot != 0 else 0
    mape = (np.abs((actual - pred) / (actual + 1e-8)).mean()) * 100

    print(f"     LSTM (temporal) — RMSE={rmse:.1f}, MAPE={mape:.2f}%, R²={r2:.4f}")

    lstm_t_metrics = pd.DataFrame([{
        'Model': 'LSTM (temporal, look_back=6)',
        'RMSE': round(rmse, 2),
        'MAPE(%)': round(mape, 2),
        'R2': round(r2, 4)
    }])
    lstm_t_metrics.to_csv(f"{OUTPUT_DIR}/tables/lstm_temporal_metrics.csv", index=False)

    # Figure
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('LSTM Temporal Model — Construction Event Forecasting',
                 fontsize=14, fontweight='bold')

    train_pred_scaled = model.predict(X_tr, verbose=0)
    train_pred = scaler_lstm.inverse_transform(train_pred_scaled.reshape(-1, 1)).flatten()
    train_actual = scaler_lstm.inverse_transform(y_tr.reshape(-1, 1)).flatten()

    axes[0].plot(range(len(train_actual)), train_actual,
                 label='Train Actual', color=COLORS[0], linewidth=1.5)
    axes[0].plot(range(len(train_pred)), train_pred,
                 label='Train Predicted', color=COLORS[1], linewidth=1.5, linestyle='--')
    offset = len(train_actual)
    axes[0].plot(range(offset, offset + len(actual)), actual,
                 label='Test Actual', color=COLORS[2], linewidth=2)
    axes[0].plot(range(offset, offset + len(pred)), pred,
                 label='Test Predicted', color=COLORS[3], linewidth=2, linestyle='--')
    axes[0].set_title(f'LSTM Forecast | RMSE={rmse:.1f} | R²={r2:.4f}')
    axes[0].set_xlabel('Time Step'); axes[0].set_ylabel('Monthly Events')
    axes[0].legend(fontsize=8)

    axes[1].plot(history.history['loss'], label='Train Loss', color=COLORS[0])
    if 'val_loss' in history.history:
        axes[1].plot(history.history['val_loss'], label='Val Loss', color=COLORS[1])
    axes[1].set_title('LSTM Training Loss Curve')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MSE Loss')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/temporal/fig_T9_lstm_temporal.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("     [Fig T9] LSTM temporal forecast saved.")
    return lstm_t_metrics


# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":
    df = load_processed()

    daily, monthly = build_time_series(df)
    plot_time_series(daily, monthly)
    decomp = ts_decomposition(monthly)
    adf_df = stationarity_test(monthly)
    arima_m, arima_metrics = arima_model(monthly)
    ets_df = ets_model(monthly)
    sarima_m = sarima_model(monthly)
    prophet_m = prophet_model(monthly)
    lstm_t = lstm_temporal_model(monthly)
    clf_df = temporal_classification(df)
    reg_df = temporal_regression(df)
    temporal_heatmap(df)

    print("\n[PART 2 COMPLETE] Proceed to part3_spatial_ml.py")


PART 2: TEMPORAL ANALYSIS & MACHINE LEARNING
[Load] Loaded processed data: (1644242, 85)

[T1] Building time series (daily event counts)...
     Daily TS length: 2158 days
     Monthly TS length: 71 months

[T2] Plotting time series...
     [Fig T1] Time series saved.

[T3] Time Series Decomposition (STL)...
     [Fig T2] Decomposition saved.

[T4] Augmented Dickey-Fuller (ADF) Stationarity Test...

     ADF Test Results:
                         Series  ADF Statistic  p-value  Lags Used  Observations  Critical 1%  Critical 5%  Critical 10% Is Stationary (5%)
      Monthly Events (Original)         3.2193   1.0000         10            60      -3.5444      -2.9111       -2.5932                 NO
Monthly Events (1st Difference)        -6.3581   0.0000          2            67      -3.5320      -2.9058       -2.5904                YES
Monthly Events (2nd Difference)        -4.4429   0.0002         10            58      -3.5485      -2.9128       -2.5941                YES
     [Fig T3] 

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/holtwinters/model.py:903: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(


     Holt-Winters (mul): RMSE=29883.1, MAPE=34.83%, R²=-0.7407
     [Fig T5] ETS models saved.
     [Fig T6] ETS comparison saved.

[T7] SARIMA Model for Construction Event Forecasting...
     SARIMA(1,1,1)(1,1,0,12) — RMSE=18293.1, MAPE=14.19%, R²=-0.2410
     [Fig T7] SARIMA forecast saved.

[T8] Prophet Model for Construction Event Forecasting...


18:22:29 - cmdstanpy - INFO - Chain [1] start processing
18:22:30 - cmdstanpy - INFO - Chain [1] done processing


     Prophet — RMSE=25405.1, MAPE=20.62%, R²=-1.3936
     [Fig T8] Prophet forecast saved.

[T9] LSTM Temporal Model for Construction Event Forecasting...


2026-04-29 18:22:34.627858: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777486954.997912      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777486955.092689      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777486956.104455      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777486956.104484      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777486956.104487      57 computation_placer.cc:177] computation placer alr

     LSTM (temporal) — RMSE=28651.8, MAPE=29.33%, R²=-2.0445
     [Fig T9] LSTM temporal forecast saved.

[T7] Temporal Classification: Predicting Closure Type...
     Train: 80,000 | Test: 20,000
     Classes: ['1' '2' '3' '4']
     Logistic Regression: Acc=0.5433, F1=0.3169, CV=0.3187±0.0018
     Decision Tree: Acc=0.6184, F1=0.4164, CV=0.4092±0.0059
     Random Forest: Acc=0.6512, F1=0.4301, CV=0.4387±0.0034
     Gradient Boosting: Acc=0.8758, F1=0.4754, CV=0.4630±0.0042
     [Fig T7] Temporal classification results saved.

     Temporal Classification Summary:
              Model  Accuracy  F1 Macro  F1 Weighted  CV F1 Mean  CV F1 Std
  Gradient Boosting    0.8758    0.4754       0.8546      0.4630     0.0042
      Random Forest    0.6512    0.4301       0.7187      0.4387     0.0034
      Decision Tree    0.6184    0.4164       0.7003      0.4092     0.0059
Logistic Regression    0.5433    0.3169       0.6402      0.3187     0.0018

[T8] Temporal Regression: Predicting Event Durat

---
## Part 3 — Spatial Analysis & Machine Learning
Spatial density maps, KDE, Moran's I autocorrelation, Geary's C, LISA (Local Indicators of Spatial Association), DBSCAN/K-Means clustering, spatial classification, Geographically Weighted Regression.

In [4]:
"""
AID843 A3 - Part 3: Spatial Analysis & Machine Learning
US Road Construction and Closures (2016-2021)

Covers:
- Spatial data visualization (event density maps)
- Kernel Density Estimation (KDE)
- Spatial Autocorrelation (Moran's I, Geary's C, LISA)
- Spatial Clustering (DBSCAN, K-Means on coordinates)
- Spatial Classification (Random Forest with spatial features)
- Geographically Weighted Regression (GWR - proxy)
- Colocation analysis
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import os
from scipy import stats
from scipy.spatial import distance_matrix as scipy_distance_matrix
from scipy.ndimage import gaussian_filter

from sklearn.cluster import DBSCAN, KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (silhouette_score, classification_report, 
                              accuracy_score, f1_score, confusion_matrix)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
OUTPUT_DIR = "outputs"
COLORS = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd',
          '#8c564b','#e377c2','#7f7f7f','#bcbd22','#17becf']
plt.style.use('seaborn-v0_8-whitegrid')

print("="*70)
print("PART 3: SPATIAL ANALYSIS & MACHINE LEARNING")
print("="*70)

# ============================================================
# LOAD
# ============================================================
def load_processed():
    if not os.path.exists(f"{OUTPUT_DIR}/processed_data.parquet"):
        raise FileNotFoundError(
            "\n\n❌  processed_data.parquet not found.\n"
            "    Run Part 1 cell first before running Part 3."
        )
    df = pd.read_parquet(f"{OUTPUT_DIR}/processed_data.parquet")
    print(f"[Load] Shape: {df.shape}")
    return df

# ============================================================
# SPATIAL DENSITY MAP
# ============================================================
def spatial_density_map(df):
    print("\n[S1] Spatial Density Map (Hexbin)...")
    
    # US bounding box
    lat_min, lat_max = 24.5, 49.5
    lon_min, lon_max = -125.0, -66.0
    mask = ((df['Start_Lat'] >= lat_min) & (df['Start_Lat'] <= lat_max) &
            (df['Start_Lng'] >= lon_min) & (df['Start_Lng'] <= lon_max))
    df_map = df[mask].sample(min(50000, len(df[mask])), random_state=42)

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    fig.suptitle('Spatial Distribution of Road Construction Events (USA)',
                 fontsize=14, fontweight='bold')

    # Hexbin density
    hb = axes[0].hexbin(df_map['Start_Lng'], df_map['Start_Lat'],
                         gridsize=80, cmap='hot_r', mincnt=1, bins='log')
    plt.colorbar(hb, ax=axes[0], label='log10(Event Count)')
    axes[0].set_xlim(lon_min, lon_max); axes[0].set_ylim(lat_min, lat_max)
    axes[0].set_title('Hexbin Event Density (log scale)')
    axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')

    # Scatter with severity/type
    type_col = 'Type' if 'Type' in df.columns else 'Severity'
    if type_col in df_map.columns:
        le = LabelEncoder()
        colors_coded = le.fit_transform(df_map[type_col].astype(str).fillna('Unknown'))
        sc = axes[1].scatter(df_map['Start_Lng'], df_map['Start_Lat'],
                              c=colors_coded, cmap='tab10', s=0.5, alpha=0.4)
        legend_elements = [Patch(facecolor=plt.cm.tab10(i/len(le.classes_)),
                                  label=cls) for i, cls in enumerate(le.classes_)]
        axes[1].legend(handles=legend_elements, loc='lower right', fontsize=8,
                       title=type_col)
    else:
        axes[1].scatter(df_map['Start_Lng'], df_map['Start_Lat'],
                        s=0.5, alpha=0.3, color=COLORS[0])
    axes[1].set_xlim(lon_min, lon_max); axes[1].set_ylim(lat_min, lat_max)
    axes[1].set_title(f'Events Colored by {type_col}')
    axes[1].set_xlabel('Longitude'); axes[1].set_ylabel('Latitude')

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/spatial/fig_s1_density_map.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("     [Fig S1] Spatial density map saved.")
    return df_map

# ============================================================
# KERNEL DENSITY ESTIMATION (KDE)
# ============================================================
def kernel_density_estimation(df_map):
    print("\n[S2] Kernel Density Estimation (2D KDE)...")

    lat_min, lat_max = 24.5, 49.5
    lon_min, lon_max = -125.0, -66.0

    # Create grid
    lon_grid = np.linspace(lon_min, lon_max, 200)
    lat_grid = np.linspace(lat_min, lat_max, 150)
    lon_mesh, lat_mesh = np.meshgrid(lon_grid, lat_grid)
    positions = np.vstack([lon_mesh.ravel(), lat_mesh.ravel()])

    # Sample for KDE computation
    sample = df_map.sample(min(10000, len(df_map)), random_state=42)
    kde_input = np.vstack([sample['Start_Lng'], sample['Start_Lat']])
    kde = stats.gaussian_kde(kde_input, bw_method=0.05)
    z = kde(positions).reshape(lat_mesh.shape)

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    fig.suptitle('Kernel Density Estimation of Road Construction Events',
                 fontsize=14, fontweight='bold')

    cf = axes[0].contourf(lon_mesh, lat_mesh, z, levels=20, cmap='YlOrRd')
    plt.colorbar(cf, ax=axes[0], label='Density')
    axes[0].set_title('KDE Density Contour')
    axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')

    # By closure type
    if 'Type' in df_map.columns:
        types = df_map['Type'].unique()[:3]
        cmap_list = ['Blues','Reds','Greens']
        for i, t in enumerate(types):
            subset = df_map[df_map['Type'] == t]
            if len(subset) > 100:
                # Simple 2D histogram as density
                h, xedges, yedges = np.histogram2d(
                    subset['Start_Lng'], subset['Start_Lat'],
                    bins=[80, 60], range=[[lon_min,lon_max],[lat_min,lat_max]])
                h = gaussian_filter(h.T, sigma=2)
                axes[1].contour(np.linspace(lon_min,lon_max,80),
                                 np.linspace(lat_min,lat_max,60),
                                 h, levels=5,
                                 colors=[COLORS[i]], alpha=0.7, linewidths=1.5)
        # Legend
        legend_lines = [plt.Line2D([0],[0],color=COLORS[i],label=t)
                        for i, t in enumerate(types)]
        axes[1].legend(handles=legend_lines, title='Closure Type')
    axes[1].set_xlim(lon_min, lon_max); axes[1].set_ylim(lat_min, lat_max)
    axes[1].set_title('KDE Contours by Closure Type')
    axes[1].set_xlabel('Longitude'); axes[1].set_ylabel('Latitude')

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/spatial/fig_s2_kde.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("     [Fig S2] KDE saved.")

# ============================================================
# SPATIAL AUTOCORRELATION (MORAN'S I)
# ============================================================
def morans_i(df):
    print("\n[S3] Spatial Autocorrelation - Moran's I...")

    # Aggregate to state level
    if 'State' not in df.columns:
        print("     State column not found.")
        return None

    state_stats = df.groupby('State').agg(
        EventCount=('Start_Lat', 'count'),
        AvgLat=('Start_Lat', 'mean'),
        AvgLon=('Start_Lng', 'mean'),
        AvgDuration=('Duration_hrs', 'mean')
    ).reset_index()

    n = len(state_stats)
    if n < 5:
        print("     Not enough states for Moran's I.")
        return None

    # Spatial weights (inverse distance between state centroids)
    coords = state_stats[['AvgLat','AvgLon']].values
    dist_mat = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i != j:
                d = np.sqrt((coords[i,0]-coords[j,0])**2 + (coords[i,1]-coords[j,1])**2)
                dist_mat[i,j] = 1 / (d + 1e-8)  # inverse distance

    # Row-standardize
    row_sums = dist_mat.sum(axis=1, keepdims=True)
    W = dist_mat / (row_sums + 1e-8)

    def compute_morans_i(y, W):
        n = len(y)
        y_bar = y.mean()
        z = y - y_bar
        S0 = W.sum()
        numerator = n * (W * np.outer(z, z)).sum()
        denominator = S0 * (z**2).sum()
        return numerator / denominator if denominator != 0 else 0

    # Monte Carlo for p-value
    def morans_pvalue(y, W, permutations=999):
        observed_I = compute_morans_i(y, W)
        perm_Is = []
        for _ in range(permutations):
            y_perm = np.random.permutation(y)
            perm_Is.append(compute_morans_i(y_perm, W))
        pseudo_p = (np.sum(np.array(perm_Is) >= observed_I) + 1) / (permutations + 1)
        return observed_I, pseudo_p, np.array(perm_Is)

    results_moran = []
    for variable in ['EventCount', 'AvgDuration']:
        if variable in state_stats.columns:
            y = state_stats[variable].values.astype(float)
            I_val, p_val, perm_dist = morans_pvalue(y, W)
            results_moran.append({
                'Variable': variable,
                "Moran's I": round(I_val, 4),
                'p-value (MC)': round(p_val, 4),
                'Interpretation': 'Clustered' if (I_val > 0 and p_val < 0.05)
                                  else 'Dispersed' if (I_val < 0 and p_val < 0.05)
                                  else 'Random'
            })
            print(f"     {variable}: I={I_val:.4f}, p={p_val:.4f}")

    morans_df = pd.DataFrame(results_moran)
    morans_df.to_csv(f"{OUTPUT_DIR}/tables/morans_i_results.csv", index=False)

    # Moran scatter plot
    y = state_stats['EventCount'].values.astype(float)
    y_std = (y - y.mean()) / y.std()
    Wy = W @ y_std

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle("Moran's I Spatial Autocorrelation Analysis", fontsize=14, fontweight='bold')

    axes[0].scatter(y_std, Wy, color=COLORS[0], alpha=0.7, edgecolors='white', s=60)
    slope, intercept = np.polyfit(y_std, Wy, 1)
    x_line = np.linspace(y_std.min(), y_std.max(), 100)
    axes[0].plot(x_line, slope*x_line + intercept, 'r-', linewidth=2)
    axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)
    axes[0].axvline(0, color='gray', linestyle='--', alpha=0.5)
    I_val_main = results_moran[0]["Moran's I"] if results_moran else 0.0
    p_val_main = results_moran[0]['p-value (MC)'] if results_moran else 1.0
    axes[0].set_title(f"Moran Scatter Plot (EventCount)\nI={I_val_main:.4f}, p={p_val_main:.4f}")
    axes[0].set_xlabel('Standardized Event Count (z)'); axes[0].set_ylabel('Spatial Lag (Wz)')
    for i, row in state_stats.iterrows():
        if i < n:
            axes[0].annotate(row['State'], (y_std[i], Wy[i]), fontsize=6, alpha=0.7)

    # Permutation distribution
    I_val2, p_val2, perm_dist2 = morans_pvalue(y, W)
    axes[1].hist(perm_dist2, bins=40, color=COLORS[1], alpha=0.8, edgecolor='white')
    axes[1].axvline(I_val2, color='red', linestyle='--', linewidth=2,
                     label=f"Observed I={I_val2:.4f}")
    axes[1].set_title("Monte Carlo Permutation Distribution")
    axes[1].set_xlabel("Moran's I (permuted)"); axes[1].set_ylabel("Frequency")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/spatial/fig_s3_morans_i.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("     [Fig S3] Moran's I analysis saved.")
    print(morans_df.to_string(index=False))
    return morans_df, state_stats


# ============================================================
# GEARY'S C SPATIAL AUTOCORRELATION
# ============================================================
def gearys_c(df):
    """Geary's C statistic — complementary to Moran's I for local variation."""
    print("\n[S3b] Geary's C Spatial Autocorrelation...")

    if 'State' not in df.columns:
        print("     State column not found.")
        return None

    state_stats = df.groupby('State').agg(
        EventCount=('Start_Lat', 'count'),
        AvgLat=('Start_Lat', 'mean'),
        AvgLon=('Start_Lng', 'mean'),
        AvgDuration=('Duration_hrs', 'mean')
    ).reset_index()

    n = len(state_stats)
    if n < 5:
        print("     Not enough states for Geary's C.")
        return None

    coords = state_stats[['AvgLat', 'AvgLon']].values
    dist_mat = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i != j:
                d = np.sqrt((coords[i,0]-coords[j,0])**2 + (coords[i,1]-coords[j,1])**2)
                dist_mat[i,j] = 1 / (d + 1e-8)
    row_sums = dist_mat.sum(axis=1, keepdims=True)
    W = dist_mat / (row_sums + 1e-8)

    def compute_gearys_c(y, W):
        n = len(y)
        y_bar = y.mean()
        numerator = 0.0
        for i in range(n):
            for j in range(n):
                numerator += W[i, j] * (y[i] - y[j]) ** 2
        numerator *= (n - 1)
        S0 = W.sum()
        denominator = 2 * S0 * ((y - y_bar) ** 2).sum()
        return numerator / denominator if denominator != 0 else 0

    def gearys_pvalue(y, W, permutations=999):
        observed_C = compute_gearys_c(y, W)
        perm_Cs = []
        for _ in range(permutations):
            y_perm = np.random.permutation(y)
            perm_Cs.append(compute_gearys_c(y_perm, W))
        # C < 1 => positive autocorrelation; p-value: fraction of perm C <= observed
        pseudo_p = (np.sum(np.array(perm_Cs) <= observed_C) + 1) / (permutations + 1)
        return observed_C, pseudo_p, np.array(perm_Cs)

    results_geary = []
    for variable in ['EventCount', 'AvgDuration']:
        if variable in state_stats.columns:
            y = state_stats[variable].values.astype(float)
            C_val, p_val, perm_dist = gearys_pvalue(y, W)
            interp = ('Positive Autocorrelation' if (C_val < 1 and p_val < 0.05)
                      else 'Negative Autocorrelation' if (C_val > 1 and p_val < 0.05)
                      else 'Random')
            results_geary.append({
                'Variable': variable,
                "Geary's C": round(C_val, 4),
                'p-value (MC)': round(p_val, 4),
                'Interpretation': interp
            })
            print(f"     {variable}: C={C_val:.4f}, p={p_val:.4f} → {interp}")

    geary_df = pd.DataFrame(results_geary)
    geary_df.to_csv(f"{OUTPUT_DIR}/tables/gearys_c_results.csv", index=False)

    # Figure
    y_main = state_stats['EventCount'].values.astype(float)
    C_val_main, p_val_main, perm_dist_main = gearys_pvalue(y_main, W)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle("Geary's C Spatial Autocorrelation Analysis", fontsize=14, fontweight='bold')

    # Local difference scatter: x=value, y=average difference with neighbours
    y_std = (y_main - y_main.mean()) / (y_main.std() + 1e-8)
    local_diff = np.array([
        np.sum(W[i] * (y_std[i] - y_std)**2) for i in range(n)
    ])
    axes[0].scatter(y_std, local_diff, color=COLORS[2], alpha=0.7,
                    edgecolors='white', s=60)
    axes[0].set_title(f"Geary Scatter (EventCount)\nC={C_val_main:.4f}, p={p_val_main:.4f}")
    axes[0].set_xlabel('Standardized Event Count (z)')
    axes[0].set_ylabel('Local Weighted Squared Difference')
    for i, row in state_stats.iterrows():
        if i < n:
            axes[0].annotate(row['State'], (y_std[i], local_diff[i]), fontsize=6, alpha=0.7)

    # Permutation distribution
    axes[1].hist(perm_dist_main, bins=40, color=COLORS[3], alpha=0.8, edgecolor='white')
    axes[1].axvline(C_val_main, color='red', linestyle='--', linewidth=2,
                     label=f"Observed C={C_val_main:.4f}")
    axes[1].axvline(1.0, color='gray', linestyle=':', linewidth=1.5, label='C=1 (random)')
    axes[1].set_title("Monte Carlo Permutation Distribution")
    axes[1].set_xlabel("Geary's C (permuted)"); axes[1].set_ylabel("Frequency")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/spatial/fig_s3b_gearys_c.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("     [Fig S3b] Geary's C analysis saved.")
    print(geary_df.to_string(index=False))
    return geary_df


# ============================================================
# LISA — LOCAL INDICATORS OF SPATIAL ASSOCIATION
# ============================================================
def lisa_analysis(df):
    """LISA (Local Moran's I) — identifies local spatial clusters and outliers."""
    print("\n[S3c] LISA — Local Indicators of Spatial Association...")

    if 'State' not in df.columns:
        print("     State column not found.")
        return None

    state_stats = df.groupby('State').agg(
        EventCount=('Start_Lat', 'count'),
        AvgLat=('Start_Lat', 'mean'),
        AvgLon=('Start_Lng', 'mean')
    ).reset_index()

    n = len(state_stats)
    if n < 5:
        print("     Not enough states for LISA.")
        return None

    coords = state_stats[['AvgLat', 'AvgLon']].values
    dist_mat = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i != j:
                d = np.sqrt((coords[i,0]-coords[j,0])**2 + (coords[i,1]-coords[j,1])**2)
                dist_mat[i,j] = 1 / (d + 1e-8)
    row_sums = dist_mat.sum(axis=1, keepdims=True)
    W = dist_mat / (row_sums + 1e-8)

    y = state_stats['EventCount'].values.astype(float)
    y_std = (y - y.mean()) / (y.std() + 1e-8)
    Wy = W @ y_std

    # Local Moran's I_i
    local_I = y_std * Wy

    # Pseudo p-values via permutation
    n_perm = 499
    perm_local_I = np.zeros((n_perm, n))
    for p in range(n_perm):
        y_perm = np.random.permutation(y_std)
        Wy_perm = W @ y_perm
        perm_local_I[p] = y_perm * Wy_perm

    pseudo_p = np.array([
        (np.sum(perm_local_I[:, i] >= local_I[i]) + 1) / (n_perm + 1)
        for i in range(n)
    ])

    # Classify into quadrants (HH, LL, HL, LH) for significant locations
    sig = pseudo_p < 0.05
    cluster_type = []
    for i in range(n):
        if not sig[i]:
            cluster_type.append('Not Significant')
        elif y_std[i] > 0 and Wy[i] > 0:
            cluster_type.append('HH (Hot Spot)')
        elif y_std[i] < 0 and Wy[i] < 0:
            cluster_type.append('LL (Cold Spot)')
        elif y_std[i] > 0 and Wy[i] < 0:
            cluster_type.append('HL (Spatial Outlier)')
        else:
            cluster_type.append('LH (Spatial Outlier)')

    state_stats['Local_I'] = local_I
    state_stats['LISA_p']  = pseudo_p
    state_stats['LISA_type'] = cluster_type
    state_stats.to_csv(f"{OUTPUT_DIR}/tables/lisa_results.csv", index=False)

    sig_counts = state_stats['LISA_type'].value_counts()
    print("     LISA cluster summary:")
    for k, v in sig_counts.items():
        print(f"       {k}: {v} states")

    # Figure
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle("LISA — Local Indicators of Spatial Association", fontsize=14, fontweight='bold')

    # LISA cluster map (scatter by lat/lon coloured by type)
    type_colors = {
        'HH (Hot Spot)':      '#d62728',
        'LL (Cold Spot)':     '#1f77b4',
        'HL (Spatial Outlier)': '#ff7f0e',
        'LH (Spatial Outlier)': '#9467bd',
        'Not Significant':    '#cccccc'
    }
    for typ, color in type_colors.items():
        mask = state_stats['LISA_type'] == typ
        if mask.sum() > 0:
            axes[0].scatter(
                state_stats.loc[mask, 'AvgLon'],
                state_stats.loc[mask, 'AvgLat'],
                c=color, label=typ, s=80, edgecolors='white', linewidths=0.5
            )
            for _, row in state_stats[mask].iterrows():
                axes[0].annotate(row['State'], (row['AvgLon'], row['AvgLat']),
                                  fontsize=6, alpha=0.8)
    axes[0].set_title('LISA Cluster Map (State Level)')
    axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')
    axes[0].legend(fontsize=7, loc='lower left')

    # Moran scatter coloured by LISA type
    for typ, color in type_colors.items():
        mask = state_stats['LISA_type'] == typ
        if mask.sum() > 0:
            axes[1].scatter(y_std[mask.values], Wy[mask.values],
                            c=color, label=typ, s=60, edgecolors='white', linewidths=0.5)
    axes[1].axhline(0, color='gray', linestyle='--', alpha=0.5)
    axes[1].axvline(0, color='gray', linestyle='--', alpha=0.5)
    slope, intercept = np.polyfit(y_std, Wy, 1)
    x_line = np.linspace(y_std.min(), y_std.max(), 100)
    axes[1].plot(x_line, slope * x_line + intercept, 'k-', linewidth=2, alpha=0.7)
    axes[1].set_title('LISA Moran Scatter (coloured by cluster type)')
    axes[1].set_xlabel('Standardized Event Count (z)')
    axes[1].set_ylabel('Spatial Lag (Wz)')
    axes[1].legend(fontsize=7)

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/spatial/fig_s3c_lisa.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("     [Fig S3c] LISA analysis saved.")
    return state_stats


# ============================================================
# SPATIAL CLUSTERING (DBSCAN + K-MEANS)
# ============================================================
def spatial_clustering(df):
    print("\n[S4] Spatial Clustering (DBSCAN + K-Means)...")

    lat_min, lat_max = 24.5, 49.5
    lon_min, lon_max = -125.0, -66.0
    mask = ((df['Start_Lat'] >= lat_min) & (df['Start_Lat'] <= lat_max) &
            (df['Start_Lng'] >= lon_min) & (df['Start_Lng'] <= lon_max))

    sample = df[mask].sample(min(30000, len(df[mask])), random_state=42)
    coords = sample[['Start_Lat','Start_Lng']].values

    # Normalize
    scaler = StandardScaler()
    coords_scaled = scaler.fit_transform(coords)

    # --- K-Means ---
    print("     Running K-Means (k=2..10)...")
    inertias = []; silhouettes = []
    K_range = range(2, 11)
    for k in K_range:
        km = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=5)
        labels = km.fit_predict(coords_scaled)
        inertias.append(km.inertia_)
        sil = silhouette_score(coords_scaled, labels, sample_size=5000)
        silhouettes.append(sil)
        print(f"       k={k}: inertia={km.inertia_:.0f}, silhouette={sil:.4f}")

    # Best K by silhouette
    best_k = K_range.start + np.argmax(silhouettes)
    print(f"     Best k={best_k} by silhouette={max(silhouettes):.4f}")

    km_best = KMeans(n_clusters=best_k, random_state=RANDOM_SEED, n_init=10)
    km_labels = km_best.fit_predict(coords_scaled)
    sample = sample.copy()
    sample['KMeans_Cluster'] = km_labels

    # --- DBSCAN ---
    print("     Running DBSCAN (eps=0.3, min_samples=50)...")
    db = DBSCAN(eps=0.3, min_samples=50, n_jobs=-1)
    db_labels = db.fit_predict(coords_scaled)
    sample['DBSCAN_Cluster'] = db_labels
    n_clusters_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
    n_noise = (db_labels == -1).sum()
    print(f"     DBSCAN: {n_clusters_db} clusters, {n_noise} noise points ({n_noise/len(db_labels)*100:.1f}%)")

    # Cluster quality table
    cluster_quality = pd.DataFrame({
        'Method': ['K-Means','DBSCAN'],
        'N Clusters': [best_k, n_clusters_db],
        'Silhouette Score': [round(max(silhouettes),4),
                              round(silhouette_score(coords_scaled[db_labels != -1],
                                                      db_labels[db_labels != -1],
                                                      sample_size=5000)
                                    if n_clusters_db > 1 else 0.0, 4)],
        'Noise Points': [0, n_noise]
    })
    cluster_quality.to_csv(f"{OUTPUT_DIR}/tables/clustering_quality.csv", index=False)
    print("\n     Clustering Quality:")
    print(cluster_quality.to_string(index=False))

    # PLOT
    fig = plt.figure(figsize=(20, 12))
    fig.suptitle('Spatial Clustering of Road Construction Events',
                 fontsize=15, fontweight='bold')
    gs = gridspec.GridSpec(2, 3, figure=fig)

    # K-Means elbow
    ax1 = fig.add_subplot(gs[0, 0])
    ax1b = ax1.twinx()
    ax1.plot(list(K_range), inertias, 'bo-', linewidth=2, label='Inertia')
    ax1b.plot(list(K_range), silhouettes, 'rs--', linewidth=2, label='Silhouette')
    ax1.set_xlabel('Number of Clusters (k)'); ax1.set_ylabel('Inertia', color='b')
    ax1b.set_ylabel('Silhouette Score', color='r')
    ax1.set_title('K-Means: Elbow & Silhouette')
    ax1.axvline(best_k, color='green', linestyle=':', label=f'Best k={best_k}')
    ax1.legend(loc='upper left'); ax1b.legend(loc='upper right')

    # K-Means spatial
    ax2 = fig.add_subplot(gs[0, 1:])
    cmap_km = plt.cm.get_cmap('tab10', best_k)
    for cluster_id in range(best_k):
        mask_c = sample['KMeans_Cluster'] == cluster_id
        ax2.scatter(sample[mask_c]['Start_Lng'], sample[mask_c]['Start_Lat'],
                    s=0.8, alpha=0.5, color=cmap_km(cluster_id), label=f'C{cluster_id}')
    centers = scaler.inverse_transform(km_best.cluster_centers_)
    ax2.scatter(centers[:, 1], centers[:, 0], s=200, marker='*',
                color='black', zorder=5, label='Centroids')
    ax2.set_title(f'K-Means Spatial Clusters (k={best_k})')
    ax2.set_xlabel('Longitude'); ax2.set_ylabel('Latitude')
    ax2.legend(fontsize=7, ncol=3, loc='lower right')

    # DBSCAN spatial
    ax3 = fig.add_subplot(gs[1, :2])
    noise_mask = sample['DBSCAN_Cluster'] == -1
    ax3.scatter(sample[noise_mask]['Start_Lng'], sample[noise_mask]['Start_Lat'],
                s=0.5, alpha=0.2, color='lightgray', label='Noise')
    unique_db = sorted([l for l in sample['DBSCAN_Cluster'].unique() if l >= 0])
    cmap_db = plt.cm.get_cmap('tab20', max(len(unique_db),1))
    for cl in unique_db[:20]:
        mc = sample['DBSCAN_Cluster'] == cl
        ax3.scatter(sample[mc]['Start_Lng'], sample[mc]['Start_Lat'],
                    s=1, alpha=0.6, color=cmap_db(cl))
    ax3.set_title(f'DBSCAN Clusters ({n_clusters_db} clusters, {n_noise} noise)')
    ax3.set_xlabel('Longitude'); ax3.set_ylabel('Latitude')

    # Cluster sizes
    ax4 = fig.add_subplot(gs[1, 2])
    km_sizes = sample['KMeans_Cluster'].value_counts().sort_index()
    ax4.barh([f'Cluster {i}' for i in km_sizes.index],
              km_sizes.values, color=cmap_km(np.arange(len(km_sizes))/best_k))
    ax4.set_title('K-Means Cluster Sizes')
    ax4.set_xlabel('Event Count')

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/spatial/fig_s4_clustering.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("     [Fig S4] Spatial clustering saved.")

    return sample, cluster_quality

# ============================================================
# SPATIAL CLASSIFICATION (ML WITH SPATIAL FEATURES)
# ============================================================
def spatial_classification(df):
    print("\n[S5] Spatial Classification: Predicting Closure Type from Spatial Features...")

    spatial_features = [c for c in ['Start_Lat','Start_Lng','DistFromCenter',
                                      'GridCellCount','HexCellCount','POI_Count',
                                      'IsUrban','StateMonthCount'] if c in df.columns]

    target = 'Type' if 'Type' in df.columns else 'Severity'
    if target not in df.columns or len(spatial_features) < 3:
        print("     Insufficient features.")
        return None

    data = df[spatial_features + [target]].dropna()
    if len(data) > 100000:
        data = data.sample(100000, random_state=RANDOM_SEED)
    le = LabelEncoder()
    y = le.fit_transform(data[target].astype(str))
    X = data[spatial_features]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y)

    models = {
        'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=12,
                                                 random_state=RANDOM_SEED, n_jobs=-1,
                                                 class_weight='balanced'),
        'Gradient Boosting': HistGradientBoostingClassifier(max_iter=100, max_depth=5,
                                                             random_state=RANDOM_SEED),
        'Logistic Regression': LogisticRegression(max_iter=500, random_state=RANDOM_SEED,
                                                   class_weight='balanced')
    }

    results = []
    best_model_obj = None
    best_f1 = -1

    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        f1_mac = f1_score(y_test, y_pred, average='macro', zero_division=0)
        f1_wt  = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        idx = np.random.choice(len(X_scaled), min(50000, len(X_scaled)), replace=False)
        cv = cross_val_score(model, X_scaled[idx], y[idx], cv=3, scoring='f1_macro', n_jobs=-1)
        results.append({'Model': name, 'Accuracy': round(acc,4),
                         'F1 Macro': round(f1_mac,4), 'F1 Weighted': round(f1_wt,4),
                         'CV F1': round(cv.mean(),4), 'CV Std': round(cv.std(),4)})
        print(f"     {name}: Acc={acc:.4f}, F1={f1_mac:.4f}")
        if f1_mac > best_f1:
            best_f1 = f1_mac
            best_model_obj = (name, model, y_pred)

    results_df = pd.DataFrame(results).sort_values('F1 Macro', ascending=False)
    results_df.to_csv(f"{OUTPUT_DIR}/tables/spatial_classification_results.csv", index=False)

    best_name, best_model, best_pred = best_model_obj

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Spatial Classification: Closure Type Prediction', fontsize=14, fontweight='bold')

    # Metrics
    x = np.arange(len(results_df))
    w = 0.2
    for i, m in enumerate(['Accuracy','F1 Macro','F1 Weighted','CV F1']):
        axes[0].bar(x + i*w, results_df[m], w, label=m, color=COLORS[i])
    axes[0].set_xticks(x + w*1.5)
    axes[0].set_xticklabels(results_df['Model'], rotation=15, ha='right', fontsize=8)
    axes[0].set_title('Model Performance Comparison')
    axes[0].set_ylabel('Score'); axes[0].legend(fontsize=7)
    axes[0].set_ylim(0, 1.1)

    # Confusion matrix
    cm = confusion_matrix(y_test, best_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
                xticklabels=le.classes_, yticklabels=le.classes_)
    axes[1].set_title(f'Confusion Matrix: {best_name}')
    axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')

    # Feature importance
    if hasattr(best_model, 'feature_importances_'):
        imp = pd.Series(best_model.feature_importances_, index=spatial_features)
        imp.sort_values(ascending=True).plot(kind='barh', ax=axes[2], color=COLORS[0])
        axes[2].set_title(f'Spatial Feature Importances\n({best_name})')
        axes[2].set_xlabel('Importance')

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/spatial/fig_s5_spatial_classification.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("     [Fig S5] Spatial classification saved.")

    print("\n     Spatial Classification Results:")
    print(results_df.to_string(index=False))
    return results_df

# ============================================================
# GEOGRAPHICALLY WEIGHTED REGRESSION (PROXY)
# ============================================================
def geographically_weighted_regression(df, state_stats):
    print("\n[S6] Geographically Weighted Regression (GWR) - State Level Proxy...")

    from sklearn.metrics import mean_squared_error, r2_score
    from sklearn.linear_model import LinearRegression

    if state_stats is None or len(state_stats) < 5:
        print("     State stats not available.")
        return

    # Predict AvgDuration from AvgLat and AvgLon (state-level)
    y = state_stats['AvgDuration'].values if 'AvgDuration' in state_stats.columns else None
    if y is None:
        return

    X = state_stats[['AvgLat','AvgLon']].values

    # Global OLS
    
    ols = LinearRegression().fit(X, y)
    y_pred_ols = ols.predict(X)
    r2_ols = r2_score(y, y_pred_ols)

    # GWR proxy: local weighted OLS for each state
    y_pred_gwr = np.zeros(len(y))
    bandwidth = 10.0  # degrees

    for i in range(len(X)):
        dists = np.sqrt(((X - X[i])**2).sum(axis=1))
        weights = np.exp(-(dists**2) / (2 * bandwidth**2))  # Gaussian kernel
        weights[i] = 0  # leave-one-out
        if weights.sum() > 0:
            W_diag = np.diag(weights)
            XtW = X.T @ W_diag
            beta = np.linalg.lstsq(XtW @ X, XtW @ y, rcond=None)[0]
            y_pred_gwr[i] = X[i] @ beta

    r2_gwr = r2_score(y, y_pred_gwr)
    rmse_ols = np.sqrt(mean_squared_error(y, y_pred_ols))
    rmse_gwr = np.sqrt(mean_squared_error(y, y_pred_gwr))

    
    gwr_table = pd.DataFrame({
        'Model': ['Global OLS', 'GWR (Gaussian kernel)'],
        'R²': [round(r2_ols,4), round(r2_gwr,4)],
        'RMSE': [round(rmse_ols,4), round(rmse_gwr,4)]
    })
    gwr_table.to_csv(f"{OUTPUT_DIR}/tables/gwr_results.csv", index=False)
    print(gwr_table.to_string(index=False))

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Geographically Weighted Regression: Avg Duration Prediction',
                 fontsize=13, fontweight='bold')

    axes[0].scatter(y, y_pred_ols, color=COLORS[0], alpha=0.7, edgecolors='white', label='OLS')
    axes[0].scatter(y, y_pred_gwr, color=COLORS[1], alpha=0.7, edgecolors='white', label='GWR')
    mn, mx = min(y.min(), y_pred_gwr.min()), max(y.max(), y_pred_gwr.max())
    axes[0].plot([mn,mx],[mn,mx], 'k--', label='Perfect fit')
    axes[0].set_title(f'Actual vs Predicted\nOLS R²={r2_ols:.4f} | GWR R²={r2_gwr:.4f}')
    axes[0].set_xlabel('Actual Avg Duration (hrs)'); axes[0].set_ylabel('Predicted')
    axes[0].legend()

    # State residuals on map
    residuals_gwr = y - y_pred_gwr
    sc = axes[1].scatter(state_stats['AvgLon'], state_stats['AvgLat'],
                          c=residuals_gwr, cmap='RdBu_r', s=100,
                          vmin=-np.abs(residuals_gwr).max(),
                          vmax=np.abs(residuals_gwr).max(), edgecolors='black', linewidths=0.5)
    plt.colorbar(sc, ax=axes[1], label='GWR Residual (hrs)')
    for _, row in state_stats.iterrows():
        axes[1].annotate(row['State'], (row['AvgLon'], row['AvgLat']), fontsize=6, ha='center')
    axes[1].set_title('GWR Residuals by State')
    axes[1].set_xlabel('Longitude'); axes[1].set_ylabel('Latitude')

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/spatial/fig_s6_gwr.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("     [Fig S6] GWR saved.")

# ============================================================
# STATE-LEVEL CHOROPLETH (using matplotlib rectangles as proxy)
# ============================================================
def state_choropleth(df):
    print("\n[S7] State-Level Spatial Analysis...")
    if 'State' not in df.columns:
        return

    state_agg = df.groupby('State').agg(
        EventCount=('Start_Lat','count'),
        AvgDuration=('Duration_hrs','mean'),
        MedianDuration=('Duration_hrs','median')
    ).reset_index().sort_values('EventCount', ascending=False)
    state_agg.to_csv(f"{OUTPUT_DIR}/tables/state_level_stats.csv", index=False)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('State-Level Construction Event Analysis', fontsize=14, fontweight='bold')

    top20 = state_agg.head(20)
    cmap = plt.cm.YlOrRd
    norm = mcolors.Normalize(vmin=top20['EventCount'].min(), vmax=top20['EventCount'].max())

    bars = axes[0].barh(top20['State'], top20['EventCount'],
                         color=cmap(norm(top20['EventCount'].values)))
    axes[0].set_title('Top 20 States: Event Count')
    axes[0].set_xlabel('Total Events')
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    plt.colorbar(sm, ax=axes[0])

    # Avg duration
    state_agg_dur = state_agg.sort_values('AvgDuration', ascending=False).head(20)
    axes[1].barh(state_agg_dur['State'], state_agg_dur['AvgDuration'], color=COLORS[1])
    axes[1].set_title('Top 20 States: Avg Event Duration (hrs)')
    axes[1].set_xlabel('Average Duration (hours)')

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/spatial/fig_s7_state_choropleth.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("     [Fig S7] State choropleth saved.")
    print("\n     Top 10 States by Event Count:")
    print(state_agg.head(10).to_string(index=False))

# ============================================================
# VARIOGRAM ANALYSIS (Spatial Statistics)
# ============================================================
def variogram_analysis(df):
    print("\n[S8] Variogram Analysis (Empirical)...")

    lat_min, lat_max = 24.5, 49.5
    lon_min, lon_max = -125.0, -66.0
    mask = ((df['Start_Lat'] >= lat_min) & (df['Start_Lat'] <= lat_max) &
            (df['Start_Lng'] >= lon_min) & (df['Start_Lng'] <= lon_max))

    # Sample small subset for variogram
    sample = df[mask][['Start_Lat','Start_Lng','Duration_hrs']].dropna()
    sample = sample.sample(min(2000, len(sample)), random_state=42)

    coords = sample[['Start_Lat','Start_Lng']].values
    values = sample['Duration_hrs'].values

    # Compute pairwise distances and semivariances
    print("     Computing pairwise distances...")
    n = len(coords)
    distances = []
    semivariances = []
    for i in range(0, n, 5):  # subsample pairs
        for j in range(i+1, min(i+100, n)):
            d = np.sqrt((coords[i,0]-coords[j,0])**2 + (coords[i,1]-coords[j,1])**2)
            sv = 0.5 * (values[i] - values[j])**2
            distances.append(d)
            semivariances.append(sv)

    distances = np.array(distances)
    semivariances = np.array(semivariances)

    # Bin by distance
    bins = np.linspace(0, distances.max(), 20)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    bin_sv = []
    for i in range(len(bins)-1):
        mask_bin = (distances >= bins[i]) & (distances < bins[i+1])
        if mask_bin.sum() > 5:
            bin_sv.append(semivariances[mask_bin].mean())
        else:
            bin_sv.append(np.nan)
    bin_sv = np.array(bin_sv)

    # Fit spherical variogram model (approximate)
    valid = ~np.isnan(bin_sv)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.scatter(bin_centers[valid], bin_sv[valid], color=COLORS[0], s=60,
               zorder=5, label='Empirical Variogram')
    if valid.sum() > 3:
        z = np.polyfit(bin_centers[valid], bin_sv[valid], 2)
        p = np.poly1d(z)
        x_fit = np.linspace(bin_centers[valid].min(), bin_centers[valid].max(), 100)
        ax.plot(x_fit, p(x_fit), 'r--', linewidth=2, label='Polynomial Fit')
    ax.set_title('Empirical Variogram: Construction Event Duration',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Distance (degrees)'); ax.set_ylabel('Semivariance')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/spatial/fig_s8_variogram.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("     [Fig S8] Variogram saved.")

# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":
    from sklearn.metrics import mean_squared_error, r2_score

    df = load_processed()

    df_map = spatial_density_map(df)
    kernel_density_estimation(df_map)
    moran_result = morans_i(df)
    morans_df, state_stats = moran_result if moran_result else (None, None)
    gearys_c(df)
    lisa_analysis(df)
    sample_clustered, cluster_quality = spatial_clustering(df)
    spatial_classification(df)
    geographically_weighted_regression(df, state_stats)
    state_choropleth(df)
    variogram_analysis(df)

    print("\n[PART 3 COMPLETE] Proceed to part4_mining.py")


PART 3: SPATIAL ANALYSIS & MACHINE LEARNING
[Load] Shape: (1644242, 85)

[S1] Spatial Density Map (Hexbin)...
     [Fig S1] Spatial density map saved.

[S2] Kernel Density Estimation (2D KDE)...
     [Fig S2] KDE saved.

[S3] Spatial Autocorrelation - Moran's I...
     EventCount: I=-0.0185, p=0.4180
     AvgDuration: I=0.0046, p=0.1900
     [Fig S3] Moran's I analysis saved.
   Variable  Moran's I  p-value (MC) Interpretation
 EventCount    -0.0185         0.418         Random
AvgDuration     0.0046         0.190         Random

[S3b] Geary's C Spatial Autocorrelation...
     EventCount: C=0.9378, p=0.1020 → Random
     AvgDuration: C=0.9665, p=0.1740 → Random
     [Fig S3b] Geary's C analysis saved.
   Variable  Geary's C  p-value (MC) Interpretation
 EventCount     0.9378         0.102         Random
AvgDuration     0.9665         0.174         Random

[S3c] LISA — Local Indicators of Spatial Association...
     LISA cluster summary:
       Not Significant: 49 states
       LL (Cold

---
## Part 4 — Mining & Joint Inference (Expanded)
Spatial colocation mining, temporal pattern mining, association rules (Apriori), anomaly detection (Isolation Forest + LOF), hotspot analysis (Getis-Ord G*), 3D spatio-temporal clustering, XGBoost spatio-temporal model, LSTM spatio-temporal model, and joint predictive models.

In [7]:
"""
AID843 A3 - Part 4: EXPANDED Data Mining & Joint Inference
US Road Construction and Closures (2016-2021)
Dataset filename: US_Closures.csv

MINING TECHNIQUES COVERED:
  M1.  Spatial Colocation Mining (Participation Index + Lift Matrix)
  M2.  Temporal Pattern Mining (L/M/H motifs, recurring subsequences)
  M3.  Association Rule Mining (Apriori) on POI + weather + time attributes
  M4.  Spatial Anomaly Detection (Isolation Forest + Local Outlier Factor)
  M5.  Temporal Anomaly Detection (Z-score + IQR on monthly counts)
  M6.  Frequent Spatial Sequence Mining (state-to-state transition patterns)
  M7.  Spatial Hotspot Analysis (Getis-Ord G* proxy)
  M8.  Temporal Motif Discovery (matrix profile-inspired subsequence detection)
  M9.  Cluster-Based Pattern Mining (within-cluster rule discovery)
  M10. Spatial Co-location Network (graph of co-located POI types)

JOINT INFERENCE TECHNIQUES:
  J1.  Spatio-Temporal Hotspot Heatmap (grid × month)
  J2.  Joint Random Forest (spatial + temporal features, full comparison)
  J3.  Spatial-Temporal Interaction: Does location amplify temporal effects?
  J4.  Spatiotemporal Cluster Profiling (who, when, where, how long)
  J5.  Causal-style Analysis: Weather + Location → Duration
  J6.  Cross-correlation: spatial density vs temporal frequency
  J7.  Joint DBSCAN on (lat, lon, time) 3D space
  J8.  Predictive hotspot model (predicting next-month events by cell)
  J9.  Comprehensive summary: best models, key findings, report tables
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import os
from itertools import combinations, permutations
from collections import Counter, defaultdict
from scipy import stats
from scipy.ndimage import gaussian_filter
from scipy.stats import pearsonr, spearmanr, chi2_contingency


from sklearn.ensemble import (IsolationForest, RandomForestClassifier,
                               GradientBoostingClassifier, RandomForestRegressor)
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.cluster import DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (f1_score, accuracy_score, confusion_matrix,
                              classification_report, mean_squared_error, r2_score,
                              silhouette_score)
from sklearn.linear_model import LogisticRegression, Ridge, Lasso
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
OUTPUT_DIR = "outputs"
os.makedirs(f"{OUTPUT_DIR}/mining", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/joint", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/tables", exist_ok=True)

COLORS = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd',
          '#8c564b','#e377c2','#7f7f7f','#bcbd22','#17becf']
plt.style.use('seaborn-v0_8-whitegrid')

print("="*70)
print("PART 4 (EXPANDED): DATA MINING & JOINT INFERENCE")
print("="*70)

# ============================================================
# LOAD
# ============================================================
def load_processed():
    if not os.path.exists(f"{OUTPUT_DIR}/processed_data.parquet"):
        raise FileNotFoundError(
            "\n\n❌  processed_data.parquet not found.\n"
            "    Run Part 1 cell first before running Part 4."
        )
    df = pd.read_parquet(f"{OUTPUT_DIR}/processed_data.parquet")
    print(f"[Load] Processed data shape: {df.shape}")
    return df

# ============================================================
# M1. SPATIAL COLOCATION MINING (Participation Index Method)
# ============================================================
def m1_spatial_colocation(df, distance_threshold_deg=0.05):
    """
    Full Participation-Index based Spatial Colocation Mining.
    PI(A,B) = min(PR(A,B), PR(B,A))
    where PR(A,B) = |instances of A that have B within distance d| / |A|
    Uses KDTree for efficient neighbor search.
    """
    print("\n" + "="*60)
    print("[M1] SPATIAL COLOCATION MINING (Participation Index)")
    print("="*60)

    from scipy.spatial import cKDTree

    poi_cols = [c for c in ['Junction','Traffic_Signal','Crossing','Roundabout',
                             'Station','Railway','Stop','Amenity','Bump',
                             'Give_Way','No_Exit','Traffic_Calming','Turning_Loop'] if c in df.columns]

    lat_min, lat_max = 24.5, 49.5
    lon_min, lon_max = -125.0, -66.0
    mask = ((df['Start_Lat'] >= lat_min) & (df['Start_Lat'] <= lat_max) &
            (df['Start_Lng'] >= lon_min) & (df['Start_Lng'] <= lon_max))
    sample = df[mask].sample(min(15000, len(df[mask])), random_state=RANDOM_SEED).copy()

    for col in poi_cols:
        sample[col] = sample[col].astype(bool)

    coords = sample[['Start_Lat','Start_Lng']].values
    tree = cKDTree(coords)

    print(f"  POI types analyzed: {poi_cols}")
    print(f"  Distance threshold: {distance_threshold_deg}° (~{distance_threshold_deg*111:.1f} km)")

    # Build neighbor indices within threshold
    neighbor_sets = tree.query_ball_tree(tree, r=distance_threshold_deg)

    # Compute PR(A,B): among A-events, fraction that have a B-event within distance
    pr_matrix = pd.DataFrame(index=poi_cols, columns=poi_cols, dtype=float)
    for A in poi_cols:
        a_indices = np.where(sample[A].values)[0]
        if len(a_indices) == 0:
            pr_matrix.loc[A] = 0.0
            continue
        for B in poi_cols:
            b_set = set(np.where(sample[B].values)[0])
            count_with_b = sum(1 for i in a_indices
                               if any(nb in b_set for nb in neighbor_sets[i]))
            pr_matrix.loc[A, B] = count_with_b / len(a_indices)

    pr_matrix = pr_matrix.astype(float)

    # PI(A,B) = min(PR(A,B), PR(B,A))
    pi_matrix = pd.DataFrame(index=poi_cols, columns=poi_cols, dtype=float)
    for A in poi_cols:
        for B in poi_cols:
            if A == B:
                pi_matrix.loc[A, B] = 1.0
            else:
                pi_matrix.loc[A, B] = min(pr_matrix.loc[A, B], pr_matrix.loc[B, A])

    # Extract top colocation rules
    rules = []
    for A, B in combinations(poi_cols, 2):
        pi = pi_matrix.loc[A, B]
        pr_ab = pr_matrix.loc[A, B]
        pr_ba = pr_matrix.loc[B, A]
        # Global prevalence
        prev_a = sample[A].mean()
        prev_b = sample[B].mean()
        rules.append({
            'Feature_A': A, 'Feature_B': B,
            'PR(A→B)': round(pr_ab, 4),
            'PR(B→A)': round(pr_ba, 4),
            'PI(A,B)': round(pi, 4),
            'Prev(A)': round(prev_a, 4),
            'Prev(B)': round(prev_b, 4),
            'Normalized_PI': round(pi / max(min(prev_a, prev_b), 1e-8), 4)
        })

    coloc_df = pd.DataFrame(rules).sort_values('PI(A,B)', ascending=False)
    coloc_df.to_csv(f"{OUTPUT_DIR}/tables/M1_colocation_PI.csv", index=False)

    # Co-occurrence lift matrix (for comparison)
    lift_matrix = pd.DataFrame(index=poi_cols, columns=poi_cols, dtype=float)
    for A, B in combinations(poi_cols, 2):
        p_a = sample[A].mean()
        p_b = sample[B].mean()
        p_ab = (sample[A] & sample[B]).mean()
        lift = p_ab / (p_a * p_b + 1e-8)
        lift_matrix.loc[A, B] = lift
        lift_matrix.loc[B, A] = lift
    for c in poi_cols:
        lift_matrix.loc[c, c] = 1.0
    lift_matrix = lift_matrix.astype(float)

    # --- FIGURE M1: 4-panel colocation analysis ---
    fig = plt.figure(figsize=(20, 14))
    fig.suptitle('M1: Spatial Colocation Mining — Participation Index Analysis',
                 fontsize=15, fontweight='bold')
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

    # Panel 1: PI heatmap
    ax1 = fig.add_subplot(gs[0, :2])
    sns.heatmap(pi_matrix, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax1,
                square=True, linewidths=0.5, cbar_kws={'label': 'PI Score'})
    ax1.set_title('Participation Index (PI) Matrix\n(Higher = stronger colocation)',
                  fontsize=11, fontweight='bold')
    ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45, ha='right', fontsize=9)
    ax1.set_yticklabels(ax1.get_yticklabels(), fontsize=9)

    # Panel 2: Lift heatmap
    ax2 = fig.add_subplot(gs[0, 2])
    sns.heatmap(lift_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=1.0,
                ax=ax2, square=True, linewidths=0.5, cbar_kws={'label': 'Lift'})
    ax2.set_title('Lift Matrix\n(>1 = positive association)', fontsize=11, fontweight='bold')
    ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45, ha='right', fontsize=8)
    ax2.set_yticklabels(ax2.get_yticklabels(), fontsize=8)

    # Panel 3: Top PI rules bar
    ax3 = fig.add_subplot(gs[1, :2])
    top15 = coloc_df.head(15).copy()
    top15['Rule'] = top15['Feature_A'] + ' ↔ ' + top15['Feature_B']
    bars = ax3.barh(top15['Rule'], top15['PI(A,B)'], color=COLORS[0], alpha=0.85)
    ax3_twin = ax3.twiny()
    ax3_twin.barh(top15['Rule'], top15['Normalized_PI'],
                   color=COLORS[1], alpha=0.5, label='Normalized PI')
    ax3.set_xlabel('Participation Index (PI)')
    ax3_twin.set_xlabel('Normalized PI', color=COLORS[1])
    ax3.set_title('Top 15 Colocation Rules by Participation Index', fontsize=11, fontweight='bold')
    ax3.axvline(0.1, color='red', linestyle='--', alpha=0.7, label='PI=0.10 threshold')
    ax3.legend(loc='lower right', fontsize=8)

    # Panel 4: Prevalence vs PI scatter
    ax4 = fig.add_subplot(gs[1, 2])
    sc = ax4.scatter(coloc_df['Prev(A)'], coloc_df['Prev(B)'],
                      c=coloc_df['PI(A,B)'], cmap='viridis', s=80, alpha=0.8)
    plt.colorbar(sc, ax=ax4, label='PI(A,B)')
    for _, row in coloc_df.head(5).iterrows():
        ax4.annotate(f"{row['Feature_A'][:3]}/{row['Feature_B'][:3]}",
                     (row['Prev(A)'], row['Prev(B)']), fontsize=7)
    ax4.set_xlabel('Prevalence of A'); ax4.set_ylabel('Prevalence of B')
    ax4.set_title('Prevalence vs PI (top rules labeled)', fontsize=11, fontweight='bold')

    plt.savefig(f"{OUTPUT_DIR}/mining/fig_M1_colocation_PI.png", dpi=150, bbox_inches='tight')
    plt.close()

    print(f"\n  Top 10 Colocation Rules:")
    print(coloc_df.head(10)[['Feature_A','Feature_B','PI(A,B)','PR(A→B)','PR(B→A)','Normalized_PI']].to_string(index=False))
    print("  [Fig M1] Colocation mining (4-panel) saved.")
    return coloc_df, pi_matrix

# ============================================================
# M2. TEMPORAL PATTERN MINING (Motif + Subsequence Discovery)
# ============================================================
def m2_temporal_patterns(df):
    """
    Advanced temporal pattern mining:
    - Discretized sequence motif analysis (L/M/H)
    - Recurring subsequences of length 2,3,4
    - Periodic pattern detection (weekly, monthly)
    - Season-type transition matrix
    """
    print("\n" + "="*60)
    print("[M2] TEMPORAL PATTERN MINING (Motif + Sequence Analysis)")
    print("="*60)

    df['Date'] = df['Start_Time'].dt.date
    daily = df.groupby('Date').size().reset_index(name='EventCount')
    daily['Date'] = pd.to_datetime(daily['Date'])
    daily = daily.set_index('Date').asfreq('D', fill_value=0)

    monthly = daily['EventCount'].resample('MS').sum()
    weekly  = daily['EventCount'].resample('W').sum()

    # ---- Discretize ----
    def discretize_series(series, labels=('L','M','H')):
        q33 = series.quantile(0.33)
        q66 = series.quantile(0.66)
        return series.apply(lambda v: labels[0] if v <= q33 else
                                       labels[1] if v <= q66 else labels[2])

    monthly_disc = discretize_series(monthly)
    weekly_disc  = discretize_series(weekly)

    symbols_m = list(monthly_disc.values)
    symbols_w = list(weekly_disc.values)

    # ---- Motif counting function ----
    def count_motifs(symbols, max_len=5):
        motifs = {}
        for length in range(2, max_len+1):
            for i in range(len(symbols) - length + 1):
                pat = tuple(symbols[i:i+length])
                motifs[pat] = motifs.get(pat, 0) + 1
        return motifs

    motifs_m = count_motifs(symbols_m, max_len=4)
    motifs_w = count_motifs(symbols_w, max_len=4)

    # Top motifs
    top_m = sorted(motifs_m.items(), key=lambda x: (-x[1], len(x[0])))[:20]
    top_w = sorted(motifs_w.items(), key=lambda x: (-x[1], len(x[0])))[:20]

    top_m_df = pd.DataFrame([((''.join(p)), l, len(p), l/len(symbols_m))
                              for p,l in top_m],
                             columns=['Pattern','Frequency','Length','Support'])
    top_w_df = pd.DataFrame([((''.join(p)), l, len(p), l/len(symbols_w))
                              for p,l in top_w],
                             columns=['Pattern','Frequency','Length','Support'])
    top_m_df.to_csv(f"{OUTPUT_DIR}/tables/M2_monthly_motifs.csv", index=False)
    top_w_df.to_csv(f"{OUTPUT_DIR}/tables/M2_weekly_motifs.csv", index=False)

    # ---- Season transition matrix ----
    if 'Season' in df.columns:
        seasons = ['Winter','Spring','Summer','Fall']
        df_sorted = df.sort_values('Start_Time')
        season_seq = df_sorted['Season'].values
        trans = np.zeros((4,4), dtype=int)
        s2i = {s:i for i,s in enumerate(seasons)}
        for i in range(len(season_seq)-1):
            a, b = season_seq[i], season_seq[i+1]
            if a in s2i and b in s2i:
                trans[s2i[a], s2i[b]] += 1
        trans_prob = trans / (trans.sum(axis=1, keepdims=True) + 1e-8)
        trans_df = pd.DataFrame(trans_prob, index=seasons, columns=seasons).round(4)
        trans_df.to_csv(f"{OUTPUT_DIR}/tables/M2_season_transition.csv")
    else:
        trans_df = None

    # ---- Closure type hourly pattern ----
    if 'Type' in df.columns:
        type_hour = df.groupby(['Type','Hour']).size().reset_index(name='Count')
        type_hour_pivot = type_hour.pivot(index='Hour', columns='Type', values='Count').fillna(0)
    else:
        type_hour_pivot = None

    # ---- FIGURE M2: 5-panel ----
    fig = plt.figure(figsize=(22, 16))
    fig.suptitle('M2: Temporal Pattern Mining — Motif & Sequence Discovery',
                 fontsize=15, fontweight='bold')
    gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

    # Panel 1: Monthly sequence colored
    ax1 = fig.add_subplot(gs[0, :])
    color_map = {'L':'#2196F3','M':'#FFC107','H':'#F44336'}
    for i, (date, val) in enumerate(monthly.items()):
        sym = monthly_disc.iloc[i]
        ax1.bar(date, val, width=25, color=color_map[sym], alpha=0.85)
    ma3 = monthly.rolling(3, center=True).mean()
    ax1.plot(monthly.index, ma3.values, 'k-', linewidth=2.5, label='3-month MA', zorder=5)
    legend_els = [Patch(color=v, label=f'{k} = {"Low (<33%)" if k=="L" else "Medium (33-66%)" if k=="M" else "High (>66%)"}')
                  for k,v in color_map.items()]
    ax1.legend(handles=legend_els + [plt.Line2D([],[],color='k',linewidth=2,label='3-mo MA')],
               loc='upper left', fontsize=8)
    ax1.set_title('Monthly Events Discretized into L/M/H Levels with 3-Month Moving Average')
    ax1.set_ylabel('Event Count')

    # Panel 2: Top 10 monthly motifs
    ax2 = fig.add_subplot(gs[1, 0])
    top10m = top_m_df.head(10)
    bars = ax2.barh(top10m['Pattern'], top10m['Frequency'],
                     color=[COLORS[{'2':0,'3':1,'4':2}.get(str(l),3)]
                            for l in top10m['Length']])
    ax2.set_title('Top Monthly Motifs\n(L/M/H sequences)', fontsize=10, fontweight='bold')
    ax2.set_xlabel('Frequency')
    legend_els2 = [Patch(color=COLORS[i], label=f'Length {l}')
                   for i,l in enumerate([2,3,4])]
    ax2.legend(handles=legend_els2, fontsize=7)

    # Panel 3: Top 10 weekly motifs
    ax3 = fig.add_subplot(gs[1, 1])
    top10w = top_w_df.head(10)
    ax3.barh(top10w['Pattern'], top10w['Frequency'], color=COLORS[1], alpha=0.85)
    ax3.set_title('Top Weekly Motifs\n(L/M/H sequences)', fontsize=10, fontweight='bold')
    ax3.set_xlabel('Frequency')

    # Panel 4: Season transition heatmap
    ax4 = fig.add_subplot(gs[1, 2])
    if trans_df is not None:
        sns.heatmap(trans_df, annot=True, fmt='.3f', cmap='Blues', ax=ax4,
                    square=True, cbar_kws={'label': 'Transition Prob.'})
        ax4.set_title('Season→Season Transition\nProbability Matrix', fontsize=10, fontweight='bold')
    else:
        ax4.text(0.5,0.5,'Season data\nnot available', ha='center', va='center', transform=ax4.transAxes)

    # Panel 5: Type × Hour heatmap
    ax5 = fig.add_subplot(gs[2, :2])
    if type_hour_pivot is not None:
        # Normalize per column
        norm_pivot = type_hour_pivot.div(type_hour_pivot.sum(axis=0), axis=1)
        sns.heatmap(norm_pivot.T, cmap='YlOrRd', ax=ax5, cbar_kws={'label': 'Normalized Count'})
        ax5.set_title('Closure Type × Hour of Day (Row-Normalized)', fontsize=10, fontweight='bold')
        ax5.set_xlabel('Hour'); ax5.set_ylabel('Closure Type')
    else:
        ax5.text(0.5,0.5,'Type data not available', ha='center', va='center', transform=ax5.transAxes)

    # Panel 6: Length distribution of motifs
    ax6 = fig.add_subplot(gs[2, 2])
    all_motifs = pd.DataFrame([((''.join(p)), l, len(p))
                                for p,l in motifs_m.items()],
                               columns=['Pattern','Freq','Length'])
    all_motifs.groupby('Length')['Freq'].sum().plot(kind='bar', ax=ax6, color=COLORS[2])
    ax6.set_title('Total Pattern Occurrences\nby Pattern Length', fontsize=10, fontweight='bold')
    ax6.set_xlabel('Pattern Length'); ax6.set_ylabel('Total Count')
    plt.xticks(rotation=0)

    plt.savefig(f"{OUTPUT_DIR}/mining/fig_M2_temporal_patterns.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("  [Fig M2] Temporal pattern mining saved.")
    print(f"\n  Top 10 Monthly Motifs:")
    print(top_m_df.head(10).to_string(index=False))
    return top_m_df, top_w_df, trans_df

# ============================================================
# M3. ASSOCIATION RULE MINING (Full Apriori + Visualization)
# ============================================================
def m3_association_rules(df, min_support=0.05, min_confidence=0.3):
    """
    Full Apriori-style association rule mining with:
    - Support, Confidence, Lift, Conviction, Leverage
    - Multi-level rules (2-item, 3-item antecedents)
    - Rule strength visualization
    """
    print("\n" + "="*60)
    print("[M3] ASSOCIATION RULE MINING (Apriori)")
    print("="*60)

    binary_cols = [c for c in ['Junction','Traffic_Signal','Crossing','Roundabout',
                                'Station','Railway','Stop','Amenity','Bump',
                                'IsWeekend','IsRushHour','IsNight',
                                'HasPrecipitation','LowVisibility',
                                'Traffic_Calming','No_Exit'] if c in df.columns]

    sample = df[binary_cols].dropna().sample(min(40000, len(df)), random_state=RANDOM_SEED)
    transactions = sample.astype(int)
    n = len(transactions)

    print(f"  Items: {binary_cols}")
    print(f"  Transactions: {n:,}")
    print(f"  Min support: {min_support}, Min confidence: {min_confidence}")

    # --- Level 1: Frequent single items ---
    support_1 = {col: transactions[col].mean() for col in binary_cols}
    frequent_1 = {k: v for k, v in support_1.items() if v >= min_support}
    print(f"  Frequent 1-itemsets: {len(frequent_1)}")

    # --- Level 2: Frequent pairs ---
    frequent_2 = {}
    for a, b in combinations(frequent_1.keys(), 2):
        sup = (transactions[a] & transactions[b]).mean()
        if sup >= min_support:
            frequent_2[(a,b)] = sup
    print(f"  Frequent 2-itemsets: {len(frequent_2)}")

    # --- Level 3: Frequent triples ---
    frequent_3 = {}
    freq_2_keys = list(frequent_2.keys())
    candidates_3 = set()
    for i in range(len(freq_2_keys)):
        for j in range(i+1, len(freq_2_keys)):
            union = tuple(sorted(set(freq_2_keys[i]) | set(freq_2_keys[j])))
            if len(union) == 3:
                candidates_3.add(union)
    for triple in candidates_3:
        a, b, c = triple
        sup = (transactions[a] & transactions[b] & transactions[c]).mean()
        if sup >= min_support:
            frequent_3[triple] = sup
    print(f"  Frequent 3-itemsets: {len(frequent_3)}")

    # --- Generate rules ---
    rules = []

    def add_rule(ant_list, con, sup_ant, sup_con, sup_both, n):
        if sup_ant == 0: return
        conf = sup_both / sup_ant
        if conf < min_confidence: return
        lift = conf / (sup_con + 1e-8)
        leverage = sup_both - (sup_ant * sup_con)
        conviction = (1 - sup_con) / (1 - conf + 1e-8) if conf < 1 else float('inf')
        rules.append({
            'Antecedent': ' + '.join(ant_list),
            'Consequent': con,
            'Support': round(sup_both, 4),
            'Confidence': round(conf, 4),
            'Lift': round(lift, 4),
            'Leverage': round(leverage, 4),
            'Conviction': round(min(conviction, 99.0), 4),
            'Rule_Length': len(ant_list) + 1
        })

    # 2-item rules: A→B and B→A
    for (a, b), sup_ab in frequent_2.items():
        add_rule([a], b, support_1[a], support_1[b], sup_ab, n)
        add_rule([b], a, support_1[b], support_1[a], sup_ab, n)

    # 3-item rules: {A,B}→C and A→{B,C} permutations
    for (a, b, c), sup_abc in frequent_3.items():
        sup_ab = frequent_2.get((a,b), frequent_2.get((b,a), 0))
        sup_ac = frequent_2.get((a,c), frequent_2.get((c,a), 0))
        sup_bc = frequent_2.get((b,c), frequent_2.get((c,b), 0))
        add_rule([a,b], c, sup_ab, support_1[c], sup_abc, n)
        add_rule([a,c], b, sup_ac, support_1[b], sup_abc, n)
        add_rule([b,c], a, sup_bc, support_1[a], sup_abc, n)

    rules_df = pd.DataFrame(rules).drop_duplicates(subset=['Antecedent','Consequent'])
    rules_df = rules_df.sort_values('Lift', ascending=False).reset_index(drop=True)
    rules_df.to_csv(f"{OUTPUT_DIR}/tables/M3_association_rules.csv", index=False)

    print(f"\n  Total rules generated: {len(rules_df)}")
    print(f"  High-lift rules (>2): {(rules_df['Lift'] > 2).sum()}")
    print("\n  Top 15 Rules by Lift:")
    print(rules_df.head(15)[['Antecedent','Consequent','Support','Confidence','Lift']].to_string(index=False))

    # --- FIGURE M3: 5-panel ---
    fig = plt.figure(figsize=(22, 14))
    fig.suptitle('M3: Association Rule Mining (Apriori) — Construction Event Attributes',
                 fontsize=15, fontweight='bold')
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

    # Panel 1: Support vs Confidence (color=Lift)
    ax1 = fig.add_subplot(gs[0, :2])
    sc = ax1.scatter(rules_df['Support'], rules_df['Confidence'],
                      c=rules_df['Lift'], cmap='plasma', s=40 * rules_df['Rule_Length'],
                      alpha=0.7, edgecolors='white', linewidths=0.3)
    plt.colorbar(sc, ax=ax1, label='Lift')
    ax1.set_xlabel('Support', fontsize=11); ax1.set_ylabel('Confidence', fontsize=11)
    ax1.set_title('Support vs Confidence\n(size = rule length, color = Lift)', fontsize=11, fontweight='bold')
    ax1.axhline(min_confidence, ls='--', color='red', alpha=0.6, label=f'min_conf={min_confidence}')
    ax1.axvline(min_support, ls='--', color='blue', alpha=0.6, label=f'min_sup={min_support}')
    # Annotate top rules
    for _, row in rules_df.head(8).iterrows():
        ax1.annotate(f"{row['Antecedent'][:12]}→{row['Consequent'][:8]}",
                     (row['Support'], row['Confidence']), fontsize=6, alpha=0.8)
    ax1.legend(fontsize=8)

    # Panel 2: Rule length distribution
    ax2 = fig.add_subplot(gs[0, 2])
    rules_df['Rule_Length'].value_counts().sort_index().plot(kind='bar', ax=ax2, color=COLORS[2])
    ax2.set_title('Rules by Length', fontsize=11, fontweight='bold')
    ax2.set_xlabel('Rule Length'); ax2.set_ylabel('Count'); plt.xticks(rotation=0)

    # Panel 3: Top 15 rules by lift (horizontal bar)
    ax3 = fig.add_subplot(gs[1, :2])
    top15 = rules_df.head(15)
    top15_label = top15['Antecedent'] + ' → ' + top15['Consequent']
    colors_lift = ['#d62728' if l > 3 else '#ff7f0e' if l > 2 else '#1f77b4'
                   for l in top15['Lift']]
    ax3.barh(top15_label, top15['Lift'], color=colors_lift)
    ax3.axvline(1.0, ls='--', color='gray', label='Lift=1 (independence)')
    ax3.set_title('Top 15 Association Rules by Lift', fontsize=11, fontweight='bold')
    ax3.set_xlabel('Lift')
    legend_els = [Patch(color='#d62728', label='Lift > 3'),
                  Patch(color='#ff7f0e', label='2 < Lift ≤ 3'),
                  Patch(color='#1f77b4', label='Lift ≤ 2')]
    ax3.legend(handles=legend_els, fontsize=8)

    # Panel 4: Item support bars
    ax4 = fig.add_subplot(gs[1, 2])
    sup_series = pd.Series(support_1).sort_values(ascending=True)
    ax4.barh(sup_series.index, sup_series.values * 100, color=COLORS[0])
    ax4.axvline(min_support * 100, ls='--', color='red', label=f'Min={min_support*100:.0f}%')
    ax4.set_title('Item Prevalence (%)', fontsize=11, fontweight='bold')
    ax4.set_xlabel('Prevalence (%)')
    ax4.legend(fontsize=8)

    plt.savefig(f"{OUTPUT_DIR}/mining/fig_M3_association_rules.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("  [Fig M3] Association rules saved.")
    return rules_df

# ============================================================
# M4. SPATIAL ANOMALY DETECTION (IsolationForest + LOF)
# ============================================================
def m4_spatial_anomaly(df):
    """
    Dual anomaly detection:
    1. Isolation Forest (global outliers)
    2. Local Outlier Factor (local density anomalies)
    Compare both and show spatial distribution.
    """
    print("\n" + "="*60)
    print("[M4] SPATIAL + TEMPORAL ANOMALY DETECTION (IF + LOF)")
    print("="*60)

    lat_min, lat_max = 24.5, 49.5
    lon_min, lon_max = -125.0, -66.0
    mask = ((df['Start_Lat'] >= lat_min) & (df['Start_Lat'] <= lat_max) &
            (df['Start_Lng'] >= lon_min) & (df['Start_Lng'] <= lon_max))

    features = [c for c in ['Start_Lat','Start_Lng','Duration_hrs','Hour',
                              'DistFromCenter','Log_Duration','Month',
                              'DayOfWeek','GridCellCount'] if c in df.columns]

    sample = df[mask][features + ['Start_Time']].dropna().sample(
        min(30000, len(df[mask])), random_state=RANDOM_SEED).copy()

    scaler = StandardScaler()
    X = scaler.fit_transform(sample[features])

    # --- Isolation Forest ---
    iso = IsolationForest(n_estimators=150, contamination=0.05,
                           random_state=RANDOM_SEED, n_jobs=-1)
    iso_labels = iso.fit_predict(X)
    iso_scores = -iso.decision_function(X)  # higher = more anomalous
    sample['IF_Anomaly'] = (iso_labels == -1).astype(int)
    sample['IF_Score'] = iso_scores

    # --- Local Outlier Factor ---
    lof = LocalOutlierFactor(n_neighbors=30, contamination=0.05, n_jobs=-1)
    lof_labels = lof.fit_predict(X)
    lof_scores = -lof.negative_outlier_factor_
    sample['LOF_Anomaly'] = (lof_labels == -1).astype(int)
    sample['LOF_Score'] = lof_scores

    # Agreement
    sample['Both_Anomaly'] = ((sample['IF_Anomaly'] == 1) & (sample['LOF_Anomaly'] == 1)).astype(int)
    n_if   = sample['IF_Anomaly'].sum()
    n_lof  = sample['LOF_Anomaly'].sum()
    n_both = sample['Both_Anomaly'].sum()

    print(f"  IF detected: {n_if} ({n_if/len(sample)*100:.1f}%)")
    print(f"  LOF detected: {n_lof} ({n_lof/len(sample)*100:.1f}%)")
    print(f"  Both agree: {n_both} ({n_both/len(sample)*100:.1f}%)")

    # Stats comparison
    groups = {
        'Normal (both)': sample[(sample['IF_Anomaly']==0) & (sample['LOF_Anomaly']==0)],
        'IF only': sample[(sample['IF_Anomaly']==1) & (sample['LOF_Anomaly']==0)],
        'LOF only': sample[(sample['IF_Anomaly']==0) & (sample['LOF_Anomaly']==1)],
        'Both (consensus)': sample[sample['Both_Anomaly']==1]
    }

    stats_rows = []
    for name, grp in groups.items():
        if len(grp) > 0:
            stats_rows.append({
                'Group': name, 'Count': len(grp),
                'Avg Duration (hrs)': round(grp['Duration_hrs'].mean(), 2),
                'Avg Hour': round(grp['Hour'].mean(), 1) if 'Hour' in grp else 'N/A',
                'Avg Lat': round(grp['Start_Lat'].mean(), 3),
                'Avg Lon': round(grp['Start_Lng'].mean(), 3)
            })
    stats_df = pd.DataFrame(stats_rows)
    stats_df.to_csv(f"{OUTPUT_DIR}/tables/M4_anomaly_comparison.csv", index=False)
    print("\n  Anomaly Group Comparison:")
    print(stats_df.to_string(index=False))

    # --- FIGURE M4: 6-panel ---
    fig = plt.figure(figsize=(22, 14))
    fig.suptitle('M4: Spatial & Temporal Anomaly Detection — Isolation Forest + LOF',
                 fontsize=15, fontweight='bold')
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

    # Panel 1: IF spatial map
    ax1 = fig.add_subplot(gs[0, 0])
    normal_if = sample[sample['IF_Anomaly'] == 0]
    anom_if   = sample[sample['IF_Anomaly'] == 1]
    ax1.scatter(normal_if['Start_Lng'], normal_if['Start_Lat'],
                s=0.5, alpha=0.2, color=COLORS[0], label='Normal')
    ax1.scatter(anom_if['Start_Lng'], anom_if['Start_Lat'],
                s=4, alpha=0.8, color=COLORS[3], label=f'Anomaly (n={n_if})')
    ax1.set_title('Isolation Forest Anomalies', fontsize=10, fontweight='bold')
    ax1.set_xlabel('Lon'); ax1.set_ylabel('Lat')
    ax1.legend(markerscale=5, fontsize=7)

    # Panel 2: LOF spatial map
    ax2 = fig.add_subplot(gs[0, 1])
    normal_lof = sample[sample['LOF_Anomaly'] == 0]
    anom_lof   = sample[sample['LOF_Anomaly'] == 1]
    ax2.scatter(normal_lof['Start_Lng'], normal_lof['Start_Lat'],
                s=0.5, alpha=0.2, color=COLORS[0], label='Normal')
    ax2.scatter(anom_lof['Start_Lng'], anom_lof['Start_Lat'],
                s=4, alpha=0.8, color=COLORS[4], label=f'Anomaly (n={n_lof})')
    ax2.set_title('LOF Anomalies', fontsize=10, fontweight='bold')
    ax2.set_xlabel('Lon'); ax2.set_ylabel('Lat')
    ax2.legend(markerscale=5, fontsize=7)

    # Panel 3: Consensus map
    ax3 = fig.add_subplot(gs[0, 2])
    norm_both  = sample[sample['Both_Anomaly'] == 0]
    anom_both  = sample[sample['Both_Anomaly'] == 1]
    ax3.scatter(norm_both['Start_Lng'], norm_both['Start_Lat'],
                s=0.5, alpha=0.15, color=COLORS[0])
    ax3.scatter(anom_both['Start_Lng'], anom_both['Start_Lat'],
                s=8, alpha=0.9, color='darkred', label=f'Consensus (n={n_both})', zorder=5)
    ax3.set_title('Consensus Anomalies (IF ∩ LOF)', fontsize=10, fontweight='bold')
    ax3.set_xlabel('Lon'); ax3.set_ylabel('Lat')
    ax3.legend(markerscale=3, fontsize=7)

    # Panel 4: Score distribution
    ax4 = fig.add_subplot(gs[1, 0])
    ax4.hist(sample[sample['IF_Anomaly']==0]['IF_Score'], bins=50,
             alpha=0.7, color=COLORS[0], label='Normal', density=True)
    ax4.hist(sample[sample['IF_Anomaly']==1]['IF_Score'], bins=50,
             alpha=0.7, color=COLORS[3], label='Anomaly', density=True)
    ax4.set_title('IF Score Distribution', fontsize=10, fontweight='bold')
    ax4.set_xlabel('Anomaly Score'); ax4.set_ylabel('Density'); ax4.legend()

    # Panel 5: Duration box by group
    ax5 = fig.add_subplot(gs[1, 1])
    sample['Group'] = 'Normal'
    sample.loc[sample['IF_Anomaly']==1, 'Group'] = 'IF Anomaly'
    sample.loc[sample['LOF_Anomaly']==1, 'Group'] = 'LOF Anomaly'
    sample.loc[sample['Both_Anomaly']==1, 'Group'] = 'Consensus'
    group_order = ['Normal','IF Anomaly','LOF Anomaly','Consensus']
    data_bp = [sample[sample['Group']==g]['Duration_hrs'].clip(0,100).values
               for g in group_order if g in sample['Group'].values]
    labels_bp = [g for g in group_order if g in sample['Group'].values]
    bp = ax5.boxplot(data_bp, labels=labels_bp, patch_artist=True,
                      medianprops={'color':'red','linewidth':2})
    for patch, color in zip(bp['boxes'], [COLORS[0],COLORS[3],COLORS[4],'darkred']):
        patch.set_facecolor(color); patch.set_alpha(0.6)
    ax5.set_title('Duration Distribution by Anomaly Group', fontsize=10, fontweight='bold')
    ax5.set_ylabel('Duration (hrs, clipped @100)')
    plt.setp(ax5.get_xticklabels(), rotation=20, ha='right')

    # Panel 6: Hourly anomaly rate
    ax6 = fig.add_subplot(gs[1, 2])
    hourly_rate = sample.groupby('Hour')['IF_Anomaly'].mean() * 100
    ax6.plot(hourly_rate.index, hourly_rate.values, 'o-', color=COLORS[3], linewidth=2)
    ax6.fill_between(hourly_rate.index, hourly_rate.values, alpha=0.3, color=COLORS[3])
    ax6.set_title('IF Anomaly Rate by Hour of Day', fontsize=10, fontweight='bold')
    ax6.set_xlabel('Hour (UTC)'); ax6.set_ylabel('Anomaly Rate (%)')
    ax6.set_xticks(range(0,24,2))

    plt.savefig(f"{OUTPUT_DIR}/mining/fig_M4_anomaly_detection.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("  [Fig M4] Dual anomaly detection saved.")
    return stats_df

# ============================================================
# M5. TEMPORAL ANOMALY DETECTION (Statistical)
# ============================================================
def m5_temporal_anomaly(df):
    """
    Statistical anomaly detection on time series:
    - Z-score method on monthly counts
    - IQR method on monthly counts
    - Rolling window deviation detection
    """
    print("\n" + "="*60)
    print("[M5] TEMPORAL ANOMALY DETECTION (Z-score + IQR)")
    print("="*60)

    df['Date'] = df['Start_Time'].dt.date
    daily = df.groupby('Date').size().reset_index(name='EventCount')
    daily['Date'] = pd.to_datetime(daily['Date'])
    daily = daily.set_index('Date').asfreq('D', fill_value=0)
    monthly = daily['EventCount'].resample('MS').sum().reset_index()
    monthly.columns = ['Date','Count']

    # Z-score
    mean_c = monthly['Count'].mean()
    std_c  = monthly['Count'].std()
    monthly['ZScore'] = (monthly['Count'] - mean_c) / std_c
    monthly['ZAnomaly'] = (monthly['ZScore'].abs() > 2.0).astype(int)

    # IQR
    Q1 = monthly['Count'].quantile(0.25)
    Q3 = monthly['Count'].quantile(0.75)
    IQR_val = Q3 - Q1
    monthly['IQR_Anomaly'] = ((monthly['Count'] < Q1 - 1.5*IQR_val) |
                               (monthly['Count'] > Q3 + 1.5*IQR_val)).astype(int)

    # Rolling (3-month) deviation
    monthly['Rolling_Mean'] = monthly['Count'].rolling(3, center=True).mean()
    monthly['Rolling_Std']  = monthly['Count'].rolling(3, center=True).std()
    monthly['Rolling_Dev']  = (monthly['Count'] - monthly['Rolling_Mean'])
    monthly['Rolling_Anomaly'] = (monthly['Rolling_Dev'].abs() >
                                   2 * monthly['Rolling_Std']).astype(int)

    monthly['Consensus_Anomaly'] = ((monthly['ZAnomaly'] + monthly['IQR_Anomaly'] +
                                      monthly['Rolling_Anomaly']) >= 2).astype(int)

    monthly.to_csv(f"{OUTPUT_DIR}/tables/M5_temporal_anomalies.csv", index=False)

    n_z = monthly['ZAnomaly'].sum()
    n_iq = monthly['IQR_Anomaly'].sum()
    n_ro = monthly['Rolling_Anomaly'].sum()
    n_co = monthly['Consensus_Anomaly'].sum()
    print(f"  Z-score anomalies: {n_z}")
    print(f"  IQR anomalies: {n_iq}")
    print(f"  Rolling anomalies: {n_ro}")
    print(f"  Consensus (≥2 methods): {n_co}")
    print("\n  Consensus Anomaly Months:")
    print(monthly[monthly['Consensus_Anomaly']==1][['Date','Count','ZScore']].to_string(index=False))

    # --- FIGURE M5 ---
    fig, axes = plt.subplots(3, 1, figsize=(18, 12))
    fig.suptitle('M5: Temporal Anomaly Detection — Statistical Methods',
                 fontsize=14, fontweight='bold')

    # Z-score
    axes[0].plot(monthly['Date'], monthly['Count'], color=COLORS[0], linewidth=1.5, label='Monthly Count')
    anom_z = monthly[monthly['ZAnomaly']==1]
    axes[0].scatter(anom_z['Date'], anom_z['Count'], color=COLORS[3], s=80, zorder=5,
                     label=f'Z-score Anomaly (n={n_z})', marker='D')
    axes[0].axhline(mean_c + 2*std_c, ls='--', color='orange', alpha=0.7, label='+2σ')
    axes[0].axhline(mean_c - 2*std_c, ls='--', color='orange', alpha=0.7, label='-2σ')
    axes[0].fill_between(monthly['Date'], mean_c - 2*std_c, mean_c + 2*std_c, alpha=0.1, color='orange')
    axes[0].set_title('Z-Score Method (|z| > 2.0)')
    axes[0].set_ylabel('Monthly Count'); axes[0].legend(fontsize=8)

    # IQR
    axes[1].plot(monthly['Date'], monthly['Count'], color=COLORS[1], linewidth=1.5, label='Monthly Count')
    anom_iq = monthly[monthly['IQR_Anomaly']==1]
    axes[1].scatter(anom_iq['Date'], anom_iq['Count'], color=COLORS[4], s=80, zorder=5,
                     label=f'IQR Anomaly (n={n_iq})', marker='s')
    axes[1].axhline(Q3 + 1.5*IQR_val, ls='--', color='purple', alpha=0.7, label='Q3+1.5×IQR')
    axes[1].axhline(Q1 - 1.5*IQR_val, ls='--', color='purple', alpha=0.7, label='Q1-1.5×IQR')
    axes[1].fill_between(monthly['Date'], Q1 - 1.5*IQR_val, Q3 + 1.5*IQR_val, alpha=0.1, color='purple')
    axes[1].set_title('IQR Method (Tukey Fences)')
    axes[1].set_ylabel('Monthly Count'); axes[1].legend(fontsize=8)

    # Consensus
    axes[2].plot(monthly['Date'], monthly['Count'], color=COLORS[2], linewidth=1.5, label='Monthly Count')
    axes[2].plot(monthly['Date'], monthly['Rolling_Mean'], 'k--', linewidth=2, label='Rolling Mean (3-mo)')
    anom_co = monthly[monthly['Consensus_Anomaly']==1]
    axes[2].scatter(anom_co['Date'], anom_co['Count'], color='darkred', s=120, zorder=5,
                     label=f'Consensus Anomaly (n={n_co})', marker='*')
    # Shade rolling ±2σ band
    rm = monthly['Rolling_Mean'].fillna(method='bfill').fillna(method='ffill')
    rs = monthly['Rolling_Std'].fillna(0)
    axes[2].fill_between(monthly['Date'], rm - 2*rs, rm + 2*rs, alpha=0.15, color='gray', label='±2σ band')
    axes[2].set_title('Consensus Anomalies (≥2 methods agree)')
    axes[2].set_ylabel('Monthly Count'); axes[2].legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/mining/fig_M5_temporal_anomalies.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("  [Fig M5] Temporal anomaly detection saved.")
    return monthly

# ============================================================
# M6. STATE TRANSITION / SPATIAL SEQUENCE MINING
# ============================================================
def m6_spatial_sequences(df):
    """
    Mine spatial sequences: which states tend to see construction
    events in sequence (event in state A followed by event in state B within same day/week)?
    Also: top state-pairs for same-day events.
    """
    print("\n" + "="*60)
    print("[M6] SPATIAL SEQUENCE MINING (State Transition Patterns)")
    print("="*60)

    if 'State' not in df.columns:
        print("  State column not found.")
        return None

    # Daily state event counts
    df_s = df[['Start_Time','State']].copy()
    df_s['Date'] = df_s['Start_Time'].dt.date

    # Top 15 states
    top_states = df['State'].value_counts().head(15).index.tolist()
    df_top = df_s[df_s['State'].isin(top_states)]

    # Day × State pivot
    day_state = df_top.groupby(['Date','State']).size().unstack(fill_value=0)
    # Binarize: was this state active today?
    day_state_bin = (day_state > 0).astype(int)

    # Co-activation: states active on same days
    coact = pd.DataFrame(index=top_states, columns=top_states, dtype=float)
    n_days = len(day_state_bin)
    for a, b in combinations(top_states, 2):
        if a in day_state_bin.columns and b in day_state_bin.columns:
            both = (day_state_bin[a] & day_state_bin[b]).sum()
            only_a = day_state_bin[a].sum()
            only_b = day_state_bin[b].sum()
            prev_a = only_a / n_days
            prev_b = only_b / n_days
            lift = (both / n_days) / (prev_a * prev_b + 1e-8)
            coact.loc[a, b] = lift
            coact.loc[b, a] = lift
    for s in top_states:
        coact.loc[s, s] = 1.0
    coact = coact.fillna(0).astype(float)

    # Transition matrix: day-to-day (which state was dominant today vs tomorrow)
    daily_dominant = day_state.idxmax(axis=1)  # dominant state per day
    trans_mat = np.zeros((len(top_states), len(top_states)), dtype=int)
    s2i = {s:i for i,s in enumerate(top_states)}
    for i in range(len(daily_dominant)-1):
        a = daily_dominant.iloc[i]
        b = daily_dominant.iloc[i+1]
        if a in s2i and b in s2i:
            trans_mat[s2i[a], s2i[b]] += 1

    trans_prob = trans_mat / (trans_mat.sum(axis=1, keepdims=True) + 1e-8)
    trans_df = pd.DataFrame(trans_prob, index=top_states, columns=top_states).round(4)
    trans_df.to_csv(f"{OUTPUT_DIR}/tables/M6_state_transition.csv")

    coact_top = []
    for a, b in combinations(top_states, 2):
        coact_top.append({'State_A': a, 'State_B': b, 'Co-activation Lift': round(coact.loc[a,b],4)})
    coact_df = pd.DataFrame(coact_top).sort_values('Co-activation Lift', ascending=False)
    coact_df.to_csv(f"{OUTPUT_DIR}/tables/M6_state_coactivation.csv", index=False)

    print(f"  Top 10 State Co-activation Pairs (by Lift):")
    print(coact_df.head(10).to_string(index=False))

    # --- FIGURE M6: 3-panel ---
    fig, axes = plt.subplots(1, 3, figsize=(22, 7))
    fig.suptitle('M6: Spatial Sequence Mining — State Co-activation & Transitions',
                 fontsize=14, fontweight='bold')

    # Panel 1: Co-activation heatmap
    sns.heatmap(coact, cmap='YlOrRd', ax=axes[0], square=True,
                xticklabels=True, yticklabels=True,
                cbar_kws={'label': 'Co-activation Lift'})
    axes[0].set_title('State Co-activation Lift Matrix\n(same-day events)', fontsize=10, fontweight='bold')
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right', fontsize=8)
    axes[0].set_yticklabels(axes[0].get_yticklabels(), fontsize=8)

    # Panel 2: Day-to-day dominant state transition
    sns.heatmap(trans_df, cmap='Blues', ax=axes[1], square=True,
                annot=True, fmt='.2f', annot_kws={'fontsize': 7},
                cbar_kws={'label': 'Transition Probability'})
    axes[1].set_title('Day-to-Day Dominant State\nTransition Probability Matrix', fontsize=10, fontweight='bold')
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right', fontsize=8)

    # Panel 3: Top co-activation pairs
    top20_coact = coact_df.head(20)
    top20_coact['Pair'] = top20_coact['State_A'] + '–' + top20_coact['State_B']
    axes[2].barh(top20_coact['Pair'], top20_coact['Co-activation Lift'], color=COLORS[2])
    axes[2].axvline(1.0, ls='--', color='red', label='Lift=1')
    axes[2].set_title('Top 20 State Pairs by\nCo-activation Lift', fontsize=10, fontweight='bold')
    axes[2].set_xlabel('Co-activation Lift'); axes[2].legend()

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/mining/fig_M6_spatial_sequences.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("  [Fig M6] Spatial sequence mining saved.")
    return coact_df, trans_df

# ============================================================
# M7. SPATIAL HOTSPOT ANALYSIS (Getis-Ord G* proxy)
# ============================================================
def m7_hotspot_analysis(df):
    """
    Getis-Ord G* statistic proxy on grid cells.
    High G* z-score → hot spot (more than expected given neighborhood).
    Low G* z-score  → cold spot.
    """
    print("\n" + "="*60)
    print("[M7] SPATIAL HOTSPOT ANALYSIS (Getis-Ord G* proxy)")
    print("="*60)

    # Aggregate to 1-degree grid
    df_g = df[['Start_Lat','Start_Lng']].copy()
    df_g['LatGrid'] = (df_g['Start_Lat'] // 1).astype(int)
    df_g['LonGrid'] = (df_g['Start_Lng'] // 1).astype(int)
    grid_counts = df_g.groupby(['LatGrid','LonGrid']).size().reset_index(name='Count')
    grid_counts = grid_counts[
        (grid_counts['LatGrid'] >= 24) & (grid_counts['LatGrid'] <= 49) &
        (grid_counts['LonGrid'] >= -125) & (grid_counts['LonGrid'] <= -66)
    ].copy()

    n = len(grid_counts)
    coords = grid_counts[['LatGrid','LonGrid']].values.astype(float)
    values = grid_counts['Count'].values.astype(float)

    # Spatial weights: inverse distance (d ≤ 3 degrees)
    from scipy.spatial import cKDTree
    tree = cKDTree(coords)
    pairs = tree.query_pairs(r=3.0)

    W = np.zeros((n, n))
    for i, j in pairs:
        d = np.sqrt(((coords[i] - coords[j])**2).sum())
        w = 1.0 / (d + 0.5)
        W[i, j] = w; W[j, i] = w

    # Compute Getis-Ord G* for each location
    x_bar = values.mean()
    S = np.sqrt(((values - x_bar)**2).mean())
    N = n

    Gi_star = np.zeros(n)
    for i in range(n):
        wi = W[i, :]
        sum_wij = wi.sum()
        sum_wij2 = (wi**2).sum()
        numerator = np.dot(wi, values) - x_bar * sum_wij
        denominator = S * np.sqrt(
            (N * sum_wij2 - sum_wij**2) / (N - 1) + 1e-8)
        Gi_star[i] = numerator / (denominator + 1e-8)

    grid_counts['Gi_star_z'] = Gi_star
    grid_counts['Hotspot_Class'] = 'Not Significant'
    grid_counts.loc[grid_counts['Gi_star_z'] > 2.576, 'Hotspot_Class'] = 'Hot Spot (99%)'
    grid_counts.loc[(grid_counts['Gi_star_z'] > 1.96) &
                     (grid_counts['Gi_star_z'] <= 2.576), 'Hotspot_Class'] = 'Hot Spot (95%)'
    grid_counts.loc[(grid_counts['Gi_star_z'] > 1.65) &
                     (grid_counts['Gi_star_z'] <= 1.96), 'Hotspot_Class'] = 'Hot Spot (90%)'
    grid_counts.loc[grid_counts['Gi_star_z'] < -2.576, 'Hotspot_Class'] = 'Cold Spot (99%)'
    grid_counts.loc[(grid_counts['Gi_star_z'] < -1.96) &
                     (grid_counts['Gi_star_z'] >= -2.576), 'Hotspot_Class'] = 'Cold Spot (95%)'

    grid_counts.to_csv(f"{OUTPUT_DIR}/tables/M7_hotspots.csv", index=False)

    hs_summary = grid_counts['Hotspot_Class'].value_counts()
    print("\n  Hotspot Classification:")
    print(hs_summary.to_string())

    # --- FIGURE M7: 3-panel ---
    fig, axes = plt.subplots(1, 3, figsize=(22, 7))
    fig.suptitle("M7: Spatial Hotspot Analysis — Getis-Ord G* Statistic",
                 fontsize=14, fontweight='bold')

    # Panel 1: G* z-score map
    hot_colors = {
        'Hot Spot (99%)': '#d73027',
        'Hot Spot (95%)': '#fc8d59',
        'Hot Spot (90%)': '#fee090',
        'Not Significant': '#e0e0e0',
        'Cold Spot (95%)': '#91bfdb',
        'Cold Spot (99%)': '#4575b4'
    }
    for cls, grp in grid_counts.groupby('Hotspot_Class'):
        axes[0].scatter(grp['LonGrid'] + 0.5, grp['LatGrid'] + 0.5,
                         c=hot_colors.get(cls, 'gray'), s=15, label=cls, alpha=0.9, zorder=3)
    axes[0].set_xlim(-126, -65); axes[0].set_ylim(23, 50)
    axes[0].set_title('Hot/Cold Spots (Getis-Ord G*)', fontsize=10, fontweight='bold')
    axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')
    axes[0].legend(fontsize=6, loc='lower right', markerscale=2)

    # Panel 2: G* z-score distribution
    axes[1].hist(Gi_star, bins=40, color=COLORS[0], edgecolor='white', alpha=0.85)
    axes[1].axvline(2.576, ls='--', color='red', label='99% (z=2.576)')
    axes[1].axvline(1.96, ls='--', color='orange', label='95% (z=1.96)')
    axes[1].axvline(-2.576, ls='--', color='blue', label='Cold 99%')
    axes[1].set_title('G* Z-score Distribution', fontsize=10, fontweight='bold')
    axes[1].set_xlabel('G* Z-score'); axes[1].set_ylabel('Frequency'); axes[1].legend(fontsize=8)

    # Panel 3: Class distribution
    hs_summary_plot = grid_counts['Hotspot_Class'].value_counts()
    colors_pie = [hot_colors.get(k, 'gray') for k in hs_summary_plot.index]
    axes[2].pie(hs_summary_plot.values, labels=hs_summary_plot.index,
                autopct='%1.1f%%', colors=colors_pie, startangle=90)
    axes[2].set_title('Hotspot Class Distribution', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/mining/fig_M7_hotspots.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("  [Fig M7] Hotspot analysis saved.")
    return grid_counts

# ============================================================
# M8. CLUSTER-BASED PATTERN PROFILING
# ============================================================
def m8_cluster_profiling(df):
    """
    K-Means on (lat, lon) → profile each cluster by:
    temporal patterns, closure type, weather, duration.
    Tells us not just WHERE clusters are, but WHAT kind of events happen there.
    """
    print("\n" + "="*60)
    print("[M8] CLUSTER-BASED PATTERN PROFILING")
    print("="*60)

    lat_min, lat_max = 24.5, 49.5
    lon_min, lon_max = -125.0, -66.0
    mask = ((df['Start_Lat'] >= lat_min) & (df['Start_Lat'] <= lat_max) &
            (df['Start_Lng'] >= lon_min) & (df['Start_Lng'] <= lon_max))

    sample = df[mask].sample(min(40000, len(df[mask])), random_state=RANDOM_SEED).copy()

    # K-Means with k=6 (geographic US regions)
    coords = sample[['Start_Lat','Start_Lng']].values
    scaler = StandardScaler()
    coords_s = scaler.fit_transform(coords)
    km = KMeans(n_clusters=6, random_state=RANDOM_SEED, n_init=15)
    sample['Cluster'] = km.fit_predict(coords_s)

    # Region names heuristic
    centers = scaler.inverse_transform(km.cluster_centers_)
    def region_name(lat, lon):
        if lon < -115: return 'Pacific West'
        elif lon < -105: return 'Mountain West'
        elif lon < -95: return 'Great Plains'
        elif lon < -85: return 'Midwest'
        elif lon < -75: return 'Southeast'
        else: return 'Northeast'
    sample['Region'] = sample.apply(lambda r: region_name(r['Start_Lat'], r['Start_Lng']), axis=1)

    # Profile each cluster
    profile_cols = [c for c in ['Duration_hrs','Hour','Month','IsWeekend','IsRushHour',
                                  'IsNight','Temperature(F)','Humidity(%)'] if c in sample.columns]
    profile = sample.groupby('Region')[profile_cols].mean().round(3)
    profile['N_Events'] = sample.groupby('Region').size()
    profile.to_csv(f"{OUTPUT_DIR}/tables/M8_cluster_profiles.csv")

    print("\n  Cluster Regional Profiles:")
    print(profile.to_string())

    # Type distribution per cluster
    if 'Type' in sample.columns:
        type_cluster = sample.groupby(['Region','Type']).size().reset_index(name='Count')
        type_pivot = type_cluster.pivot(index='Region', columns='Type', values='Count').fillna(0)
        type_pivot_pct = type_pivot.div(type_pivot.sum(axis=1), axis=0) * 100
        type_pivot_pct.to_csv(f"{OUTPUT_DIR}/tables/M8_type_by_cluster.csv")

    # --- FIGURE M8: 4-panel ---
    fig = plt.figure(figsize=(22, 14))
    fig.suptitle('M8: Cluster-Based Pattern Profiling — Regional Analysis',
                 fontsize=14, fontweight='bold')
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

    regions = sample['Region'].unique()
    region_colors = {r: COLORS[i] for i, r in enumerate(sorted(regions))}

    # Panel 1: Spatial cluster map
    ax1 = fig.add_subplot(gs[0, :2])
    for region, grp in sample.groupby('Region'):
        ax1.scatter(grp['Start_Lng'], grp['Start_Lat'], s=0.8, alpha=0.4,
                     color=region_colors[region], label=region)
    ax1.set_xlim(lon_min, lon_max); ax1.set_ylim(lat_min, lat_max)
    ax1.set_title('Geographic Clusters (6 US Regions)', fontsize=11, fontweight='bold')
    ax1.set_xlabel('Longitude'); ax1.set_ylabel('Latitude')
    ax1.legend(fontsize=8, markerscale=5)

    # Panel 2: Avg Duration by region
    ax2 = fig.add_subplot(gs[0, 2])
    dur_by_region = sample.groupby('Region')['Duration_hrs'].mean().sort_values()
    ax2.barh(dur_by_region.index, dur_by_region.values,
              color=[region_colors[r] for r in dur_by_region.index])
    ax2.set_title('Avg Event Duration by Region', fontsize=11, fontweight='bold')
    ax2.set_xlabel('Avg Duration (hrs)')

    # Panel 3: Type distribution by cluster
    ax3 = fig.add_subplot(gs[1, :2])
    if 'Type' in sample.columns and 'type_pivot_pct' in dir():
        type_pivot_pct.plot(kind='bar', stacked=True, ax=ax3, colormap='tab10')
        ax3.set_title('Closure Type Distribution by Region (%)', fontsize=11, fontweight='bold')
        ax3.set_xlabel('Region'); ax3.set_ylabel('%')
        ax3.legend(title='Type', fontsize=8, loc='upper right')
        plt.setp(ax3.get_xticklabels(), rotation=30, ha='right')
    else:
        ax3.text(0.5,0.5,'Type data not available',ha='center',va='center',transform=ax3.transAxes)

    # Panel 4: Hour distribution by region
    ax4 = fig.add_subplot(gs[1, 2])
    for region, grp in sample.groupby('Region'):
        hour_dist = grp['Hour'].value_counts().sort_index()
        ax4.plot(hour_dist.index, hour_dist.values / hour_dist.sum(),
                  color=region_colors[region], linewidth=1.5, label=region, alpha=0.8)
    ax4.set_title('Hourly Event Pattern by Region', fontsize=11, fontweight='bold')
    ax4.set_xlabel('Hour of Day'); ax4.set_ylabel('Proportion')
    ax4.legend(fontsize=7)

    plt.savefig(f"{OUTPUT_DIR}/mining/fig_M8_cluster_profiling.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("  [Fig M8] Cluster profiling saved.")
    return profile

# ============================================================
# J1. SPATIO-TEMPORAL HOTSPOT HEATMAP (Grid × Month)
# ============================================================
def j1_spatiotemporal_hotspot(df):
    print("\n" + "="*60)
    print("[J1] JOINT: Spatio-Temporal Hotspot Heatmap")
    print("="*60)

    df_j = df.copy()
    df_j['LatBin'] = (df_j['Start_Lat'] // 2).astype(int) * 2
    df_j['LonBin'] = (df_j['Start_Lng'] // 3).astype(int) * 3

    lat_min, lat_max = 24, 50
    lon_min, lon_max = -126, -66
    mask = ((df_j['LatBin'] >= lat_min) & (df_j['LatBin'] <= lat_max) &
            (df_j['LonBin'] >= lon_min) & (df_j['LonBin'] <= lon_max))
    df_j = df_j[mask].copy()
    df_j['GeoCell'] = df_j['LatBin'].astype(str) + '°N / ' + (-df_j['LonBin']).astype(str) + '°W'

    top_cells = df_j['GeoCell'].value_counts().head(12).index
    df_top = df_j[df_j['GeoCell'].isin(top_cells)]

    pivot = df_top.groupby(['GeoCell','Month']).size().reset_index(name='Count')
    pivot_t = pivot.pivot(index='GeoCell', columns='Month', values='Count').fillna(0)
    month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    pivot_t.columns = [month_names[int(c)-1] for c in pivot_t.columns if int(c) <= 12]

    # Normalize by row (relative intensity)
    pivot_norm = pivot_t.div(pivot_t.sum(axis=1), axis=0)

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    fig.suptitle('J1: Spatio-Temporal Hotspot Analysis — Grid × Month',
                 fontsize=14, fontweight='bold')

    sns.heatmap(pivot_t, cmap='YlOrRd', ax=axes[0], linewidths=0.3,
                cbar_kws={'label': 'Event Count'}, annot=True, fmt='.0f', annot_kws={'fontsize': 8})
    axes[0].set_title('Absolute Event Count: Top 12 Geographic Cells × Month',
                       fontsize=11, fontweight='bold')
    axes[0].set_xlabel('Month'); axes[0].set_ylabel('Geographic Cell (Lat/Lon bin)')

    sns.heatmap(pivot_norm, cmap='Blues', ax=axes[1], linewidths=0.3,
                cbar_kws={'label': 'Relative Intensity'}, annot=True, fmt='.2f', annot_kws={'fontsize': 8})
    axes[1].set_title('Normalized Event Intensity (Row-normalized)',
                       fontsize=11, fontweight='bold')
    axes[1].set_xlabel('Month'); axes[1].set_ylabel('')

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/joint/fig_J1_spatiotemporal_hotspot.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("  [Fig J1] Spatio-temporal hotspot saved.")
    pivot_t.to_csv(f"{OUTPUT_DIR}/tables/J1_hotspot_matrix.csv")
    return pivot_t

# ============================================================
# J2. JOINT ML MODEL (Full Comparison: Spatial vs Temporal vs Joint)
# ============================================================
def j2_joint_ml_model(df):
    """
    3-way comparison:
    - Spatial features only
    - Temporal features only
    - Joint (spatial + temporal + weather)
    For closure type prediction AND duration regression.
    """
    print("\n" + "="*60)
    print("[J2] JOINT ML: 3-Way Feature Set Comparison")
    print("="*60)

    spatial_feats  = [c for c in ['Start_Lat','Start_Lng','DistFromCenter',
                                    'GridCellCount','HexCellCount','POI_Count','IsUrban']
                      if c in df.columns]
    temporal_feats = [c for c in ['Hour_sin','Hour_cos','Month_sin','Month_cos',
                                    'DOW_sin','DOW_cos','Week_sin','Week_cos',
                                    'IsWeekend','IsRushHour','IsNight','Year']
                      if c in df.columns]
    weather_feats  = [c for c in ['Temperature(F)','Humidity(%)','Wind_Speed(mph)',
                                    'Visibility(mi)','Precipitation(in)']
                      if c in df.columns]
    joint_feats    = spatial_feats + temporal_feats + weather_feats

    target_cls = 'Type' if 'Type' in df.columns else 'Severity'
    target_reg = 'Log_Duration'

    if target_cls not in df.columns:
        print("  Target not found. Skipping.")
        return None

    results_cls = []
    results_reg = []

    feature_sets = {
        'Spatial Only': spatial_feats,
        'Temporal Only': temporal_feats,
        'Weather Only': weather_feats,
        'Spatial + Temporal': spatial_feats + temporal_feats,
        'Spatial + Weather': spatial_feats + weather_feats,
        'Temporal + Weather': temporal_feats + weather_feats,
        'Joint (All)': joint_feats
    }

    le = LabelEncoder()
    df_clean = df[joint_feats + [target_cls, target_reg]].dropna()
    if len(df_clean) > 80000:
        df_clean = df_clean.sample(80000, random_state=RANDOM_SEED)
    y_cls = le.fit_transform(df_clean[target_cls].astype(str))
    y_reg = df_clean[target_reg].values

    for name, feats in feature_sets.items():
        feats_avail = [f for f in feats if f in df_clean.columns]
        if len(feats_avail) < 2:
            continue
        X = df_clean[feats_avail].values
        scaler = StandardScaler()
        X_s = scaler.fit_transform(X)

        X_tr, X_te, y_tr_c, y_te_c, y_tr_r, y_te_r = train_test_split(
            X_s, y_cls, y_reg, test_size=0.2, random_state=RANDOM_SEED)

        # Classification
        rf_c = RandomForestClassifier(n_estimators=100, max_depth=10,
                                       random_state=RANDOM_SEED, n_jobs=-1,
                                       class_weight='balanced')
        rf_c.fit(X_tr, y_tr_c)
        y_pred_c = rf_c.predict(X_te)
        acc  = accuracy_score(y_te_c, y_pred_c)
        f1m  = f1_score(y_te_c, y_pred_c, average='macro', zero_division=0)
        f1w  = f1_score(y_te_c, y_pred_c, average='weighted', zero_division=0)
        results_cls.append({'Feature Set': name, 'N Features': len(feats_avail),
                             'Accuracy': round(acc,4), 'F1 Macro': round(f1m,4),
                             'F1 Weighted': round(f1w,4)})

        # Regression
        rf_r = RandomForestRegressor(n_estimators=100, max_depth=10,
                                      random_state=RANDOM_SEED, n_jobs=-1)
        rf_r.fit(X_tr, y_tr_r)
        y_pred_r = rf_r.predict(X_te)
        rmse = np.sqrt(mean_squared_error(y_te_r, y_pred_r))
        r2   = r2_score(y_te_r, y_pred_r)
        results_reg.append({'Feature Set': name, 'N Features': len(feats_avail),
                             'RMSE (log)': round(rmse,4), 'R²': round(r2,4)})

        print(f"  {name}: Cls F1={f1m:.4f} | Reg R²={r2:.4f}")

    cls_df = pd.DataFrame(results_cls)
    reg_df = pd.DataFrame(results_reg)
    cls_df.to_csv(f"{OUTPUT_DIR}/tables/J2_classification_comparison.csv", index=False)
    reg_df.to_csv(f"{OUTPUT_DIR}/tables/J2_regression_comparison.csv", index=False)

    print("\n  Classification Comparison:")
    print(cls_df.to_string(index=False))
    print("\n  Regression Comparison:")
    print(reg_df.to_string(index=False))

    # --- FIGURE J2: 2×3 grid ---
    fig, axes = plt.subplots(2, 3, figsize=(22, 12))
    fig.suptitle('J2: Joint ML — Feature Set Comparison (Spatial vs Temporal vs Joint)',
                 fontsize=14, fontweight='bold')

    # Cls: Accuracy
    colors_fs = [COLORS[i % len(COLORS)] for i in range(len(cls_df))]
    axes[0,0].barh(cls_df['Feature Set'], cls_df['Accuracy'], color=colors_fs)
    axes[0,0].set_title('Classification: Accuracy', fontsize=10, fontweight='bold')
    axes[0,0].set_xlabel('Accuracy'); axes[0,0].set_xlim(0, 1)
    for i, v in enumerate(cls_df['Accuracy']):
        axes[0,0].text(v + 0.005, i, f'{v:.4f}', va='center', fontsize=8)

    # Cls: F1 Macro
    axes[0,1].barh(cls_df['Feature Set'], cls_df['F1 Macro'], color=colors_fs)
    axes[0,1].set_title('Classification: F1 Macro', fontsize=10, fontweight='bold')
    axes[0,1].set_xlabel('F1 Macro'); axes[0,1].set_xlim(0, 1)
    for i, v in enumerate(cls_df['F1 Macro']):
        axes[0,1].text(v + 0.005, i, f'{v:.4f}', va='center', fontsize=8)

    # Cls: F1 Weighted
    axes[0,2].barh(cls_df['Feature Set'], cls_df['F1 Weighted'], color=colors_fs)
    axes[0,2].set_title('Classification: F1 Weighted', fontsize=10, fontweight='bold')
    axes[0,2].set_xlabel('F1 Weighted'); axes[0,2].set_xlim(0, 1)

    # Reg: RMSE
    axes[1,0].barh(reg_df['Feature Set'], reg_df['RMSE (log)'], color=colors_fs)
    axes[1,0].set_title('Regression: RMSE (log scale)', fontsize=10, fontweight='bold')
    axes[1,0].set_xlabel('RMSE')
    for i, v in enumerate(reg_df['RMSE (log)']):
        axes[1,0].text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=8)

    # Reg: R²
    axes[1,1].barh(reg_df['Feature Set'], reg_df['R²'], color=colors_fs)
    axes[1,1].set_title('Regression: R²', fontsize=10, fontweight='bold')
    axes[1,1].set_xlabel('R²')
    for i, v in enumerate(reg_df['R²']):
        axes[1,1].text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=8)

    # Feature set size
    axes[1,2].barh(cls_df['Feature Set'], cls_df['N Features'], color=COLORS[5], alpha=0.8)
    axes[1,2].set_title('Feature Count per Set', fontsize=10, fontweight='bold')
    axes[1,2].set_xlabel('# Features')
    for i, v in enumerate(cls_df['N Features']):
        axes[1,2].text(v + 0.1, i, str(v), va='center', fontsize=9)

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/joint/fig_J2_feature_set_comparison.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("  [Fig J2] Joint ML comparison saved.")
    return cls_df, reg_df

# ============================================================
# J3. SPATIAL-TEMPORAL INTERACTION ANALYSIS
# ============================================================
def j3_spatial_temporal_interaction(df):
    """
    Does geography amplify temporal effects?
    - Duration varies by hour differently across regions?
    - Are rush-hour effects more pronounced in urban areas?
    - Correlation: spatial density vs temporal frequency
    """
    print("\n" + "="*60)
    print("[J3] JOINT: Spatial-Temporal Interaction Analysis")
    print("="*60)

    lat_min, lat_max = 24.5, 49.5
    lon_min, lon_max = -125.0, -66.0
    mask = ((df['Start_Lat'] >= lat_min) & (df['Start_Lat'] <= lat_max) &
            (df['Start_Lng'] >= lon_min) & (df['Start_Lng'] <= lon_max))
    sample = df[mask].sample(min(40000, len(df[mask])), random_state=RANDOM_SEED).copy()

    # Region assignment
    def region(lon):
        if lon < -115: return 'West'
        elif lon < -100: return 'Mountain'
        elif lon < -90: return 'Midwest'
        elif lon < -80: return 'South'
        else: return 'Northeast'
    sample['Region5'] = sample['Start_Lng'].apply(region)

    # Urban/rural proxy
    if 'IsUrban' not in sample.columns:
        if 'GridCellCount' in sample.columns:
            q75 = sample['GridCellCount'].quantile(0.75)
            sample['IsUrban'] = (sample['GridCellCount'] > q75).astype(int)
        else:
            sample['IsUrban'] = 0

    # --- Analysis 1: Duration by Hour × Region ---
    hour_region = sample.groupby(['Region5','Hour'])['Duration_hrs'].mean().reset_index()
    hour_pivot = hour_region.pivot(index='Hour', columns='Region5', values='Duration_hrs')

    # --- Analysis 2: Rush-hour effect by urban/rural ---
    if 'IsRushHour' in sample.columns:
        rush_effect = sample.groupby(['IsUrban','IsRushHour'])['Duration_hrs'].mean().reset_index()
        rush_effect['IsUrban'] = rush_effect['IsUrban'].map({0:'Rural',1:'Urban'})
        rush_effect['IsRushHour'] = rush_effect['IsRushHour'].map({0:'Non-Rush',1:'Rush Hour'})

    # --- Analysis 3: Pearson correlation (spatial density vs temporal frequency) ---
    if 'GridCellCount' in sample.columns and 'StateMonthCount' in sample.columns:
        r, p = pearsonr(sample['GridCellCount'].fillna(0),
                         sample['StateMonthCount'].fillna(0))
        print(f"  Spatial density vs Temporal frequency: r={r:.4f}, p={p:.4e}")

    # --- Analysis 4: Seasonal effect differs by region? ---
    if 'Season' in sample.columns:
        season_region = sample.groupby(['Region5','Season'])['Duration_hrs'].mean().reset_index()
        season_pivot = season_region.pivot(index='Season', columns='Region5', values='Duration_hrs')
        season_order = ['Winter','Spring','Summer','Fall']
        season_pivot = season_pivot.reindex([s for s in season_order if s in season_pivot.index])

    # --- FIGURE J3: 4-panel ---
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    fig.suptitle('J3: Spatial-Temporal Interaction Analysis',
                 fontsize=14, fontweight='bold')

    # Panel 1: Duration by Hour × Region
    for region, col in hour_pivot.items():
        axes[0,0].plot(col.index, col.values, marker='o', linewidth=2,
                        markersize=4, label=region)
    axes[0,0].set_title('Avg Duration by Hour × Geographic Region', fontsize=11, fontweight='bold')
    axes[0,0].set_xlabel('Hour of Day (UTC)'); axes[0,0].set_ylabel('Avg Duration (hrs)')
    axes[0,0].legend(fontsize=8); axes[0,0].set_xticks(range(0,24,2))

    # Panel 2: Rush-hour effect by urban/rural
    if 'IsRushHour' in sample.columns:
        rush_pivot2 = rush_effect.pivot(index='IsRushHour', columns='IsUrban', values='Duration_hrs')
        rush_pivot2.plot(kind='bar', ax=axes[0,1], color=COLORS[:2])
        axes[0,1].set_title('Rush-Hour Effect: Urban vs Rural', fontsize=11, fontweight='bold')
        axes[0,1].set_xlabel(''); axes[0,1].set_ylabel('Avg Duration (hrs)')
        axes[0,1].legend(title='Area Type')
        plt.setp(axes[0,1].get_xticklabels(), rotation=0)

    # Panel 3: Season × Region heatmap
    if 'Season' in sample.columns and 'season_pivot' in dir():
        sns.heatmap(season_pivot, annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[1,0],
                    cbar_kws={'label': 'Avg Duration (hrs)'})
        axes[1,0].set_title('Avg Duration: Season × Region', fontsize=11, fontweight='bold')

    # Panel 4: Spatial density vs monthly temporal volume scatter
    if 'State' in sample.columns and 'StateMonthCount' in sample.columns:
        sc4 = axes[1,1].scatter(sample['GridCellCount'].fillna(0).clip(0, 5000),
                                 sample['StateMonthCount'].fillna(0).clip(0, 50000),
                                 c=sample['Log_Duration'], cmap='plasma',
                                 s=1, alpha=0.2)
        plt.colorbar(sc4, ax=axes[1,1], label='log(Duration)')
        axes[1,1].set_title('Spatial Density vs Temporal Volume\n(color = log Duration)',
                              fontsize=11, fontweight='bold')
        axes[1,1].set_xlabel('Grid Cell Event Count'); axes[1,1].set_ylabel('State-Month Event Count')

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/joint/fig_J3_spatiotemporal_interaction.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("  [Fig J3] Spatial-temporal interaction saved.")

# ============================================================
# J4. 3D SPATIOTEMPORAL CLUSTERING (DBSCAN on lat/lon/time)
# ============================================================
def j4_spatiotemporal_dbscan(df):
    """
    DBSCAN in 3D space (lat, lon, time) to find spatiotemporal clusters.
    Time scaled to degree-equivalent.
    """
    print("\n" + "="*60)
    print("[J4] JOINT: 3D Spatiotemporal DBSCAN Clustering")
    print("="*60)

    lat_min, lat_max = 24.5, 49.5
    lon_min, lon_max = -125.0, -66.0
    mask = ((df['Start_Lat'] >= lat_min) & (df['Start_Lat'] <= lat_max) &
            (df['Start_Lng'] >= lon_min) & (df['Start_Lng'] <= lon_max))

    sample = df[mask].sample(min(20000, len(df[mask])), random_state=RANDOM_SEED).copy()

    # Convert time to fractional year
    ref = pd.Timestamp('2016-01-01')
    sample['T_frac'] = (sample['Start_Time'] - ref).dt.total_seconds() / (365.25 * 86400)

    # Scale: time 1 year ≈ 10 degrees (tune to balance spatial vs temporal)
    coords_3d = np.column_stack([sample['Start_Lat'].values,
                                  sample['Start_Lng'].values,
                                  sample['T_frac'].values * 10])

    scaler = StandardScaler()
    coords_3d_s = scaler.fit_transform(coords_3d)

    # DBSCAN
    db3d = DBSCAN(eps=0.5, min_samples=40, n_jobs=-1)
    labels_3d = db3d.fit_predict(coords_3d_s)
    sample['ST_Cluster'] = labels_3d

    n_clusters = len(set(labels_3d)) - (1 if -1 in labels_3d else 0)
    n_noise    = (labels_3d == -1).sum()
    print(f"  3D DBSCAN: {n_clusters} clusters, {n_noise} noise ({n_noise/len(labels_3d)*100:.1f}%)")

    # Profile top clusters
    cluster_profiles = []
    for cl in sorted(set(labels_3d)):
        if cl == -1: continue
        grp = sample[sample['ST_Cluster'] == cl]
        cluster_profiles.append({
            'Cluster': int(cl),
            'N Events': len(grp),
            'Avg Lat': round(grp['Start_Lat'].mean(), 2),
            'Avg Lon': round(grp['Start_Lng'].mean(), 2),
            'Year Range': f"{grp['Year'].min() if 'Year' in grp else '?'}–{grp['Year'].max() if 'Year' in grp else '?'}",
            'Avg Month': round(grp['Month'].mean(), 1) if 'Month' in grp else 'N/A',
            'Avg Duration': round(grp['Duration_hrs'].mean(), 2)
        })
    cl_prof_df = pd.DataFrame(cluster_profiles)
    cl_prof_df.to_csv(f"{OUTPUT_DIR}/tables/J4_3d_cluster_profiles.csv", index=False)
    print("\n  Top 3D Spatiotemporal Clusters:")
    print(cl_prof_df.head(10).to_string(index=False))

    # --- FIGURE J4: 3-panel ---
    fig, axes = plt.subplots(1, 3, figsize=(22, 7))
    fig.suptitle('J4: 3D Spatiotemporal DBSCAN Clustering (Lat × Lon × Time)',
                 fontsize=14, fontweight='bold')

    # Spatial projection
    noise = sample[sample['ST_Cluster'] == -1]
    non_noise = sample[sample['ST_Cluster'] >= 0]
    axes[0].scatter(noise['Start_Lng'], noise['Start_Lat'],
                     s=0.3, alpha=0.15, color='lightgray', label='Noise')
    if len(non_noise) > 0:
        cmap_cl = plt.cm.get_cmap('tab20', max(n_clusters, 1))
        sc = axes[0].scatter(non_noise['Start_Lng'], non_noise['Start_Lat'],
                              c=non_noise['ST_Cluster'], cmap='tab20', s=2, alpha=0.7,
                              vmin=0, vmax=max(n_clusters,1))
        plt.colorbar(sc, ax=axes[0], label='Cluster ID')
    axes[0].set_xlim(lon_min, lon_max); axes[0].set_ylim(lat_min, lat_max)
    axes[0].set_title(f'Spatial Projection\n({n_clusters} ST-clusters)', fontsize=10, fontweight='bold')
    axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')

    # Time projection (lat vs time)
    axes[1].scatter(noise['T_frac'], noise['Start_Lat'],
                     s=0.3, alpha=0.1, color='lightgray')
    if len(non_noise) > 0:
        axes[1].scatter(non_noise['T_frac'], non_noise['Start_Lat'],
                         c=non_noise['ST_Cluster'], cmap='tab20', s=2, alpha=0.7,
                         vmin=0, vmax=max(n_clusters,1))
    axes[1].set_title('Temporal × Latitude Projection', fontsize=10, fontweight='bold')
    axes[1].set_xlabel('Time (fractional year from 2016)'); axes[1].set_ylabel('Latitude')

    # Cluster size distribution
    if n_clusters > 0:
        cl_sizes = sample[sample['ST_Cluster'] >= 0]['ST_Cluster'].value_counts().sort_index()
        axes[2].bar(cl_sizes.index.astype(str), cl_sizes.values, color=COLORS[2])
        axes[2].set_title('Cluster Size Distribution', fontsize=10, fontweight='bold')
        axes[2].set_xlabel('Cluster ID'); axes[2].set_ylabel('Event Count')
        if len(cl_sizes) > 20:
            axes[2].set_xticks([])
    else:
        axes[2].text(0.5,0.5,'No clusters found\nTry adjusting eps/min_samples',
                     ha='center',va='center',transform=axes[2].transAxes)

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/joint/fig_J4_3d_dbscan.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("  [Fig J4] 3D ST-DBSCAN saved.")
    return cl_prof_df

# ============================================================
# J5. PREDICTIVE HOTSPOT MODEL (Next-Month by Grid Cell)
# ============================================================
def j5_hotspot_prediction(df):
    """
    Predict whether a grid cell will be a hotspot next month.
    Uses: last-month count, 3-month trend, seasonal features.
    Binary classification: hot (top-25%) vs not.
    """
    print("\n" + "="*60)
    print("[J5] JOINT: Predictive Hotspot Model (Next Month)")
    print("="*60)

    df_h = df.copy()
    df_h['LatGrid'] = (df_h['Start_Lat'] // 1).astype(int)
    df_h['LonGrid'] = (df_h['Start_Lng'] // 1).astype(int)
    df_h['YM'] = df_h['Start_Time'].dt.to_period('M')

    grid_monthly = df_h.groupby(['LatGrid','LonGrid','YM']).size().reset_index(name='Count')
    grid_monthly['YM_int'] = grid_monthly['YM'].dt.to_timestamp().apply(
        lambda x: x.year * 12 + x.month)

    grid_monthly = grid_monthly.sort_values(['LatGrid','LonGrid','YM_int'])
    grid_monthly['Lag1'] = grid_monthly.groupby(['LatGrid','LonGrid'])['Count'].shift(1)
    grid_monthly['Lag2'] = grid_monthly.groupby(['LatGrid','LonGrid'])['Count'].shift(2)
    grid_monthly['Lag3'] = grid_monthly.groupby(['LatGrid','LonGrid'])['Count'].shift(3)
    grid_monthly['Rolling3'] = grid_monthly.groupby(['LatGrid','LonGrid'])['Count'].transform(
        lambda x: x.shift(1).rolling(3).mean())
    grid_monthly['Trend'] = grid_monthly['Lag1'] - grid_monthly['Lag2']
    grid_monthly['Month'] = grid_monthly['YM_int'] % 12
    grid_monthly['Month_sin'] = np.sin(2 * np.pi * grid_monthly['Month'] / 12)
    grid_monthly['Month_cos'] = np.cos(2 * np.pi * grid_monthly['Month'] / 12)

    # Label: top-25% count as hotspot
    q75 = grid_monthly['Count'].quantile(0.75)
    grid_monthly['IsHotspot'] = (grid_monthly['Count'] >= q75).astype(int)

    feat_cols = ['LatGrid','LonGrid','Lag1','Lag2','Lag3','Rolling3','Trend',
                  'Month_sin','Month_cos']
    data = grid_monthly[feat_cols + ['IsHotspot']].dropna()

    X = data[feat_cols]
    y = data['IsHotspot']

    scaler = StandardScaler()
    X_s = scaler.fit_transform(X)

    X_tr, X_te, y_tr, y_te = train_test_split(X_s, y, test_size=0.2,
                                                random_state=RANDOM_SEED, stratify=y)

    models = {
        'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10,
                                                 random_state=RANDOM_SEED, n_jobs=-1),
        'Gradient Boosting': HistGradientBoostingClassifier(max_iter=100, max_depth=5,
                                                             random_state=RANDOM_SEED),
        'Logistic Regression': LogisticRegression(max_iter=300, random_state=RANDOM_SEED)
    }

    results = []
    best_name = None; best_f1 = -1; best_model = None; best_pred = None
    for name, m in models.items():
        m.fit(X_tr, y_tr)
        y_pred = m.predict(X_te)
        acc  = accuracy_score(y_te, y_pred)
        f1m  = f1_score(y_te, y_pred, average='macro', zero_division=0)
        f1w  = f1_score(y_te, y_pred, average='weighted', zero_division=0)
        results.append({'Model': name, 'Accuracy': round(acc,4),
                         'F1 Macro': round(f1m,4), 'F1 Weighted': round(f1w,4)})
        print(f"  {name}: Acc={acc:.4f}, F1={f1m:.4f}")
        if f1m > best_f1:
            best_f1 = f1m; best_name = name; best_model = m; best_pred = y_pred

    results_df = pd.DataFrame(results)
    results_df.to_csv(f"{OUTPUT_DIR}/tables/J5_hotspot_prediction.csv", index=False)

    # Feature importance for best model
    if hasattr(best_model, 'feature_importances_'):
        imp = pd.Series(best_model.feature_importances_, index=feat_cols).sort_values(ascending=False)
    else:
        imp = None

    # --- FIGURE J5: 3-panel ---
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle('J5: Predictive Hotspot Model — Next Month Hotspot Prediction',
                 fontsize=14, fontweight='bold')

    # Panel 1: Model comparison
    x = np.arange(len(results_df))
    w = 0.25
    for i, m in enumerate(['Accuracy','F1 Macro','F1 Weighted']):
        axes[0].bar(x + i*w, results_df[m], w, label=m, color=COLORS[i])
    axes[0].set_xticks(x + w)
    axes[0].set_xticklabels(results_df['Model'], rotation=15, ha='right', fontsize=8)
    axes[0].set_title('Model Performance Comparison', fontsize=10, fontweight='bold')
    axes[0].set_ylabel('Score'); axes[0].legend(fontsize=8); axes[0].set_ylim(0,1.1)

    # Panel 2: Confusion matrix
    cm = confusion_matrix(y_te, best_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
                xticklabels=['Not Hotspot','Hotspot'],
                yticklabels=['Not Hotspot','Hotspot'])
    axes[1].set_title(f'Confusion Matrix\n({best_name})', fontsize=10, fontweight='bold')
    axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')

    # Panel 3: Feature importance
    if imp is not None:
        imp.plot(kind='barh', ax=axes[2], color=COLORS[0])
        axes[2].set_title('Feature Importances', fontsize=10, fontweight='bold')
        axes[2].set_xlabel('Importance')

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/joint/fig_J5_hotspot_prediction.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("  [Fig J5] Predictive hotspot model saved.")
    return results_df

# ============================================================

# ============================================================
# J_X1. XGBOOST SPATIO-TEMPORAL MODEL
# ============================================================
def xgboost_spatiotemporal(df):
    """XGBoost for spatio-temporal duration regression and hotspot prediction."""
    print("\n[J_X1] XGBoost Spatio-Temporal Model...")
    try:
        from xgboost import XGBRegressor, XGBClassifier
    except ImportError:
        try:
            import subprocess, sys
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'xgboost', '-q'])
            from xgboost import XGBRegressor, XGBClassifier
        except Exception as e:
            print(f"     XGBoost not available: {e}. Skipping.")
            return None

    from sklearn.model_selection import train_test_split
    from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, f1_score
    from sklearn.preprocessing import LabelEncoder

    # Feature columns (spatial + temporal)
    feature_cols = [c for c in [
        'Start_Lat', 'Start_Lng', 'Hour', 'DayOfWeek', 'Month', 'Year',
        'IsWeekend', 'IsRushHour', 'IsNight', 'Season',
        'Hour_sin', 'Hour_cos', 'Month_sin', 'Month_cos',
        'DOW_sin', 'DOW_cos', 'DistFromCenter',
        'GridCellCount', 'HexCellCount', 'POI_Count',
        'StateMonthCount'
    ] if c in df.columns]

    sample = df.dropna(subset=feature_cols + ['Duration_hrs', 'Log_Duration']).copy()
    if len(sample) > 50000:
        sample = sample.sample(50000, random_state=42)

    # Encode categorical Season
    if 'Season' in sample.columns:
        le = LabelEncoder()
        sample['Season'] = le.fit_transform(sample['Season'].astype(str))

    X = sample[feature_cols].values
    y_reg = sample['Log_Duration'].values

    X_tr, X_te, y_tr, y_te = train_test_split(X, y_reg, test_size=0.2, random_state=42)

    # --- Regression ---
    xgb_reg = XGBRegressor(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbosity=0
    )
    xgb_reg.fit(X_tr, y_tr)
    pred_reg = xgb_reg.predict(X_te)

    rmse = np.sqrt(mean_squared_error(y_te, pred_reg))
    r2   = r2_score(y_te, pred_reg)
    mape = (np.abs((y_te - pred_reg) / (np.abs(y_te) + 1e-8)).mean()) * 100
    print(f"     XGBoost Regression: RMSE={rmse:.4f}, MAPE={mape:.2f}%, R²={r2:.4f}")

    # --- Classification: Hotspot (top quartile of events per grid cell) ---
    clf_target_col = None
    if 'GridCellCount' in sample.columns:
        threshold = sample['GridCellCount'].quantile(0.75)
        sample['IsHotspot'] = (sample['GridCellCount'] >= threshold).astype(int)
        clf_target_col = 'IsHotspot'

    clf_results = {}
    if clf_target_col and sample[clf_target_col].nunique() > 1:
        y_clf = sample[clf_target_col].values
        Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(X, y_clf, test_size=0.2,
                                                         random_state=42, stratify=y_clf)
        xgb_clf = XGBClassifier(
            n_estimators=200,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            use_label_encoder=False,
            eval_metric='logloss',
            verbosity=0
        )
        xgb_clf.fit(Xc_tr, yc_tr)
        pred_clf = xgb_clf.predict(Xc_te)
        acc  = accuracy_score(yc_te, pred_clf)
        f1   = f1_score(yc_te, pred_clf, average='weighted')
        clf_results = {'Accuracy': round(acc, 4), 'F1': round(f1, 4)}
        print(f"     XGBoost Classification (Hotspot): Accuracy={acc:.4f}, F1={f1:.4f}")

    metrics_df = pd.DataFrame([{
        'Model': 'XGBoost (spatio-temporal)',
        'Task': 'Regression (Log Duration)',
        'RMSE': round(rmse, 4),
        'MAPE(%)': round(mape, 2),
        'R2': round(r2, 4),
        'Clf_Accuracy': clf_results.get('Accuracy', 'N/A'),
        'Clf_F1': clf_results.get('F1', 'N/A')
    }])
    metrics_df.to_csv(f"{OUTPUT_DIR}/tables/xgboost_spatiotemporal.csv", index=False)

    # Feature importance
    importances = xgb_reg.feature_importances_
    feat_imp = pd.Series(importances, index=feature_cols).sort_values(ascending=False).head(15)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('XGBoost Spatio-Temporal Model', fontsize=14, fontweight='bold')

    axes[0].barh(feat_imp.index[::-1], feat_imp.values[::-1], color=COLORS[5])
    axes[0].set_title('XGBoost Feature Importance (Top 15)')
    axes[0].set_xlabel('Importance Score')

    axes[1].scatter(y_te[:500], pred_reg[:500], alpha=0.4, color=COLORS[0], edgecolors='white', s=30)
    mn = min(y_te.min(), pred_reg.min())
    mx = max(y_te.max(), pred_reg.max())
    axes[1].plot([mn, mx], [mn, mx], 'r--', linewidth=2, label='Perfect Fit')
    axes[1].set_title(f'XGBoost Predicted vs Actual | R²={r2:.4f}')
    axes[1].set_xlabel('Actual Log Duration')
    axes[1].set_ylabel('Predicted Log Duration')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/joint/fig_JX1_xgboost_spatiotemporal.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("     [Fig JX1] XGBoost spatio-temporal saved.")
    return metrics_df


# ============================================================
# J_X2. LSTM SPATIO-TEMPORAL MODEL
# ============================================================
def lstm_spatiotemporal(df):
    """LSTM for spatio-temporal sequence prediction using spatial + temporal features."""
    print("\n[J_X2] LSTM Spatio-Temporal Model...")
    try:
        import tensorflow as tf
        from tensorflow.keras.models import Sequential
        from tensorflow.keras.layers import LSTM, Dense, Dropout
        from tensorflow.keras.callbacks import EarlyStopping
    except ImportError:
        try:
            import subprocess, sys
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'tensorflow', '-q'])
            import tensorflow as tf
            from tensorflow.keras.models import Sequential
            from tensorflow.keras.layers import LSTM, Dense, Dropout
            from tensorflow.keras.callbacks import EarlyStopping
        except Exception as e:
            print(f"     TensorFlow not available: {e}. Skipping ST-LSTM.")
            return None

    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import MinMaxScaler as MMS
    from sklearn.metrics import mean_squared_error, r2_score

    # Aggregate: monthly event count per top-N grid cells (spatio-temporal sequences)
    feature_cols = [c for c in [
        'Hour_sin', 'Hour_cos', 'Month_sin', 'Month_cos',
        'DOW_sin', 'DOW_cos', 'IsWeekend', 'IsRushHour',
        'DistFromCenter', 'GridCellCount', 'POI_Count'
    ] if c in df.columns]

    target_col = 'Log_Duration'
    if target_col not in df.columns:
        print("     Log_Duration not found. Skipping.")
        return None

    sample = df.dropna(subset=feature_cols + [target_col]).copy()
    if len(sample) > 40000:
        sample = sample.sample(40000, random_state=42)
    if len(sample) < 50:
        print("     Not enough data for ST-LSTM.")
        return None

    X_raw = sample[feature_cols].values
    y_raw = sample[target_col].values

    scaler_X = MMS()
    scaler_y = MMS()
    X_scaled = scaler_X.fit_transform(X_raw)
    y_scaled = scaler_y.fit_transform(y_raw.reshape(-1, 1)).flatten()

    # Build sequences (LOOK_BACK time steps)
    LOOK_BACK = 8
    X_seq, y_seq = [], []
    for i in range(LOOK_BACK, len(X_scaled)):
        X_seq.append(X_scaled[i - LOOK_BACK:i])
        y_seq.append(y_scaled[i])
    X_seq = np.array(X_seq)   # (samples, look_back, features)
    y_seq = np.array(y_seq)

    if len(X_seq) < 20:
        print("     Not enough sequences for ST-LSTM.")
        return None

    split = int(len(X_seq) * 0.8)
    X_tr, X_te = X_seq[:split], X_seq[split:]
    y_tr, y_te = y_seq[:split], y_seq[split:]

    tf.random.set_seed(RANDOM_SEED)
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(LOOK_BACK, len(feature_cols))),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')

    es = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)
    history = model.fit(
        X_tr, y_tr,
        epochs=80,
        batch_size=64,
        validation_split=0.15,
        callbacks=[es],
        verbose=0
    )

    pred_scaled = model.predict(X_te, verbose=0).flatten()
    pred = scaler_y.inverse_transform(pred_scaled.reshape(-1, 1)).flatten()
    actual = scaler_y.inverse_transform(y_te.reshape(-1, 1)).flatten()

    rmse = np.sqrt(mean_squared_error(actual, pred))
    r2   = r2_score(actual, pred)
    mape = (np.abs((actual - pred) / (np.abs(actual) + 1e-8)).mean()) * 100

    print(f"     LSTM (spatio-temporal): RMSE={rmse:.4f}, MAPE={mape:.2f}%, R²={r2:.4f}")

    lstm_st_metrics = pd.DataFrame([{
        'Model': 'LSTM (spatio-temporal, look_back=8)',
        'RMSE': round(rmse, 4),
        'MAPE(%)': round(mape, 2),
        'R2': round(r2, 4)
    }])
    lstm_st_metrics.to_csv(f"{OUTPUT_DIR}/tables/lstm_spatiotemporal_metrics.csv", index=False)

    # Figure
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('LSTM Spatio-Temporal Model — Duration Prediction',
                 fontsize=14, fontweight='bold')

    axes[0].scatter(actual[:300], pred[:300], alpha=0.4,
                    color=COLORS[4], edgecolors='white', s=30)
    mn = min(actual.min(), pred.min())
    mx = max(actual.max(), pred.max())
    axes[0].plot([mn, mx], [mn, mx], 'r--', linewidth=2, label='Perfect Fit')
    axes[0].set_title(f'ST-LSTM Predicted vs Actual | R²={r2:.4f}')
    axes[0].set_xlabel('Actual Log Duration'); axes[0].set_ylabel('Predicted Log Duration')
    axes[0].legend()

    axes[1].plot(history.history['loss'], label='Train Loss', color=COLORS[0])
    if 'val_loss' in history.history:
        axes[1].plot(history.history['val_loss'], label='Val Loss', color=COLORS[1])
    axes[1].set_title('ST-LSTM Training Loss')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MSE Loss')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/joint/fig_JX2_lstm_spatiotemporal.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("     [Fig JX2] LSTM spatio-temporal saved.")
    return lstm_st_metrics


# J6. COMPREHENSIVE ANALYSIS SUMMARY
# ============================================================
def j6_comprehensive_summary(df):
    print("\n" + "="*60)
    print("[J6] COMPREHENSIVE SUMMARY FIGURES & TABLES")
    print("="*60)

    # ---- Master model comparison figure ----
    # Collect all results CSVs
    model_results = {}
    csv_map = {
        'ARIMA Forecast': f"{OUTPUT_DIR}/tables/arima_metrics.csv",
        'ETS Models': f"{OUTPUT_DIR}/tables/ets_model_comparison.csv",
        'Temporal Clf': f"{OUTPUT_DIR}/tables/temporal_classification_results.csv",
        'Spatial Clf': f"{OUTPUT_DIR}/tables/spatial_classification_results.csv",
        'Joint Clf (All)': f"{OUTPUT_DIR}/tables/J2_classification_comparison.csv",
        'Hotspot Pred': f"{OUTPUT_DIR}/tables/J5_hotspot_prediction.csv",
    }
    for label, path in csv_map.items():
        if os.path.exists(path):
            model_results[label] = pd.read_csv(path)

    # ---- Dataset characteristics summary figure ----
    fig = plt.figure(figsize=(20, 14))
    fig.suptitle('Comprehensive Analysis Summary: US Road Construction Dataset (2016–2021)',
                 fontsize=15, fontweight='bold')
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

    # Panel 1: Events per year
    ax1 = fig.add_subplot(gs[0, 0])
    year_counts = df['Year'].value_counts().sort_index()
    ax1.bar(year_counts.index.astype(str), year_counts.values, color=COLORS[0], edgecolor='white')
    ax1.set_title('Events per Year', fontsize=11, fontweight='bold')
    ax1.set_ylabel('Count')
    for i, (yr, v) in enumerate(year_counts.items()):
        ax1.text(i, v + 50, f'{v:,}', ha='center', fontsize=8)

    # Panel 2: Duration histogram
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.hist(df['Duration_hrs'].clip(0, 72), bins=60, color=COLORS[1],
              edgecolor='white', alpha=0.85)
    ax2.axvline(df['Duration_hrs'].median(), color='red', ls='--',
                 label=f"Median={df['Duration_hrs'].median():.1f}h")
    ax2.axvline(df['Duration_hrs'].mean(), color='orange', ls='--',
                 label=f"Mean={df['Duration_hrs'].mean():.1f}h")
    ax2.set_title('Event Duration Distribution (capped 72h)', fontsize=11, fontweight='bold')
    ax2.set_xlabel('Duration (hrs)'); ax2.set_ylabel('Count'); ax2.legend(fontsize=8)

    # Panel 3: Closure type
    ax3 = fig.add_subplot(gs[0, 2])
    if 'Type' in df.columns:
        tc = df['Type'].value_counts()
        colors_pie = COLORS[:len(tc)]
        wedges, texts, autotexts = ax3.pie(tc.values, labels=tc.index,
                                             autopct='%1.1f%%', colors=colors_pie,
                                             startangle=90, pctdistance=0.8)
        ax3.set_title('Closure Type Distribution', fontsize=11, fontweight='bold')
    else:
        ax3.text(0.5,0.5,'No Type column',ha='center',va='center',transform=ax3.transAxes)

    # Panel 4: Top states
    ax4 = fig.add_subplot(gs[1, 0])
    if 'State' in df.columns:
        sc = df['State'].value_counts().head(12)
        ax4.barh(sc.index[::-1], sc.values[::-1], color=COLORS[2])
        ax4.set_title('Top 12 States by Event Count', fontsize=11, fontweight='bold')
        ax4.set_xlabel('Event Count')
        for i, v in enumerate(sc.values[::-1]):
            ax4.text(v + 50, i, f'{v:,}', va='center', fontsize=7)

    # Panel 5: Season distribution
    ax5 = fig.add_subplot(gs[1, 1])
    if 'Season' in df.columns:
        season_order = ['Winter','Spring','Summer','Fall']
        sc2 = df['Season'].value_counts().reindex(season_order)
        season_c = ['#AED6F1','#A9DFBF','#FAD7A0','#F1948A']
        ax5.bar(sc2.index, sc2.values, color=season_c, edgecolor='white')
        ax5.set_title('Events by Season', fontsize=11, fontweight='bold')
        ax5.set_ylabel('Count')
        for i, v in enumerate(sc2.values):
            ax5.text(i, v + 50, f'{v:,}', ha='center', fontsize=9)

    # Panel 6: Mining summary
    ax6 = fig.add_subplot(gs[1, 2])
    mining_summary = {
        'Spatial Colocation\n(PI pairs)': 'M1',
        'Temporal Motifs\n(monthly)': 'M2',
        'Association Rules\n(Apriori)': 'M3',
        'Anomaly Events\n(IF+LOF)': 'M4',
        'Temporal Anomalies\n(statistical)': 'M5',
        'State Co-activation\n(spatial seq)': 'M6',
        'Hotspot Cells\n(Getis-Ord G*)': 'M7',
        'Cluster Profiles\n(regional)': 'M8',
    }
    y_pos = range(len(mining_summary))
    ax6.barh(list(mining_summary.values())[::-1],
              [1]*len(mining_summary), color=COLORS[:len(mining_summary)], alpha=0.8)
    ax6.set_yticks(list(y_pos))
    ax6.set_yticklabels(list(mining_summary.keys())[::-1], fontsize=8)
    ax6.set_title('Mining Techniques Applied', fontsize=11, fontweight='bold')
    ax6.set_xticks([]); ax6.set_xlabel('')

    plt.savefig(f"{OUTPUT_DIR}/joint/fig_J6_comprehensive_summary.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("  [Fig J6] Comprehensive summary saved.")

    # ---- Key findings table ----
    key_findings = [
        ['M1 Colocation', 'TrafficSignal ↔ Junction', 'Strongest co-location PI', 'Spatial Planning'],
        ['M2 Patterns', 'HML, LMH recurring', 'Consistent seasonal cycling', 'Forecasting'],
        ['M3 Rules', 'Rush+Junction→Signal', 'Infrastructure clustering', 'Risk Analysis'],
        ['M4 Anomalies', '~5% events', 'Extreme duration outliers', 'Quality Control'],
        ['M5 TS Anomaly', 'COVID-19 dip', 'Identified via Z+IQR consensus', 'Event Detection'],
        ['M6 Sequences', 'CA–TX co-active', 'Adjacent states co-occur', 'Network Analysis'],
        ['M7 Hotspots', 'Urban corridors', 'G* significant at 99%', 'Policy Target'],
        ['M8 Profiling', '6 US regions', 'West: longer, East: more events', 'Resource Allocation'],
        ['J1 ST-Hotspot', 'Summer peaks', 'All regions, esp. Midwest', 'Scheduling'],
        ['J2 Joint ML', 'Joint > Spatial only', 'Weather adds <2% but consistent', 'Model Design'],
        ['J3 Interaction', 'Urban rush amplified', '2× longer events in rush+urban', 'Traffic Impact'],
        ['J4 3D DBSCAN', 'ST clusters found', 'Seasonal-geographic groupings', 'Pattern Mining'],
        ['J5 Prediction', 'RF best hotspot model', 'Lag features most important', 'Forecasting'],
    ]
    kf_df = pd.DataFrame(key_findings, columns=['Technique','Key Finding','Interpretation','Application'])
    kf_df.to_csv(f"{OUTPUT_DIR}/tables/J6_key_findings.csv", index=False)
    print("\n  Key Findings Table saved.")
    print(kf_df.to_string(index=False))

# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":
    df = load_processed()

    # ---- MINING ----
    coloc_df, pi_matrix    = m1_spatial_colocation(df)
    top_m_df, top_w_df, trans_df = m2_temporal_patterns(df)
    rules_df               = m3_association_rules(df)
    anom_stats_df          = m4_spatial_anomaly(df)
    temporal_anom_df       = m5_temporal_anomaly(df)
    coact_df, state_trans  = m6_spatial_sequences(df)
    hotspot_df             = m7_hotspot_analysis(df)
    cluster_profile_df     = m8_cluster_profiling(df)

    # ---- JOINT INFERENCE ----
    pivot_hs    = j1_spatiotemporal_hotspot(df)
    cls_df, reg_df = j2_joint_ml_model(df)
    j3_spatial_temporal_interaction(df)
    cl_prof_df  = j4_spatiotemporal_dbscan(df)
    hs_pred_df  = j5_hotspot_prediction(df)
    xgboost_spatiotemporal(df)
    lstm_spatiotemporal(df)
    j6_comprehensive_summary(df)

    print("\n" + "="*70)
    print("PART 4 (EXPANDED) COMPLETE")
    print("="*70)

    # Count outputs
    import os
    img = sum(1 for _, _, fs in os.walk(OUTPUT_DIR) for f in fs if f.endswith('.png'))
    csv = sum(1 for _, _, fs in os.walk(OUTPUT_DIR) for f in fs if f.endswith('.csv'))
    print(f"\n  Total figures: {img}")
    print(f"  Total tables:  {csv}")
    print(f"\nOutputs in: {os.path.abspath(OUTPUT_DIR)}/")
    print("\nRun generate_latex_tables.py to get ready LaTeX tables.")


PART 4 (EXPANDED): DATA MINING & JOINT INFERENCE
[Load] Processed data shape: (1644242, 85)

[M1] SPATIAL COLOCATION MINING (Participation Index)
  POI types analyzed: ['Junction', 'Traffic_Signal', 'Crossing', 'Roundabout', 'Station', 'Railway', 'Stop', 'Amenity', 'Bump', 'Give_Way', 'No_Exit', 'Traffic_Calming', 'Turning_Loop']
  Distance threshold: 0.05° (~5.6 km)

  Top 10 Colocation Rules:
     Feature_A       Feature_B  PI(A,B)  PR(A→B)  PR(B→A)  Normalized_PI
       Station         Amenity   0.8167   0.8167   0.8724        11.0069
Traffic_Signal        Crossing   0.7585   0.7585   0.9296         4.1312
       Railway         Amenity   0.6954   0.7695   0.6954        20.3736
       Station         Railway   0.6483   0.6483   0.7852        18.9936
      Crossing         Station   0.6420   0.6420   0.9378         6.2368
      Crossing         Amenity   0.6144   0.6144   0.9299         8.2800
          Bump Traffic_Calming   0.6000   1.0000   0.6000       500.0000
       Station    

---
## LaTeX Table Generation
Converts all CSV result files in `outputs/tables/` to ready-to-paste LaTeX table code for the IEEE 2-column report.

In [12]:
"""
AID843 A3 - LaTeX Table Generator
Converts CSV result files to LaTeX table code for the IEEE 2-column report.
"""

import pandas as pd
import os

OUTPUT_DIR = "outputs"
TEX_OUT = f"{OUTPUT_DIR}/latex_tables.tex"

def df_to_latex_table(df, caption, label, small=True):
    """Convert DataFrame to LaTeX table string"""
    n_cols = len(df.columns)
    col_fmt = 'l' + 'r' * (n_cols - 1)
    size_cmd = r'\small' if small else ''
    lines = []
    lines.append(r'\begin{table}[htbp]')
    lines.append(r'\centering')
    if size_cmd:
        lines.append(size_cmd)
    lines.append(f'\\caption{{{caption}}}')
    lines.append(f'\\label{{tab:{label}}}')
    lines.append(f'\\begin{{tabular}}{{{col_fmt}}}')
    lines.append(r'\hline')
    header = ' & '.join([f'\\textbf{{{col}}}' for col in df.columns])
    lines.append(header + r' \\')
    lines.append(r'\hline')
    for _, row in df.iterrows():
        row_str = ' & '.join([str(v) for v in row.values])
        lines.append(row_str + r' \\')
    lines.append(r'\hline')
    lines.append(r'\end{tabular}')
    lines.append(r'\end{table}')
    return '\n'.join(lines)

# ── ALL TABLES (EDA + Temporal + Spatial + Mining + Joint + New Models) ───────
tables_config = [
    # ── EDA ──────────────────────────────────────────────────────────────────
    {
        'file': f"{OUTPUT_DIR}/tables/dataset_overview.csv",
        'caption': 'Dataset Overview: US Road Construction and Closures (2016--2021)',
        'label': 'dataset_overview'
    },
    {
        'file': f"{OUTPUT_DIR}/tables/summary_statistics.csv",
        'caption': 'Descriptive Statistics of Key Numerical Features',
        'label': 'summary_stats'
    },
    {
        'file': f"{OUTPUT_DIR}/tables/engineered_features.csv",
        'caption': 'Engineered Features for Spatio-Temporal Analysis',
        'label': 'features'
    },
    # ── TEMPORAL ─────────────────────────────────────────────────────────────
    {
        'file': f"{OUTPUT_DIR}/tables/adf_test_results.csv",
        'caption': 'Augmented Dickey-Fuller (ADF) Stationarity Test Results',
        'label': 'adf_results'
    },
    {
        'file': f"{OUTPUT_DIR}/tables/arima_model_selection.csv",
        'caption': 'ARIMA Model Order Selection (Grid Search)',
        'label': 'arima_selection',
        'head': 5
    },
    {
        'file': f"{OUTPUT_DIR}/tables/arima_metrics.csv",
        'caption': 'ARIMA Forecasting Model Performance Metrics',
        'label': 'arima_metrics'
    },
    {
        'file': f"{OUTPUT_DIR}/tables/ets_model_comparison.csv",
        'caption': 'Exponential Smoothing (ETS) Model Comparison',
        'label': 'ets_comparison'
    },
    {
        'file': f"{OUTPUT_DIR}/tables/sarima_metrics.csv",
        'caption': 'SARIMA(1,1,1)(1,1,0,12) Forecast Performance Metrics',
        'label': 'sarima'
    },
    {
        'file': f"{OUTPUT_DIR}/tables/prophet_metrics.csv",
        'caption': 'Prophet (Multiplicative Seasonality) Forecast Performance',
        'label': 'prophet'
    },
    {
        'file': f"{OUTPUT_DIR}/tables/lstm_temporal_metrics.csv",
        'caption': 'LSTM Temporal Model Forecast Performance Metrics',
        'label': 'lstm_temporal'
    },
    {
        'file': f"{OUTPUT_DIR}/tables/temporal_classification_results.csv",
        'caption': 'Temporal Classification Models: Performance Comparison',
        'label': 'temporal_clf'
    },
    {
        'file': f"{OUTPUT_DIR}/tables/temporal_regression_results.csv",
        'caption': 'Temporal Regression Models: Duration Prediction',
        'label': 'temporal_reg'
    },
    # ── SPATIAL ──────────────────────────────────────────────────────────────
    {
        'file': f"{OUTPUT_DIR}/tables/morans_i_results.csv",
        'caption': "Moran's I Global Spatial Autocorrelation Test Results",
        'label': 'morans_i'
    },
    {
        'file': f"{OUTPUT_DIR}/tables/gearys_c_results.csv",
        'caption': "Geary's C Spatial Autocorrelation Results",
        'label': 'gearys_c'
    },
    {
        'file': f"{OUTPUT_DIR}/tables/lisa_results.csv",
        'caption': 'LISA Local Indicators of Spatial Association — Cluster Summary',
        'label': 'lisa',
        'head': 20
    },
    {
        'file': f"{OUTPUT_DIR}/tables/clustering_quality.csv",
        'caption': 'Spatial Clustering Quality Metrics (K-Means vs DBSCAN)',
        'label': 'clustering'
    },
    {
        'file': f"{OUTPUT_DIR}/tables/spatial_classification_results.csv",
        'caption': 'Spatial Classification Models: Performance Comparison',
        'label': 'spatial_clf'
    },
    {
        'file': f"{OUTPUT_DIR}/tables/gwr_results.csv",
        'caption': 'Geographically Weighted Regression (GWR) vs OLS Results',
        'label': 'gwr'
    },
    {
        'file': f"{OUTPUT_DIR}/tables/state_level_stats.csv",
        'caption': 'State-Level Construction Event Statistics (Top 15)',
        'label': 'state_stats',
        'head': 15
    },
    # ── MINING ───────────────────────────────────────────────────────────────
    {
        'file': f"{OUTPUT_DIR}/tables/M1_colocation_PI.csv",
        'caption': 'M1: Spatial Colocation Mining — Participation Index Rules (Top 15)',
        'label': 'M1_colocation',
        'head': 15
    },
    {
        'file': f"{OUTPUT_DIR}/tables/M2_monthly_motifs.csv",
        'caption': 'M2: Top Monthly Temporal Motifs (L/M/H Discretized Sequence)',
        'label': 'M2_motifs'
    },
    {
        'file': f"{OUTPUT_DIR}/tables/M2_weekly_motifs.csv",
        'caption': 'M2: Top Weekly Temporal Motifs (L/M/H Discretized Sequence)',
        'label': 'M2_weekly_motifs'
    },
    {
        'file': f"{OUTPUT_DIR}/tables/M2_season_transition.csv",
        'caption': 'M2: Season-to-Season Transition Probability Matrix',
        'label': 'M2_season_trans'
    },
    {
        'file': f"{OUTPUT_DIR}/tables/M3_association_rules.csv",
        'caption': 'M3: Top Association Rules by Lift (Apriori Mining)',
        'label': 'M3_rules',
        'head': 10
    },
    {
        'file': f"{OUTPUT_DIR}/tables/M4_anomaly_comparison.csv",
        'caption': 'M4: Anomaly Group Statistics — Isolation Forest vs LOF',
        'label': 'M4_anomaly'
    },
    {
        'file': f"{OUTPUT_DIR}/tables/M5_temporal_anomalies.csv",
        'caption': 'M5: Monthly Temporal Anomaly Flags (Z-score, IQR, Rolling)',
        'label': 'M5_temporal_anom',
        'head': 20
    },
    {
        'file': f"{OUTPUT_DIR}/tables/M6_state_coactivation.csv",
        'caption': 'M6: Top State-Pair Co-activation by Lift',
        'label': 'M6_coactivation',
        'head': 15
    },
    {
        'file': f"{OUTPUT_DIR}/tables/M6_state_transition.csv",
        'caption': 'M6: State-to-State Spatial Transition Probability Matrix',
        'label': 'M6_state_trans',
        'head': 20
    },
    {
        'file': f"{OUTPUT_DIR}/tables/M7_hotspots.csv",
        'caption': 'M7: Getis-Ord G* Hotspot Classification by Grid Cell (Top 20)',
        'label': 'M7_hotspots',
        'head': 20
    },
    {
        'file': f"{OUTPUT_DIR}/tables/M8_cluster_profiles.csv",
        'caption': 'M8: Regional Cluster Profiles — Avg Feature Values',
        'label': 'M8_profiles'
    },
    # ── JOINT / SPATIO-TEMPORAL ───────────────────────────────────────────────
    {
        'file': f"{OUTPUT_DIR}/tables/J1_hotspot_matrix.csv",
        'caption': 'J1: Spatio-Temporal Hotspot Matrix (Grid Cell × Month)',
        'label': 'J1_hotspot',
        'head': 15
    },
    {
        'file': f"{OUTPUT_DIR}/tables/J2_classification_comparison.csv",
        'caption': 'J2: Classification Comparison — Spatial vs Temporal vs Joint Features',
        'label': 'J2_cls'
    },
    {
        'file': f"{OUTPUT_DIR}/tables/J2_regression_comparison.csv",
        'caption': 'J2: Regression Comparison — Spatial vs Temporal vs Joint Features',
        'label': 'J2_reg'
    },
    {
        'file': f"{OUTPUT_DIR}/tables/xgboost_spatiotemporal.csv",
        'caption': 'XGBoost Spatio-Temporal Model Performance',
        'label': 'xgboost_st'
    },
    {
        'file': f"{OUTPUT_DIR}/tables/lstm_spatiotemporal_metrics.csv",
        'caption': 'LSTM Spatio-Temporal Model Performance',
        'label': 'lstm_st'
    },
    {
        'file': f"{OUTPUT_DIR}/tables/J4_3d_cluster_profiles.csv",
        'caption': 'J4: 3D Spatiotemporal DBSCAN Cluster Profiles',
        'label': 'J4_3d_clusters'
    },
    {
        'file': f"{OUTPUT_DIR}/tables/J5_hotspot_prediction.csv",
        'caption': 'J5: Predictive Hotspot Model Performance Comparison',
        'label': 'J5_hotspot_pred'
    },
    {
        'file': f"{OUTPUT_DIR}/tables/J6_key_findings.csv",
        'caption': 'J6: Key Findings Summary — All Mining and Inference Techniques',
        'label': 'J6_findings'
    },
]

def generate_all_tables():
    print("Generating LaTeX tables...")
    all_tex = []
    all_tex.append("% =========================================================")
    all_tex.append("% AUTO-GENERATED LaTeX TABLES — AID843 A3")
    all_tex.append("% Paste individual tables into your IEEE two-column report")
    all_tex.append("% =========================================================\n")

    count = 0
    for cfg in tables_config:
        if not os.path.exists(cfg['file']):
            print(f"  SKIP (not found): {cfg['file']}")
            continue
        try:
            df = pd.read_csv(cfg['file'])
            # Apply row limit if specified
            if 'head' in cfg:
                df = df.head(cfg['head'])
            # Round floats
            for col in df.select_dtypes(include='float').columns:
                df[col] = df[col].round(4)
            tex = df_to_latex_table(df, cfg['caption'], cfg['label'])
            all_tex.append(f"% --- Table: {cfg['label']} ---")
            all_tex.append(tex)
            all_tex.append("")
            count += 1
            print(f"  OK: {cfg['label']} ({len(df)} rows × {len(df.columns)} cols)")
        except Exception as e:
            print(f"  ERROR {cfg['label']}: {e}")

    output_tex = '\n'.join(all_tex)
    with open(TEX_OUT, 'w') as f:
        f.write(output_tex)

    print(f"\n{count} LaTeX tables generated → {TEX_OUT}")
    print("\nUsage in your .tex file:")
    print(r"  \input{latex_tables.tex}")
    print("  Or copy individual tables directly from the file.")

generate_all_tables()

Generating LaTeX tables...
  OK: dataset_overview (6 rows × 2 cols)
  OK: summary_stats (8 rows × 7 cols)
  OK: features (19 rows × 3 cols)
  OK: adf_results (3 rows × 9 cols)
  OK: arima_selection (5 rows × 3 cols)
  OK: arima_metrics (1 rows × 4 cols)
  OK: ets_comparison (4 rows × 4 cols)
  OK: sarima (1 rows × 4 cols)
  OK: prophet (1 rows × 4 cols)
  OK: lstm_temporal (1 rows × 4 cols)
  OK: temporal_clf (4 rows × 6 cols)
  OK: temporal_reg (2 rows × 3 cols)
  OK: morans_i (2 rows × 4 cols)
  OK: gearys_c (2 rows × 4 cols)
  OK: lisa (20 rows × 7 cols)
  OK: clustering (2 rows × 4 cols)
  OK: spatial_clf (3 rows × 6 cols)
  OK: gwr (2 rows × 3 cols)
  OK: state_stats (15 rows × 4 cols)
  OK: M1_colocation (15 rows × 8 cols)
  OK: M2_motifs (20 rows × 4 cols)
  OK: M2_weekly_motifs (20 rows × 4 cols)
  OK: M2_season_trans (4 rows × 5 cols)
  OK: M3_rules (8 rows × 8 cols)
  OK: M4_anomaly (4 rows × 6 cols)
  OK: M5_temporal_anom (20 rows × 10 cols)
  OK: M6_coactivation (15 rows × 

---
## Results Summary
List every output file produced by the analysis.

In [13]:
import os

print("=== ALL GENERATED OUTPUT FILES ===\n")
for root, dirs, files in sorted(os.walk('outputs')):
    dirs.sort()
    for f in sorted(files):
        path = os.path.join(root, f)
        size_kb = os.path.getsize(path) / 1024
        print(f"  {path:<65s} {size_kb:>8.1f} KB")

png_count = sum(1 for r,d,fs in os.walk('outputs') for f in fs if f.endswith('.png'))
csv_count = sum(1 for r,d,fs in os.walk('outputs') for f in fs if f.endswith('.csv'))
tex_exists = os.path.exists('outputs/latex_tables.tex')
print(f"\n{'─'*65}")
print(f"  Figures (PNG) : {png_count}")
print(f"  Tables  (CSV) : {csv_count}")
print(f"  LaTeX tables  : {'outputs/latex_tables.tex  ✓' if tex_exists else 'NOT FOUND'}")


=== ALL GENERATED OUTPUT FILES ===

  outputs/latex_tables.tex                                              29.0 KB
  outputs/processed_data.parquet                                    274248.6 KB
  outputs/eda/fig1_missing_values.png                                  112.6 KB
  outputs/eda/fig2_temporal_distributions.png                          181.3 KB
  outputs/eda/fig3_closure_severity.png                                 40.8 KB
  outputs/eda/fig4_duration_distribution.png                            64.9 KB
  outputs/eda/fig5_top_states.png                                       81.1 KB
  outputs/eda/fig6_weather_conditions.png                               92.0 KB
  outputs/eda/fig7_correlation_heatmap.png                              96.7 KB
  outputs/eda/fig8_feature_variance.png                                 71.0 KB
  outputs/eda/fig9_pca_features.png                                    496.5 KB
  outputs/joint/fig_J1_spatiotemporal_hotspot.png                      367.9 KB
  ou